<div style="background-color: #0F172A; color: #F8FAFC;
            font-family: Segoe UI, Arial, sans-serif;
            padding: 20px; border-radius: 12px;">
  <h2 style="color: #38BDF8; font-family: Georgia, serif;">
    0. Problème
  </h2>   
</div>

>Seules les probabilités sont notées. L’objectif est donc d’estimer la loi conditionnelle du prochain composé(`next_compound`), sachant les informations disponibles à la coupure, pour les voitures prises en compte dans l’évaluation. Toute la méthode découle de ce constat. 
>
>**Notations** : Pour la voiture $i$, $x_i$ désigne les informations issues des tours terminés avant l’instant de coupure $T$, ainsi que les colonnes oracle explicitement fournies par le concours. Si $L_i$ est son dernier tour visible, $Y_i$ désigne le composé monté après son prochain arrêt strictement postérieur à $L_i$. Un arrêt sur $L_i$ lui-même ne compte pas comme un arrêt futur.La cible appartient à cinq classes : `SOFT`, `MEDIUM`, `HARD`, `WET_OR_INTER` et `NONE`. La classe NONE indique que la voiture ne s’arrête plus jusqu’à l’arrivée.
>
>Pour chaque voiture, on soumet cinq probabilités $p_{ik}$. Sur les $N$ voitures retenues par le critère de résultat final du concours, la métrique est :
>$$ \mathrm{LogLoss} = -\frac{1}{N} \sum_{i=1}^{N}\sum_{k=1}^{5} y_{ik}\ln \hat p_{ik}, \qquad \hat p_{ik} = \frac{p_{ik}}{\sum_{j=1}^{5}p_{ij}},$$
>où $$ y_{ik} = \mathbf{1}_{\{Y_i=k\}} = \begin{cases} 1 & \text{si }Y_i=k,\\ 0 & \text{sinon.}\end{cases}$$
>La perte d’une voiture est donc simplement le logarithme négatif de la probabilité attribuée à sa vraie classe. Plus la __log loss__ est faible, meilleure est la prédiction.
>
>**Pourquoi viser la vraie loi conditionnelle ?**
>
>Fixons $x$ et notons $q_k=\mathbb{P}(Y=k\mid X=x)$ la vraie probabilité de la classe $k$, dans la population évaluée. Pour une prédiction normalisée $p$, la perte conditionnelle espérée se décompose ainsi :
>$$ \mathbb{E}[-\ln p_Y\mid X=x] = -\sum_{k=1}^{5}q_k\ln p_k = \underbrace{-\sum_{k=1}^{5}q_k\ln q_k}_{H(q)} + \underbrace{\sum_{k=1}^{5}q_k \ln\frac{q_k}{p_k}}_{\mathrm{KL}(q\|p)}.$$
>
>L’entropie $H(q)$ ne dépend pas de la prédiction. La divergence de Kullback-Leibler est positive ou nulle et s’annule uniquement lorsque $(p=q)$. La __log loss__ est donc une règle de score strictement propre : sa perte espérée est minimale lorsque les probabilités prédites correspondent aux vraies probabilités conditionnelles.
>
>**Cela entraîne trois conséquences pratiques :**
> - La qualité des probabilités compte au-delà de la classe prédite. Attribuer systématiquement $90 \%$ à une issue qui survient seulement dans $60 \%$ des situations comparables traduit une confiance excessive et augmente la perte espérée.
> - Les erreurs très confiantes coûtent cher. Une probabilité de $0{,}001$ attribuée à la vraie classe entraîne une perte de $6{,}91$, contre $1{,}61$ pour une prédiction uniforme sur cinq classes. Une probabilité exactement nulle pour la vraie classe entraîne une perte théorique infinie. Dans notre notebook, nous utilisons un clipping numérique à $10^{-15}$ lors du calcul des pertes individuelles.
> - Les fréquences des classes constituent une référence naturelle. Sans aucune variable explicative, la meilleure prédiction constante est la vraie distribution marginale des classes, dont la perte espérée vaut $H(Y)$. En pratique, nous estimons ces fréquences uniquement sur l’apprentissage, avec un lissage, puis mesurons leur log loss sur la validation.
>
>**Lien avec la construction des données et la validation.**
>
> L’identité précédente est mathématique. En revanche, obtenir de bonnes performances sur de nouvelles courses suppose que les relations apprises restent suffisamment pertinentes dans les situations évaluées.
>
> Le train contient des courses complètes, tandis que le test ne montre que leur début jusqu’à une coupure située entre $30$ et $50 \%$ de la distance prévue. Nous construisons donc des observations d’apprentissage à des coupures similaires, en utilisant uniquement l’historique visible et les informations oracle autorisées pour calculer les features. La suite de la course sert à connaître la cible.
>
>Enfin, les voitures d’une même course partagent de nombreuses conditions. Nous séparons donc les données par course lors de la validation et réalisons les prétraitements ainsi que le réglage des modèles uniquement sur les données d’apprentissage de chaque pli.
​
 


<details>
<summary>La Perte Espérée</summary>
$$ \boxed{
    \text{Perte espérée} = 
    \text{incertitude réelle} 
    + \text{erreur des probabilités prédites}
}
$$


1. Fixons une situation $x$
Par exemple : une voiture en MEDIUM, des pneus âgés de 15 tours et de la pluie annoncée.

On note :
- $q_k=P(Y=k\mid X=x)$ : la vraie probabilité de chaque classe ;
- $p_k$ : la probabilité prédite par notre modèle.
Si la classe réalisée est $k$, notre perte est $-\ln p_k$. La perte espérée est donc la moyenne des pertes possibles, pondérées par leurs vraies probabilités :
$$
\mathbb E[-\ln p_Y\mid X=x]
=
-\sum_k q_k\ln p_k.
$$
2. On ajoute et soustrait le même terme

Ajouter puis soustraire $\sum_k q_k\ln q_k$ ne change pas la valeur :
$$
-\sum_k q_k\ln p_k
=
-\sum_k q_k\ln q_k
+
\sum_k q_k\ln q_k
-
\sum_k q_k\ln p_k.
$$
On regroupe les deux dernières sommes en utilisant
$\ln q_k-\ln p_k=\ln(q_k/p_k)$ :
$$
\boxed{
\mathbb E[-\ln p_Y\mid X=x]
=
\underbrace{-\sum_k q_k\ln q_k}_{H(q)}
+
\underbrace{\sum_k q_k\ln\frac{q_k}{p_k}}_{\mathrm{KL}(q\|p)}
}
$$
3. Que signifient les deux termes ?
Terme	Intuition
$H(q)$, l’entropie	L’incertitude qui subsiste avec les informations $x$, même si les probabilités sont parfaitement connues
$\mathrm{KL}(q|p)$	La perte supplémentaire due à l’écart entre nos probabilités et les vraies


La divergence $KL$ est toujours positive ou nulle. La meilleure prédiction est donc $p=q$ : la partie supplémentaire devient zéro.

</details>

<div style="background-color: #0F172A; color: #F8FAFC;
            font-family: Segoe UI, Arial, sans-serif;
            padding: 20px; border-radius: 12px;">
  <h2 style="color: #38BDF8; font-family: Georgia, serif;">
    1. Setup
  </h2>
  <p></p>
</div>

In [2]:
import numpy as np
import pandas as pd
import seaborn as sns
import math as m
import copy 
import os 
import datetime
import matplotlib.pyplot as plt
from scipy import stats  # test de Kolmogorov-Smirnov
import json  # lecture et écriture des chiffres clés et des hyperparamètres
import warnings  # filtrage des avertissements
import lightgbm as lgb  # gradient boosting sur arbres de décision
import numpy as np  # calcul numérique
import pandas as pd  # tables de données
from scipy import stats  # test de Kolmogorov-Smirnov
from sklearn.metrics import roc_auc_score  # aire sous la courbe ROC (validation adverse)
from sklearn.pipeline import Pipeline

<script>
  MathJax = {
    tex: {
      inlineMath: [['$', '$'], ['\\(', '\\)']]
    }
  };
</script>

<script id="MathJax-script" async
  src="https://cdn.jsdelivr.net/npm/mathjax@4/tex-mml-chtml.js">
</script>

<div style="background-color: #0F172A; color: #F8FAFC;
            font-family: Segoe UI, Arial, sans-serif;
            padding: 20px; border-radius: 12px;">
  <h2 style="color: #38BDF8; font-family: Georgia, serif;">
    2. Exploratory Data Analysis
  </h2>
</div>

In [3]:
train_data= pd.read_parquet("train_laps.parquet")
train_data.head()

,lap_id,race_id,year,round,race_name,circuit,circuit_id,date,driver_code,driver_number,...,rainfall,wind_speed_ms,wind_direction_deg,fastest_lap_Lplus14_s,safety_car_Lplus14,virtual_sc_Lplus14,red_flag_Lplus14,rainfall_Lplus14,wind_speed_Lplus14_ms,pressure_Lplus14_mbar
0,2018_01_ALO_001,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,ALO,14,...,False,4.1,307.0,88.248,False,False,False,False,4.1,997.0
1,2018_01_ALO_002,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,ALO,14,...,False,3.8,296.0,87.982,False,False,False,False,3.8,996.9
2,2018_01_ALO_003,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,ALO,14,...,False,4.3,289.0,87.753,False,False,False,False,4.3,997.1
3,2018_01_ALO_004,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,ALO,14,...,False,2.7,210.0,87.852,False,False,False,False,2.9,997.1
4,2018_01_ALO_005,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,ALO,14,...,False,2.5,267.0,88.488,False,False,False,False,2.5,997.2


>Regardons maintenant les dimensions et les types des colonnes.

In [4]:
print(f"Nombre de lignes : {train_data.shape[0]:,}")
print(f"Nombre de colonnes : {train_data.shape[1]}")
print(f"Nombre de courses : {train_data['race_id'].nunique()}")

train_data.info(verbose=True, show_counts=True)

Nombre de lignes : 104,757
Nombre de colonnes : 72
Nombre de courses : 95
<class 'pandas.DataFrame'>
RangeIndex: 104757 entries, 0 to 104756
Data columns (total 72 columns):
 #   Column                           Non-Null Count   Dtype  
---  ------                           --------------   -----  
 0   lap_id                           104757 non-null  str    
 1   race_id                          104757 non-null  str    
 2   year                             104757 non-null  int64  
 3   round                            104757 non-null  int64  
 4   race_name                        104757 non-null  str    
 5   circuit                          104757 non-null  str    
 6   circuit_id                       104757 non-null  str    
 7   date                             104757 non-null  object 
 8   driver_code                      104757 non-null  str    
 9   driver_number                    104757 non-null  Int64  
 10  team                             104754 non-null  str    
 11  ci

>Le jeu d’entraînement contient 104 757 observations, 72 colonnes et 95 courses. Chaque ligne décrit un pilote sur un tour enregistré et est identifiée par la combinaison de race_id (la course), driver_code (le pilote) et lap_number (le tour). Aucun doublon sur cette combinaison n’a été observé. Les colonnes regroupent la cible next_compound, des identifiants, des métadonnées et des informations relatives à la course.

>Pour chaque couple course–pilote, plusieurs lignes retracent ainsi le déroulement de la course. Notre modélisation consiste à résumer l’historique visible jusqu’à la coupure en une seule ligne de variables explicatives par voiture retenue.

>Les observations d’une même course partagent notamment les conditions météorologiques, les caractéristiques du circuit et les événements de piste. Afin d’évaluer la généralisation à de nouvelles courses, nous séparons donc les ensembles d’apprentissage et de validation par course, plutôt que par ligne.

In [5]:
missing_summary = pd.DataFrame({
    "dtype": train_data.dtypes,
    "missing_count": train_data.isna().sum(),
    "missing_percentage": train_data.isna().mean() * 100
})

missing_summary = (
    missing_summary
    .loc[lambda table: (table["missing_count"] > 0) ]
    .sort_values("missing_percentage", ascending=False)
)

display(missing_summary.round(2))

,dtype,missing_count,missing_percentage
pit_lane_time_s,Float64,101383,96.78
fastest_lap_Lplus14_s,float64,22536,21.51
virtual_sc_Lplus14,boolean,22536,21.51
red_flag_Lplus14,boolean,22536,21.51
pressure_Lplus14_mbar,float64,22536,21.51
rainfall_Lplus14,boolean,22536,21.51
wind_speed_Lplus14_ms,float64,22536,21.51
safety_car_Lplus14,boolean,22536,21.51
gap_to_car_ahead_s,float64,5964,5.69
gap_to_car_behind_s,float64,5964,5.69


>Certaines valeurs manquantes ont une explication liée au problème.
>
>- `next_compound` contient $3 566$ valeurs manquantes. D’après la définition fournie, cela peut correspondre à une voiture qui abandonne sans nouvel arrêt ou à un composé inconnu après l’arrêt. La classe `NONE`, elle, signifie que la voiture termine sans nouvel arrêt.
>- `pit_lane_time_s` est renseignée sur seulement $3 374$ lignes. Comme cette mesure concerne les passages dans les stands, nous vérifierons son lien avec `official_stop` avant de choisir un traitement.
>- Les colonnes oracle `Lplus14` ont chacune $22 536$ valeurs manquantes. L’énoncé indique qu’elles sont vides lorsque la course est terminée au tour concerné.


| Observation | Interprétation à vérifier |
|---|---|
| `pit_lane_time_s` manque sur $96,78 \%$ des lignes | Cette durée concerne les passages dans les stands, qui représentent peu de tours. |
| Les colonnes `Lplus14` manquent sur $21,51 \%$ des lignes | L’énoncé prévoit des valeurs absentes lorsque le tour futur dépasse la fin de course. |
| `compound_hardness` et `hardness_scale` manquent sur $4,77 \%$ des lignes | L’absence pourrait notamment concerner les pneus pluie, dont la dureté ne suit pas l’échelle des pneus secs. |
| Plusieurs variables de pénalité ont exactement $940$ absences | Elles pourraient partager une même source d’information manquante. |
| `next_compound` manque sur $3,40 \%$ des lignes | La définition de la cible prévoit certains cas où le prochain composé est inconnu. |
>

>Des nombres identiques d’absences suggèrent un lien, mais il faut vérifier qu’elles concernent les mêmes lignes.

<details>
<summary> Question : l’absence de durée est-elle entièrement expliquée par l’absence d’arrêt officiel ? </summary>

Pour chaque ligne $i$ du dataset, correspondant à un pilote sur un
tour d’une course, définissons :

$$
S_i = \mathbf{1}\{\texttt{official\_stop}_i = \mathrm{True}\},
\qquad
M_i = \mathbf{1}\{\texttt{pit\_lane\_time\_s}_i
\text{ est manquant}\}.
$$

La relation à vérifier sur les données est :

$$
M_i = 1 - S_i \qquad \text{pour toute ligne } i.
$$

Autrement dit, la durée aux stands serait manquante si et seulement
si aucun arrêt officiel n’est enregistré sur cette ligne. Nous ne supposons pas cette relation vraie : nous la confrontons aux données.

Les deux fréquences conditionnelles empiriques sont :

$$
\widehat{P}(M=1\mid S=0)
=
\frac{\sum_{i=1}^{n} M_i(1-S_i)}
{\sum_{i=1}^{n}(1-S_i)}
$$

et

$$
\widehat{P}(M=1\mid S=1)
=
\frac{\sum_{i=1}^{n} M_iS_i}
{\sum_{i=1}^{n}S_i}.
$$

La première mesure la proportion de durées manquantes parmi les
lignes sans arrêt officiel. La seconde mesure cette proportion parmi
les lignes avec arrêt officiel.

Si les deux groupes sont non vides et si la relation
$M_i = 1-S_i$ est vérifiée pour toutes les lignes, ces fréquences
valent respectivement $1$ et $0$.
</details>

In [6]:
stop = train_data["official_stop"] # S_i
missing_duration = train_data["pit_lane_time_s"].isna() # M_i

# Vérifier que l'indicateur d'arrêt est entièrement renseigné
assert stop.notna().all(), "L'indicateur official_stop contient des valeurs manquantes."

# Compter les quatre combinaisons possibles entre arrêt et durée manquante
counts = pd.crosstab(
    stop,
    missing_duration,
).reindex(
    index=[False, True],
    columns=[False, True],
    fill_value=0,
)

counts.index.name = "Official stop"
counts.columns.name = "Missing duration"

# Calculer les fréquences conditionnelles : chaque ligne non vide totalise 100 %
conditional_pct = (
    counts.div(
        counts.sum(axis=1).replace(0, np.nan),
        axis=0,
    )
    * 100
)

# Identifier les lignes qui contredisent la relation M_i = 1 - S_i
exceptions = missing_duration.ne(~stop.astype(bool)) # ne = not equal, ~ = logical NOT , 1-S_i

print("Effectifs")
display(counts)

print("Percentages conditional on the occurrence of a pit stop.")
display(conditional_pct.round(2))

print("Number of exceptions to M_i = 1 - S_i:", int(exceptions.sum()))

### Cela nous permet de comprendre si le taux de 96,78 % reflète principalement les tours sans passage aux stands.

Effectifs


Missing duration,False,True
Official stop,,
False,0,101383
True,3374,0


Percentages conditional on the occurrence of a pit stop.


Missing duration,False,True
Official stop,,
False,0.0,100.0
True,100.0,0.0


Number of exceptions to M_i = 1 - S_i: 0


>Dans le train, l’absence de `pit_lane_time_s` correspond exactement à l’absence d’arrêt officiel : aucune exception à $M_i=1-S_i$ n’est observée. Les $3 374$ arrêts enregistrés disposent tous d’une durée renseignée. Nous interprétons donc ces valeurs manquantes comme structurelles et ne les remplaçons pas globalement par une médiane. Cette relation est un constat sur le train, à vérifier sur toute nouvelle table.

<details>
<summary>Question : la dureté manque-t-elle principalement parce qu’elle ne s’applique pas aux pneus pluie ?</summary>

D’après le train, `compound_hardness` décrit le pneu sec utilisé. Nous allons vérifier le lien entre son absence et le composé actuel.

Pour chaque ligne $i$, définissons :
$$
H_i=\mathbf{1}\{\texttt{compound\_hardness}_i\text{ est manquant}\},
$$
et notons $C_i$ le composé actuel. Pour chaque composé $c$, nous calculerons :
$$
\widehat P(H=1\mid C=c)
=
\frac{\sum_i H_i\,\mathbf{1}\{C_i=c\}}
{\sum_i\mathbf{1}\{C_i=c\}}.
$$
**Intuition :** si la dureté manque pour tous les pneus pluie mais reste renseignée pour les pneus secs, le manque correspond à une variable non applicable. Une dureté manquante pour un pneu sec demanderait une autre explication.
</details>

In [7]:
# Conserver les composés inconnus comme groupe distinct pour l'analyse
compound = train_data["compound"].fillna("UNKNOWN")
missing_hardness = train_data["compound_hardness"].isna()

# Compter les duretés renseignées et manquantes pour chaque composé
hardness_counts = pd.crosstab(
    compound,
    missing_hardness,
).reindex(
    columns=[False, True],
    fill_value=0,
)

hardness_counts.index.name = "Current compound"
hardness_counts.columns.name = "Missing hardness"

# Calculer la proportion de duretés manquantes dans chaque groupe
hardness_summary = pd.DataFrame({
    "observations": hardness_counts.sum(axis=1),
    "missing_count": hardness_counts[True],
})

hardness_summary["missing_percentage"] = (
    100
    * hardness_summary["missing_count"]
    / hardness_summary["observations"]
)

display(hardness_counts)
display(hardness_summary.round(2))

Missing hardness,False,True
Current compound,,
HARD,45240,0
INTERMEDIATE,0,4844
MEDIUM,38306,0
SOFT,16212,0
UNKNOWN,0,27
WET,0,128


,observations,missing_count,missing_percentage
Current compound,,,
HARD,45240,0,0.0
INTERMEDIATE,4844,4844,100.0
MEDIUM,38306,0,0.0
SOFT,16212,0,0.0
UNKNOWN,27,27,100.0
WET,128,128,100.0


>Les $4 999$ valeurs manquantes de `compound_hardness` se répartissent entre $4 972$ observations en pneus pluie, pour lesquelles la référence de pneu sec n’est pas applicable, et $27$ observations dont le composé est lui-même inconnu. Aucune dureté ne manque parmi les $99 758$ observations en pneus secs (SOFT, MEDIUM, HARD) identifiés. Nous distinguons donc les absences structurelles des informations inconnues et conservons les valeurs brutes à ce stade.
>
>C’est cohérent avec la description de `compound_hardness`, qui représente l’échelle des composés secs.

Examinons maintenant les colonnes oracle à $L+14$

<details> 
<summary> Question : ces valeurs manquent-elles parce que le tour futur L+ 14 dépasse la fin de la course ?</summary>

Pour chaque ligne $i$, notons :

- $L_i$ : le numéro du tour ;
- $R_{r(i)}$ : le dernier tour enregistré dans sa course ;
- $O_i=1$ si `fastest_lap_Lplus14_s` est manquant ;
- $B_i=1$ si $L_i+14>R_{r(i)}$.
Ainsi :
$$
B_i=\mathbf{1}_{\{L_i+14>R_{r(i)}\}}.
$$

**Intuition**: Si une course se termine au tour $58$, une ligne du tour $50$ demanderait une information sur le tour $64$. Ce tour n’existe pas : l’absence d’information serait alors attendue.

Nous utilisons ici le dernier tour enregistré, pas nécessairement la distance officiellement accomplie. C’est un premier diagnostic, dont nous examinerons les exceptions.

</details>

In [8]:
# Dernier tour enregistré dans chaque course
last_recorded_lap = (
    train_data.groupby("race_id")["lap_number"]
    .transform("max")
)

# Le tour visé par l'oracle dépasse-t-il le dernier tour enregistré ?
beyond_last_lap = (
    train_data["lap_number"] + 14 > last_recorded_lap
)

# Examiner d'abord une colonne oracle représentative
missing_oracle = train_data["fastest_lap_Lplus14_s"].isna()

oracle_counts = pd.crosstab(
    beyond_last_lap,
    missing_oracle,
).reindex(
    index=[False, True],
    columns=[False, True],
    fill_value=0,
)

oracle_counts.index.name = "Beyond last recorded lap"
oracle_counts.columns.name = "Missing oracle"

display(oracle_counts)

# Repérer les manquants non expliqués par ce dépassement
unexplained_missing = missing_oracle & ~beyond_last_lap

print(
    "Valeurs manquantes sans dépassement du dernier tour enregistré :",
    int(unexplained_missing.sum()),
)

Missing oracle,False,True
Beyond last recorded lap,,
False,82221,53
True,0,22483


Valeurs manquantes sans dépassement du dernier tour enregistré : 53


**Intuition**: Notre explication fonctionne pour presque tous les manquants. Cependant, « le numéro du tour futur ne dépasse pas le maximum enregistré » ne garantit pas qu’une information oracle ait été produite pour ce tour.

Nous ne pouvons donc pas affirmer $O_i=B_i$ pour toutes les lignes. Nous observons seulement :
$$
B_i=1\ \Longrightarrow\ O_i=1
\quad\text{dans le train}
$$

In [9]:
#Notre hypothèse explique donc presque toutes les absences. Les 53 exceptions demandent une inspection. Un numéro de tour inférieur au maximum ne garantit pas que les mesures de ce tour soient disponibles.

# Localiser les manquants que le dépassement ne suffit pas à expliquer
oracle_exceptions = train_data.loc[
    unexplained_missing,
    ["race_id", "race_name", "lap_number"],
].copy()

oracle_exceptions["future_lap"] = (
    oracle_exceptions["lap_number"] + 14
)

exception_summary = (
    oracle_exceptions
    .groupby(
        ["race_id", "race_name", "lap_number", "future_lap"]
    )
    .size()
    .rename("observations")
    .reset_index()
)

display(exception_summary)

,race_id,race_name,lap_number,future_lap,observations
0,2018_07,Canadian Grand Prix,55,69,17
1,2018_07,Canadian Grand Prix,56,70,17
2,2019_17,Japanese Grand Prix,39,53,19


>**Intuition** : Plusieurs pilotes partagent une information oracle relative au même tour futur. Une indisponibilité à ce niveau peut donc se répéter sur plusieurs lignes. Ces $53$ lignes ne représentent pas nécessairement $53$ problèmes indépendants.

À ce stade, nous connaissons leur localisation, mais pas encore leur cause.

La question suivante est : les tours futurs concernés existent-ils dans le fichier avec des temps de tour renseignés ?

In [10]:
# Identifier les couples course-tour visés par les exceptions
future_keys = (
    exception_summary[["race_id", "future_lap"]]
    .drop_duplicates()
    .rename(columns={"future_lap": "lap_number"})
)

# Rechercher les observations enregistrées pour ces tours
future_rows = train_data.merge(
    future_keys,
    on=["race_id", "lap_number"],
    how="inner",
    validate="many_to_one",
)

future_check = (
    future_rows
    .groupby(["race_id", "lap_number"])
    .agg(
        observations=("driver_code", "size"),
        known_lap_times=("lap_time_s", "count"),
        fastest_lap_s=("lap_time_s", "min"),
    )
    .reset_index()
)

display(future_check)

,race_id,lap_number,observations,known_lap_times,fastest_lap_s
0,2018_07,69,14,14,73.839
1,2018_07,70,6,6,73.835
2,2019_17,53,5,5,92.683


Pourquoi cette vérification ? Elle permet de distinguer un tour futur absent ou sans chronométrage d’un tour disposant de données, mais dont l’oracle reste manquant.

>Pour `fastest_lap_Lplus14_s`, $22 483$ des $22 536$ valeurs manquantes correspondent à un tour futur dépassant le dernier tour enregistré, soit environ $99,76 \%$. Les $53$ autres sont concentrées sur trois couples course–tour futur au Canada $2018$ et au Japon $2019$. Ces tours disposent de chronométrages, mais leur information oracle est absente. La cause exacte n’étant pas établie, nous conservons ces valeurs manquantes sans reconstruction à partir du futur.

<details>
<summary>Vérifions maintenant si les sept colonnes oracle sont manquantes sur les mêmes lignes </summary>

Elles ont chacune $22 536$ manquants, mais des nombres identiques ne garantissent pas des emplacements identiques.

Pour chaque colonne oracle $j$, définissons :
$$
O_{ij}=\mathbf{1}_{\{\text{valeur oracle }j\text{ manquante sur la ligne }i\}}.
$$
En prenant `fastest_lap_Lplus14_s` comme référence, nous calculons :
$$
D_j=\sum_{i=1}^{n}\mathbf{1}\{O_{ij}\neq O_{i,\mathrm{ref}}\}.
$$
Intuition : $D_j$ compte les lignes où la disponibilité de la colonne $j$ diffère de celle de la référence. Si $D_j=0$, leurs valeurs manquantes se trouvent exactement aux mêmes endroits.

</details>

In [11]:
oracle_columns = [
    "fastest_lap_Lplus14_s",
    "safety_car_Lplus14",
    "virtual_sc_Lplus14",
    "red_flag_Lplus14",
    "rainfall_Lplus14",
    "wind_speed_Lplus14_ms",
    "pressure_Lplus14_mbar",
]

# Construire les indicateurs de valeurs manquantes
oracle_missing = train_data[oracle_columns].isna()
reference_missing = oracle_missing["fastest_lap_Lplus14_s"]

# Comparer chaque colonne à la référence, ligne par ligne
oracle_missing_summary = pd.DataFrame({
    "missing_count": oracle_missing.sum(),
    "different_from_reference": (
        oracle_missing
        .ne(reference_missing, axis=0)
        .sum()
    ),
})

display(oracle_missing_summary)

,missing_count,different_from_reference
fastest_lap_Lplus14_s,22536,0
safety_car_Lplus14,22536,0
virtual_sc_Lplus14,22536,0
red_flag_Lplus14,22536,0
rainfall_Lplus14,22536,0
wind_speed_Lplus14_ms,22536,0
pressure_Lplus14_mbar,22536,0


> Les sept colonnes oracle présentent un masque de valeurs manquantes identique : aucune différence n’est observée entre leurs indicateurs de manque. Les $22 536$ lignes concernées correspondent donc à une indisponibilité commune du bloc. Nous conservons ces valeurs manquantes et distinguerons, dans les features, l’absence d’événement de l’absence d’information.

les cibles manquantes de `next_compound`

>Selon la définition de la cible, une voiture qui abandonne sans nouvel arrêt peut avoir une cible manquante. Ce tableau permettra de vérifier dans quels statuts se concentrent ces absences. Les effectifs représentent toujours des lignes pilote-tour, donc une même voiture peut contribuer plusieurs fois.

<details>
<summary> Question : une cible manque-t-elle chez des voitures qui satisfont le critère d’évaluation ? </summary>

Pour chaque ligne $i$, définissons :

$$
A_i=\mathbf{1}_{\{\text{résultat final égal à Finished ou }+n\text{ Lap(s)}\}},
$$

$$
U_i=\mathbf{1}_{\{\texttt{next\_compound}_i\text{ est manquant}\}}.
$$
Nous vérifions l’implication :
$$
U_i=1\Longrightarrow A_i=0.
$$
Elle n’est pas supposée vraie à l’avance : la documentation indique qu’un composé inconnu après un arrêt peut également produire une cible manquante.

Intuition. Une cible absente n’est pas une sixième classe. Et `NONE` signifie « aucun nouvel arrêt jusqu’à l’arrivée », pas « réponse inconnue ».
</details>

In [12]:
# Appliquer le critère de résultat final indiqué dans l'énoncé
eligible_finish = (
    train_data["result_status"]
    .str.fullmatch(r"Finished|\+\d+ Laps?", na=False)
)

# Identifier les cibles manquantes
missing_target = train_data["next_compound"].isna()

# Compter les combinaisons entre résultat admissible et cible manquante
target_counts = pd.crosstab(
    eligible_finish,
    missing_target,
).reindex(
    index=[False, True],
    columns=[False, True],
    fill_value=0,
)

target_counts.index.name = "Eligible finish"
target_counts.columns.name = "Missing target"

# Calculer les fréquences conditionnelles par groupe
target_conditional_pct = (
    target_counts.div(
        target_counts.sum(axis=1).replace(0, np.nan),
        axis=0,
    )
    * 100
)

# Identifier les exceptions à l'implication U_i = 1 => A_i = 0
target_exceptions = missing_target & eligible_finish

print("Effectifs")
display(target_counts)

print("Pourcentages conditionnels selon le résultat final")
display(target_conditional_pct.round(2))

print(
    "Cibles manquantes malgré un résultat admissible :",
    int(target_exceptions.sum()),
)

Effectifs


Missing target,False,True
Eligible finish,,
False,4484,3566
True,96707,0


Pourcentages conditionnels selon le résultat final


Missing target,False,True
Eligible finish,,
False,55.7,44.3
True,100.0,0.0


Cibles manquantes malgré un résultat admissible : 0


<details>
<summary>L’implication est donc vérifiée dans le train</summary>

$$
U_i=1\Longrightarrow A_i=0.
$$
Mais la réciproque est fausse : parmi les lignes non admissibles, $4 484$ possèdent une cible renseignée.

**Intuition**  Une voiture peut effectuer un arrêt, dont le composé est connu, puis abandonner plus tard. Sur les tours précédant cet arrêt, `next_compound` peut donc être renseigné malgré le résultat final non admissible.
</details>

>Les $3 566$ valeurs manquantes de `next_compound` sont exclusivement associées à des résultats finaux non admissibles selon le critère du concours. Aucune cible ne manque parmi les $96 707$ lignes admissibles. Toutefois, $4 484$ lignes non admissibles disposent d’une cible connue : l’absence de cible et la non-admissibilité ne sont donc pas équivalentes. Nous conservons les cibles manquantes sans imputation et distinguons explicitement leur disponibilité du critère d’évaluation.

Passons aux écarts entre voitures, en commençant par `gap_to_car_ahead_s`, qui contient $5 964$ valeurs manquantes.

<details>
<summary>Question : ces absences concernent-elles surtout le leader, qui n’a aucune voiture devant lui au classement ?</summary>


Pour chaque ligne $i$, définissons :
$$
G_i=\mathbf{1}_{\{\texttt{gap\_to\_car\_ahead\_s}_i\text{ est manquant}\}}.
$$
Lorsque la position est connue, distinguons le leader $(\texttt{position}=1)$ des autres pilotes $(\texttt{position}>1)$. Nous conserverons aussi un groupe séparé pour les positions inconnues.

**Intuition :** un écart absent pour le leader peut être non applicable. Pour une voiture en deuxième position ou au-delà, il peut s’agir d’une information indisponible. Nous devons vérifier cette distinction avant de choisir un traitement.

L’intuition pour notre prédiction est que deux voitures ayant les mêmes pneus et le même âge de pneus peuvent adopter des stratégies différentes selon le trafic. Un arrêt peut notamment être envisagé pour tenter de gagner une position.
</details>

In [13]:
# Distinguer les positions connues et inconnues
position_group = pd.Series(
    "UNKNOWN_POSITION",
    index=train_data.index,
    dtype="string",
)

position_group.loc[
    train_data["position"].eq(1).fillna(False)
] = "LEADER"

position_group.loc[
    train_data["position"].gt(1).fillna(False)
] = "OTHER"

# Identifier les écarts manquants avec la voiture devant
missing_gap_ahead = train_data["gap_to_car_ahead_s"].isna()

# Compter les écarts renseignés et manquants dans chaque groupe
gap_counts = pd.crosstab(
    position_group,
    missing_gap_ahead,
).reindex(
    index=["LEADER", "OTHER", "UNKNOWN_POSITION"],
    columns=[False, True],
    fill_value=0,
)

gap_counts.index.name = "Position group"
gap_counts.columns.name = "Missing gap ahead"

# Calculer les proportions de valeurs manquantes par groupe
gap_summary = pd.DataFrame({
    "observations": gap_counts.sum(axis=1),
    "missing_count": gap_counts[True],
})

gap_summary["missing_percentage"] = (
    100
    * gap_summary["missing_count"]
    / gap_summary["observations"].replace(0, np.nan)
)

display(gap_counts)
display(gap_summary.round(2))

Missing gap ahead,False,True
Position group,,
LEADER,3,5758
OTHER,98790,31
UNKNOWN_POSITION,0,175


,observations,missing_count,missing_percentage
Position group,,,
LEADER,5761,5758,99.95
OTHER,98821,31,0.03
UNKNOWN_POSITION,175,175,100.00


Intuition. Pour le leader, l’écart avec une voiture devant au classement n’est normalement pas applicable. Cela correspond bien au comportement observé, avec toutefois trois lignes de leader dont l’écart est renseigné.

>Nous avons donc deux petits groupes à examiner :
>- 3 lignes avec position = 1 et un écart renseigné ;
>- 31 lignes avec position > 1 et un écart manquant.

<details>
<summary>Regardons d’abord les trois exceptions chez les leaders :</summary>

Pourquoi ? Avant de remplacer ou supprimer une valeur, nous devons voir ce qui est effectivement enregistré. Un écart nul, une valeur positive ou une valeur négative ne soulèveraient pas exactement les mêmes questions.
</details>

In [14]:
# Repérer les leaders dont l'écart avec la voiture devant est renseigné
leader_with_gap = (
    train_data["position"].eq(1).fillna(False)
    & train_data["gap_to_car_ahead_s"].notna()
)

display(
    train_data.loc[
        leader_with_gap,
        [
            "race_id",
            "driver_code",
            "lap_number",
            "position",
            "race_time_s",
            "gap_to_leader_s",
            "gap_to_car_ahead_s",
            "track_status_code",
        ],
    ]
)

,race_id,driver_code,lap_number,position,race_time_s,gap_to_leader_s,gap_to_car_ahead_s,track_status_code
8687,2018_18,RAI,5,1,501.242,266.498,266.498,1
8694,2018_18,RAI,12,1,1255.661,874.654,874.654,6>7>1
52606,2022_19,VET,40,1,4524.351,0.220,0.220,1


>Parmi les $5 761$ lignes où `position` $= 1$, $5 758$ présentent un écart à la voiture précédente manquant, soit $99,95 \%$. Cette absence est cohérente avec le fait que le leader n’a pas de voiture devant lui au classement. Les trois autres lignes présentent des écarts positifs et nécessitent une vérification. Nous conservons les données brutes sans correction automatique.

Regardons maintenant les 31 valeurs manquantes chez les voitures dont la position est supérieure à 1. Pour elles, l’absence ne s’explique pas par le fait d’être leader.

In [15]:
# Repérer les écarts manquants pour les voitures qui ne sont pas premières
other_missing_gap = (
    train_data["position"].gt(1).fillna(False)
    & train_data["gap_to_car_ahead_s"].isna()
)

# Examiner les informations de classement et de chronométrage associées
display(
    train_data.loc[
        other_missing_gap,
        [
            "race_id",
            "driver_code",
            "lap_number",
            "position",
            "race_time_s",
            "gap_to_leader_s",
            "gap_to_car_ahead_s",
            "gap_to_car_behind_s",
            "result_status",
        ],
    ].sort_values(["race_id", "lap_number", "position"])
)

##Nous utilisons ici result_status pour comprendre les données après la course, pas comme variable prédictive disponible à la coupure.

,race_id,driver_code,lap_number,position,race_time_s,gap_to_leader_s,gap_to_car_ahead_s,gap_to_car_behind_s,result_status
121,2018_01,ERI,6,18,NaN,NaN,NaN,NaN,Steering
135,2018_01,GAS,14,18,NaN,NaN,NaN,NaN,Engine
2676,2018_05,VAN,46,15,NaN,NaN,NaN,NaN,Gearbox
2849,2018_07,ALO,41,15,NaN,NaN,NaN,NaN,Exhaust
13362,2019_06,LEC,17,20,NaN,NaN,NaN,NaN,Collision
14197,2019_07,ALB,60,16,NaN,NaN,NaN,NaN,Collision damage
15703,2019_08,GRO,45,16,NaN,NaN,NaN,NaN,Retired
21487,2019_19,VET,8,20,NaN,NaN,NaN,NaN,Suspension
43908,2022_05,MAG,57,15,NaN,NaN,NaN,NaN,Front wing
51911,2022_19,HAM,40,2,4524.131,0.0,NaN,0.22,Finished


<details>
<summary>Ces résultats distinguent deux situations parmi les 31 lignes.</summary>

1. Pour $30$ lignes, le chronométrage et tous les écarts affichés sont manquants.

Leur résultat final indique un abandon ou un non-départ. L’absence de `gap_to_car_ahead_s` s’inscrit donc dans une absence plus générale d’informations temporelles.

Cela est compatible avec un tour non terminé ou non chronométré, mais le résultat final ne suffit pas à prouver ce qui s’est passé précisément sur chaque tour.

Pour notre construction des coupures, ces 30 lignes ne satisferont pas le filtre :  race["race_time_s"] <= cutoff_time

Nous conserverons leurs valeurs manquantes dans les données brutes. Cela n’exclut pas automatiquement les autres tours du même pilote.

2. La ligne de HAM est différente et rejoint l’anomalie de VET observée précédemment.

Les deux concernent la course 2022_19, au tour 40 :
| Pilote | Position | race_time_s | Écart au leader | Écart devant | Écart derrière |
|---|---:|---:|---:|---:|---:|
| HAM | 2 | 4524,131 | 0 | NA | 0,220 |
| VET | 1 | 4524,351 | 0,220 | 0,220 | — |


Nous avons :
$$
4524{,}351-4524{,}131=0{,}220\ \text{s}.
$$
Les temps et les écarts sont cohérents entre eux : HAM termine ce tour 0,220 seconde avant VET. En revanche, les positions affichées indiquent l’ordre inverse.

Cela suggère que les positions et les écarts ne correspondent pas exactement au même instant, ou qu’une anomalie affecte certaines valeurs. Nous ne pouvons pas choisir entre ces explications avec ce tableau seul.
</details>

>Parmi les $31$ lignes où la position est supérieure à $1$ et l’écart à la voiture précédente est manquant, $30$ présentent également un temps de course et les autres écarts manquants. La dernière, concernant HAM au tour $40$ de $2022\_19$, révèle une discordance entre le classement affiché et les informations temporelles, également visible chez VET au même tour. Nous conservons ces valeurs sans imputation automatique : remplacer tous les écarts manquants par zéro confondrait plusieurs situations distinctes.

Examinons maintenant les $175$ lignes où position est manquante

**Intuition :** une position inconnue ne signifie pas nécessairement un abandon. Nous vérifions cette association dans les données plutôt que de la supposer. De même, nous distinguons une absence isolée de classement d’une absence simultanée de chronométrage et d’écarts.

In [16]:
#L’objectif est de vérifier si cette absence s’accompagne, là aussi, d’un chronométrage manquant.

# Sélectionner les lignes dont la position est inconnue

unknown_position = train_data.loc[
    train_data["position"].isna()
].copy()

# Mesurer les absences dans les colonnes associées
columns_to_check = [
    "race_time_s",
    "lap_time_s",
    "gap_to_leader_s",
    "gap_to_car_ahead_s",
    "gap_to_car_behind_s",
    "next_compound",
]

unknown_summary = pd.DataFrame({
    "missing_count": unknown_position[columns_to_check].isna().sum(),
    "known_count": unknown_position[columns_to_check].notna().sum(),
})

print("Nombre de lignes avec une position inconnue :", len(unknown_position))
display(unknown_summary)

# Examiner le résultat final uniquement pour comprendre ces absences
display(
    unknown_position["result_status"]
    .value_counts(dropna=False)
    .rename_axis("result_status")
    .to_frame("observations")
)

Nombre de lignes avec une position inconnue : 175


,missing_count,known_count
race_time_s,173,2
lap_time_s,172,3
gap_to_leader_s,173,2
gap_to_car_ahead_s,175,0
gap_to_car_behind_s,173,2
next_compound,172,3


,observations
result_status,
Retired,71
Collision,42
Accident,14
Engine,10
Power Unit,5
Did not start,4
Brakes,3
Hydraulics,3
Suspension,3


<details>
<summary> Ces résultats confirment qu’une position manquante ne signifie pas systématiquement un abandon.</summary>

Parmi les $175$ lignes :
- $172$ correspondent à un résultat final non admissible à l’évaluation ;
- $3$ ont le statut $+1$ Lap : ces voitures sont classées à l’arrivée et sont donc admissibles ;
- $173$ ont un `race_time_s` manquant ; les $2$ autres peuvent satisfaire notre filtre temporel, si leur temps est inférieur ou égal à la coupure.
Attention : ce sont des lignes pilote-tour, pas nécessairement $175$ voitures différentes.
Mathématiquement, en notant $U_i=\mathbf{1}_{\{\text{position manquante}\}}$ et $A_i=\mathbf{1}_{\{\text{résultat final admissible}\}}$, nous observons :
$$
\widehat P(A=1\mid U=1)=\frac{3}{175}\approx1{,}71\%.
$$
La relation « position manquante implique résultat non admissible » est donc fausse dans le train.

Autre précaution : les effectifs identiques ne prouvent pas que les valeurs manquent sur les mêmes lignes. Les $172$ cibles manquantes et les $172$ temps au tour manquants pourraient correspondre à des ensembles différents.
</details>

Regardons maintenant les lignes classées à l’arrivée, ainsi que celles dont le temps de course est connu :

In [17]:
# Identifier les résultats admissibles parmi les positions inconnues
eligible_finish = (
    unknown_position["result_status"]
    .str.fullmatch(r"Finished|\+\d+ Laps?", na=False)
)

# Examiner les lignes classées ou disposant d'un temps de course
rows_to_inspect = (
    eligible_finish
    | unknown_position["race_time_s"].notna()
)

display(
    unknown_position.loc[
        rows_to_inspect,
        [
            "race_id",
            "driver_code",
            "lap_number",
            "position",
            "race_time_s",
            "lap_time_s",
            "gap_to_leader_s",
            "gap_to_car_ahead_s",
            "gap_to_car_behind_s",
            "next_compound",
            "result_status",
        ],
    ].sort_values(["race_id", "driver_code", "lap_number"])
)

,race_id,driver_code,lap_number,position,race_time_s,lap_time_s,gap_to_leader_s,gap_to_car_ahead_s,gap_to_car_behind_s,next_compound,result_status
8147,2018_18,ALO,5,<NA>,234.744,100.933,0.0,NaN,266.498,<NA>,Collision
8316,2018_18,GRO,12,<NA>,381.007,101.982,0.0,NaN,874.654,<NA>,Collision damage
20128,2019_17,PER,52,<NA>,NaN,NaN,NaN,NaN,NaN,NONE,+1 Lap
62566,2023_14,HUL,51,<NA>,NaN,NaN,NaN,NaN,NaN,NONE,+1 Lap
62719,2023_14,MAG,51,<NA>,NaN,NaN,NaN,NaN,NaN,NONE,+1 Lap


<details>
<summary> Ce tableau relie les anomalies que nous avions observées chez RAI aux deux lignes de ALO et GRO. </summary>


Pour la course $2018\_18$ :
- Au tour $5$, ALO a un temps de $234,744$ s, contre $501,242$ s pour RAI :
$$
  501{,}242-234{,}744=266{,}498\ \text{s}.
  $$
- Au tour $12$, GRO a un temps de $381,007$ s, contre $1 255,661$ s pour RAI :
$$
  1\,255{,}661-381{,}007=874{,}654\ \text{s}.
  $$
Ces différences correspondent exactement aux écarts attribués à RAI, pourtant indiqué en première position. ALO et GRO ont, eux, un écart au leader nul et une position inconnue.

Cela suggère que leurs temps ont servi de référence au calcul des écarts. L’égalité numérique est établie ; le mécanisme de génération reste à vérifier. Nous ne devons donc pas déduire de `gap_to_leader_s` $= 0$ qu’ALO et GRO étaient réellement leaders.


Ces voitures sont classées à l’arrivée. Une cible connue peut donc coexister avec des mesures manquantes : connaître leur prochain choix, ici, aucun nouvel arrêt, ne permet pas de retrouver leur position ou leur temps sur ce tour.

**Conséquence importante pour notre méthode :** un `race_time_s` renseigné permet d’appliquer le filtre temporel, mais ne garantit pas, à lui seul, la cohérence de la ligne. Les deux lignes ALO/GRO nécessitent donc une vérification avant de considérer notre construction des coupures comme entièrement validée.
</details>



Pour poursuivre sans modifier les données, examinons leur historique :

**Intuition :** pour une même voiture, le temps cumulé doit progresser avec les tours. Nous allons vérifier si les lignes suspectes s’insèrent correctement dans cette progression.

In [18]:
# Examiner la cohérence entre les numéros de tour et les temps cumulés
suspect_history = train_data.loc[
    train_data["race_id"].eq("2018_18")
    & train_data["driver_code"].isin(["ALO", "GRO"]),
    [
        "driver_code",
        "lap_number",
        "lap_time_s",
        "race_time_s",
        "position",
        "gap_to_leader_s",
    ],
].sort_values(["driver_code", "lap_number"])

display(suspect_history)

,driver_code,lap_number,lap_time_s,race_time_s,position,gap_to_leader_s
8146,ALO,1,133.811,133.811,20,34.422
8147,ALO,5,100.933,234.744,<NA>,0.000
8314,GRO,1,129.819,129.819,19,30.430
8315,GRO,2,149.206,279.025,19,79.268
8316,GRO,12,101.982,381.007,<NA>,0.000


<details>
<summary> Les temps augmentent, mais ils correspondent exactement à la somme des seuls tours présents, malgré les trous dans la numérotation. </summary>

Pour ALO, les tours $2$ à $4$ sont absents :

$$
\underbrace{234{,}744}_{\text{temps indiqué au tour 5}}
=
\underbrace{133{,}811}_{\text{tour 1}}
+
\underbrace{100{,}933}_{\text{durée du tour 5}}.
$$
Pour GRO, les tours $3$ à $11$ sont absents :
$$
\underbrace{381{,}007}_{\text{temps indiqué au tour 12}}
=
\underbrace{279{,}025}_{\text{temps au tour 2}}
+
\underbrace{101{,}982}_{\text{durée du tour 12}}.
$$

**Intuition :** on semble avoir additionné les durées disponibles comme si les lignes se suivaient sans interruption. Or un temps cumulé au tour $L$ doit aussi tenir compte du temps écoulé pendant les tours intermédiaires.

Cela suggère une somme cumulative sur les lignes disponibles, mais nous ne pouvons pas confirmer l’origine de l’anomalie sans le code de génération. Le numéro de tour lui-même pourrait aussi être incorrect.
</details>

>Pour ALO et GRO dans $2018_18$, les temps cumulés présentent une incohérence avec la numérotation des tours : l’incrément entre deux lignes est exactement égal à la durée du dernier tour enregistré, alors que plusieurs tours intermédiaires sont absents. Ces temps ne peuvent donc pas être considérés comme des instants de fin de tour fiables sans vérification supplémentaire.

Cela nuance notre affirmation précédente : race_time_s renseigné ne signifie pas automatiquement race_time_s fiable. Le filtre `race_time_s` <= `cutoff_time` reste notre règle de sélection temporelle, mais il suppose que les temps utilisés sont cohérents.

<details>
<summary> Nous allons rechercher le même motif dans tout le train, sans modifier les données.</summary>


Pour deux lignes successives d’une même voiture, notons :
$$
\Delta L_i=L_i-L_{i-1},
\qquad
\Delta T_i=T_i-T_{i-1},
$$
où $L_i$ est le numéro du tour, $T_i$ son `race_time_s` et $d_i$ son `lap_time_s`. Ici, $i-1$ désigne la ligne précédente disponible, pas nécessairement le tour précédent.

Le motif suspect est :
$$
\Delta L_i>1
\quad\text{et}\quad
|\Delta T_i-d_i|\leq\varepsilon.
$$
**Intuition :** plusieurs numéros de tour ont été franchis, mais le temps cumulé n’a augmenté que de la durée d’un seul tour. Nous choisissons $\varepsilon=0{,}01$ seconde pour tolérer les petits écarts d’arrondi. Ce paramètre est une tolérance de contrôle, pas un paramètre du modèle.
</details>

In [19]:
# Trier les tours de chaque voiture sans modifier la table d'origine
time_audit = train_data.sort_values(
    ["race_id", "driver_code", "lap_number"]
).copy()

car_groups = time_audit.groupby(["race_id", "driver_code"])

# Récupérer le tour et le temps de la ligne précédente disponible
time_audit["previous_lap"] = car_groups["lap_number"].shift(1)
time_audit["previous_race_time_s"] = car_groups["race_time_s"].shift(1)

# Calculer les différences entre deux lignes successives disponibles
time_audit["lap_step"] = (
    time_audit["lap_number"] - time_audit["previous_lap"]
)
time_audit["elapsed_time_s"] = (
    time_audit["race_time_s"] - time_audit["previous_race_time_s"]
)
time_audit["time_residual_s"] = (
    time_audit["elapsed_time_s"] - time_audit["lap_time_s"]
)

# Distinguer les trous de numérotation des cas réellement vérifiables
lap_gap = time_audit["lap_step"].gt(1).fillna(False)

checkable = (
    lap_gap
    & time_audit[
        ["race_time_s", "previous_race_time_s", "lap_time_s"]
    ].notna().all(axis=1)
)

# Rechercher le motif observé chez ALO et GRO
tolerance_s = 0.01

suspicious = (
    checkable
    & time_audit["time_residual_s"]
        .abs().le(tolerance_s).fillna(False)
)

print("Transitions avec des tours intermédiaires absents :", int(lap_gap.sum()))
print("Transitions vérifiables :", int(checkable.sum()))
print("Transitions présentant le motif suspect :", int(suspicious.sum()))

display(
    time_audit.loc[
        suspicious,
        [
            "race_id",
            "driver_code",
            "previous_lap",
            "lap_number",
            "previous_race_time_s",
            "race_time_s",
            "lap_time_s",
            "elapsed_time_s",
            "time_residual_s",
        ],
    ]
)

Transitions avec des tours intermédiaires absents : 3
Transitions vérifiables : 2
Transitions présentant le motif suspect : 2


,race_id,driver_code,previous_lap,lap_number,previous_race_time_s,race_time_s,lap_time_s,elapsed_time_s,time_residual_s
8147,2018_18,ALO,1.0,5,133.811,234.744,100.933,100.933,-1.421085e-14
8316,2018_18,GRO,2.0,12,279.025,381.007,101.982,101.982,2.842171e-14


<details>
<summary> Le contrôle retrouve uniquement les deux cas déjà identifiés : ALO et GRO dans 2018_18. </summary>

Sur tout le train :
- 3 transitions présentent des tours intermédiaires absents ;
- 2 sont vérifiables, car les temps nécessaires sont renseignés ;
- ces 2 transitions présentent le motif suspect.
Les résidus $-1{,}42\times10^{-14}$ et $2{,}84\times10^{-14}$ seconde sont des erreurs de représentation numérique : ils sont pratiquement nuls. L’égalité

$$
T_i-T_{i-1}=d_i
$$

est donc vérifiée à la précision numérique près, malgré plusieurs tours intermédiaires absents.

Cela ne démontre pas que tous les autres temps sont corrects, seulement que ce motif précis est limité à ces deux transitions parmi les cas vérifiables.
</details>


La troisième transition devrait correspondre au cas RIC que nous avions observé, du tour 9 au tour 49. Vérifions-le explicitement :

In [20]:
# Examiner les trous de numérotation impossibles à contrôler
uncheckable_gaps = lap_gap & ~checkable

display(
    time_audit.loc[
        uncheckable_gaps,
        [
            "race_id",
            "driver_code",
            "previous_lap",
            "lap_number",
            "previous_race_time_s",
            "race_time_s",
            "lap_time_s",
        ],
    ]
)

,race_id,driver_code,previous_lap,lap_number,previous_race_time_s,race_time_s,lap_time_s
8748,2018_18,RIC,9.0,49,NaN,NaN,100.433


le troisième cas est RIC, avec un passage du tour $9$ au tour $49$. Les deux temps cumulés sont manquants ; la seule durée du tour $49$, $100,433$ s, ne permet pas de reconstruire son instant de fin.

>L’audit des séquences de tours révèle trois transitions avec des tours intermédiaires absents, toutes dans la course 2018_18. Pour ALO et GRO, les temps cumulés présentent une incohérence apparente avec la numérotation des tours. Pour RIC, les temps cumulés manquants empêchent toute vérification. Nous conservons les données brutes et documentons ces anomalies, sans inventer de temps ni modifier les numéros de tour.

<div style="background-color: #0F172A; color: #F8FAFC;
            font-family: Segoe UI, Arial, sans-serif;
            padding: 20px; border-radius: 12px;">
  <h2 style="color: #38BDF8; font-family: Georgia, serif;">
     Q1 : comment le test a-t-il été coupé ?
  </h2>
</div>

<details>
<summary> Interprétation </summary>

Pour une course de $D$ tours prévus :
1. Choisir uniformément un tour admissible du leader :
   $$
   L\sim\mathcal U\left(
   \left\{\lceil0{,}30D\rceil,\ldots,\lfloor0{,}50D\rfloor\right\}
   \right).
   $$
2. Déterminer l’instant $T$ auquel le leader termine ce tour, à partir de race_time_s.
3. Conserver uniquement les tours terminés au plus tard à cet instant :
   $$
   \text{tour visible}\iff
   \texttt{race\_time\_s}\text{ est renseigné et }\texttt{race\_time\_s}\leq T.
   $$
**Intuition :** on arrête la course sur une photographie prise à un instant commun $T$. Le leader peut avoir terminé son tour $23$, tandis que d’autres voitures n’ont terminé que leur tour $22$. On ne coupe donc pas chaque pilote au même numéro de tour.


Le tour choisi $L$ doit vérifier :
$$
0{,}30\leq\frac{L}{D}\leq0{,}50.
$$
Comme $L$ est entier :
$$
L_{\min}=\lceil0{,}30D\rceil,
\qquad
L_{\max}=\lfloor0{,}50D\rfloor.
$$
**Intuition :** nous utilisons la distance prévue de la course. Le nombre de tours effectivement enregistrés peut différer et ne constitue pas automatiquement le bon dénominateur.

**Exemple**: Pour $58$ tours prévus, nous attendons les tours $18$ à $29$ inclus, soit $12$ possibilités. Un tirage uniforme donne alors à chaque tour la probabilité :
$$
P(L=\ell)=\frac{1}{12}.
$$


</details>


In [21]:
# Sélectionner une course pour comprendre le mécanisme
example_race_id = "2018_01"

race = train_data.loc[
    train_data["race_id"].eq(example_race_id)
].copy()

# Vérifier que la distance prévue est connue et unique
assert not race.empty, "La course sélectionnée est absente."
assert race["scheduled_laps"].notna().all(), "Distance prévue manquante."
assert race["scheduled_laps"].nunique() == 1, "Distances prévues contradictoires."

scheduled_laps = int(race["scheduled_laps"].iloc[0])

# Calculer les tours entiers compris entre 30 % et 50 %
cutoff_low = 0.30
cutoff_high = 0.50

first_cutoff_lap = int(np.ceil(cutoff_low * scheduled_laps)) # np.ciel renvoie le plus petit entier supérieur ou égal à l'argument
last_cutoff_lap = int(np.floor(cutoff_high * scheduled_laps)) #np.floor renvoie le plus grand entier inférieur ou égal à l'argument

candidate_laps = np.arange(first_cutoff_lap, last_cutoff_lap + 1)

print("Course :", example_race_id)
print("Nombre de tours prévus :", scheduled_laps)
print("Tours admissibles :", candidate_laps.tolist())
print("Nombre de choix possibles :", len(candidate_laps))

Course : 2018_01
Nombre de tours prévus : 58
Tours admissibles : [18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29]
Nombre de choix possibles : 12


<details>
<summary> Passons maintenant du numéro de tour L à l’instant de coupure T.</summary>

Pour comprendre le mécanisme et retrouver notre exemple précédent, fixons provisoirement $L=23$. Ce choix est pédagogique ; nous ferons le tirage aléatoire ensuite.

La première voiture à terminer le tour $23$ est celle qui atteint ce numéro de tour en premier. Avec des temps cumulés cohérents :
$$
T=\min_{i:\,\text{lap\_number}_i=23}
\text{race\_time\_s}_i.
$$
**Intuition :** les voitures terminent leur tour $23$ à des instants différents. Nous prenons le premier de ces instants, et non la moyenne.
Nous examinerons ce tableau avant de filtrer les tours visibles. La colonne `position` sert ici de contrôle ; c’est le temps qui définit $T$.
</details>

In [22]:
# Fixer un tour admissible pour expliquer la coupure
cutoff_lap = 23
assert cutoff_lap in candidate_laps, "Le tour choisi est hors de la fenêtre."

# Examiner les temps connus de fin du tour choisi
cutoff_candidates = (
    race.loc[
        race["lap_number"].eq(cutoff_lap)
        & race["race_time_s"].notna(),
        ["driver_code", "lap_number", "position", "race_time_s"],
    ]
    .sort_values("race_time_s")
)

assert not cutoff_candidates.empty, "Aucun temps disponible pour ce tour."

# Le premier passage à ce tour définit l'instant de coupure
cutoff_time = cutoff_candidates["race_time_s"].min()

display(cutoff_candidates)
print("Instant de coupure, en secondes :", cutoff_time)

,driver_code,lap_number,position,race_time_s
904,VET,23,1,2056.900
182,HAM,23,2,2068.952
551,RAI,23,3,2074.718
158,GRO,23,4,2083.839
609,RIC,23,5,2084.833
297,HUL,23,6,2088.082
22,ALO,23,7,2092.120
788,VAN,23,8,2099.692
493,PER,23,9,2103.020
80,BOT,23,10,2103.548


Instant de coupure, en secondes : 2056.9


<details>
<summary>Nous avons notre instant de coupure  : </summary>

 $T=2\,056{,}9$ secondes. VET termine son tour $23$ en premier, et sa position vaut bien $1$.

À cet instant :
- le tour $23$ de VET est visible, car son temps est égal à $T$ ;
- le tour $23$ de HAM est caché, car $2\,068{,}952>T$ ;
- les tours précédents de HAM sont visibles s’ils sont terminés avant $T$.
Nous appliquons donc le même instant à toutes les voitures, pas le même numéro de tour.

Mathématiquement, l’historique visible est :
$$
\mathcal V(T)=
\left\{i:\ T_i\text{ est renseigné et }T_i\leq T\right\}.
$$
</details>


In [23]:
# Conserver les tours dont la fin est connue et antérieure ou égale à T
visible_laps = race.loc[
    race["race_time_s"].notna()
    & race["race_time_s"].le(cutoff_time)
].copy()

# Afficher le dernier tour visible de chaque pilote
last_visible = (
    visible_laps
    .sort_values(["driver_code", "lap_number"])
    .drop_duplicates("driver_code", keep="last")
    [
        ["driver_code", "lap_number", "race_time_s"]
    ]
    .sort_values(["lap_number", "race_time_s"], ascending=False)
)

print("Nombre de tours visibles :", len(visible_laps))
display(last_visible)

## Attention à la distinction : ce tableau contient encore l’historique de pilotes ayant éventuellement abandonné avant T.

Nombre de tours visibles : 396


,driver_code,lap_number,race_time_s
904,VET,23,2056.900
354,LEC,22,2052.541
666,SAI,22,2028.744
729,STR,22,2027.276
845,VER,22,2019.759
434,OCO,22,2017.688
79,BOT,22,2013.290
492,PER,22,2012.779
412,MAG,22,2011.467
787,VAN,22,2009.881


<details>
<summary>Ce tableau illustre exactement la coupure à </summary>

 $T=2\,056{,}9$ secondes.

Nous conservons $396$ lignes pilote-tour, puis nous affichons ici seulement la dernière ligne visible de chacun des $20$ pilotes.
| Pilote | Dernier tour visible | Explication |
|---|---:|---|
| VET | $23$ | Il termine ce tour exactement à $T$ |
| HAM | $22$ | Son tour $22$ est terminé, mais son tour $23$ se termine après $T$ |
| HAR | $21$ | Son dernier tour terminé et chronométré avant $T$ est le $21$ |
| SIR | $4$ | Son historique visible s’arrête au tour $4$ |

Pour HAM, nous pouvons maintenant écrire précisément :
$$
\underbrace{1\,980{,}724}_{\text{fin du tour 22}}
<
\underbrace{2\,056{,}900}_{T}
<
\underbrace{2\,068{,}952}_{\text{fin du tour 23}}.
$$

À la coupure, HAM est donc entre la fin de son tour $22$ et la fin de son tour $23$. Nous utilisons ses lignes jusqu’au tour $22$.

****Deux précisions importantes :**
- Ce tableau n’est pas un classement sportif : il est trié par numéro de tour puis par temps décroissant.
- Avoir un historique visible ne prouve pas qu’une voiture est encore en course à $T$. Les anciens tours d’une voiture ayant abandonné restent présents.

</details>

>Pour la course $2018\_01$, la coupure au premier passage au tour $23$ correspond à $T=2\,056{,}9$ secondes. Le filtre temporel conserve 396 observations pilote-tour concernant $20$ pilotes. Le dernier tour visible varie selon le pilote : $23$ pour VET, $22$ pour HAM et $21$ pour HAR. Cette sélection décrit l’historique disponible ; elle ne détermine pas encore quelles voitures doivent recevoir une prédiction.

##### Il faut distinguer « avoir des tours visibles » et « être encore en course à $T$ ». Une voiture ayant abandonné avant $T$ conserve un historique, mais ne doit plus recevoir de prédiction.

<details>
<summary> Avant de sélectionner les voitures, construisons donc un tableau de contrôle.</summary>

Pour chaque pilote $d$, calculons :

$$
T_d^{\mathrm{last}}
=\max\{\texttt{race\_time\_s}\text{ renseignés pour ce pilote}\}.
$$

Si $T_d^{\mathrm{last}}>T$, nous disposons d’un tour terminé après la coupure : sous réserve de temps cohérents, cela confirme que la voiture roulait encore à $T$.

En revanche, $T_d^{\mathrm{last}}\leq T$ ne prouve pas à lui seul un abandon avant $T$ : la voiture peut abandonner après la coupure sans terminer un nouveau tour.
</details>



In [24]:
# Résumer l'historique complet pour contrôler la sélection des voitures
car_audit = (
    race.groupby("driver_code")
    .agg(
        last_recorded_lap=("lap_number", "max"),
        last_known_time_s=("race_time_s", "max"),
    )
)

# Ajouter le dernier tour terminé avant ou à la coupure
car_audit["last_visible_lap"] = (
    visible_laps.groupby("driver_code")["lap_number"].max()
)

# Vérifier l'existence d'un tour chronométré après T
car_audit["timed_lap_after_cutoff"] = (
    car_audit["last_known_time_s"].gt(cutoff_time).fillna(False)
)

# Comparer avec la règle fondée uniquement sur les numéros de tour
car_audit["recorded_lap_after_visible"] = (
    car_audit["last_recorded_lap"]
    .gt(car_audit["last_visible_lap"])
    .fillna(False)
)

display(car_audit.sort_index())

,last_recorded_lap,last_known_time_s,last_visible_lap,timed_lap_after_cutoff,recorded_lap_after_visible
driver_code,,,,,
ALO,58,5401.169,22,True,True
BOT,58,5407.622,22,True,True
ERI,6,475.096,5,False,True
GAS,14,1211.451,13,False,True
GRO,24,2192.878,22,True,True
HAM,58,5378.319,22,True,True
HAR,57,5395.759,21,True,True
HUL,58,5405.954,22,True,True
LEC,58,5449.042,22,True,True


Pour poursuivre notre exemple, nous pouvons reprendre la convention initial : retenir les voitures disposant d’un tour chronométré après $T$.
$$
\mathcal D_{\mathrm{retenus}}(T)
=
\{d:T_d^{\mathrm{last}}>T\}.
$$
C’est une approximation opérationnelle, pas une règle exacte fournie par l’énoncé. Elle peut exclure une voiture encore en course à $T$ qui abandonne avant de terminer un nouveau tour.

In [25]:
# Appliquer la convention retenue pour simuler la population à prédire
retained_drivers = car_audit.index[
    car_audit["timed_lap_after_cutoff"]
]

# Conserver uniquement leur historique visible
visible_selected = visible_laps.loc[
    visible_laps["driver_code"].isin(retained_drivers)
].copy()

print("Nombre de voitures retenues :", len(retained_drivers))
print(
    "Pilotes exclus par cette convention :",
    sorted(set(last_visible["driver_code"]) - set(retained_drivers)),
)

Nombre de voitures retenues : 16
Pilotes exclus par cette convention : ['ERI', 'GAS', 'MAG', 'SIR']


>La prochaine étape est de construire une ligne par voiture, correspondant à son dernier tour visible. C’est cette ligne qui porte la cible `next_compound` à prédire.

<details>
<summary>Intuition :  </summary>

Pour HAM, nous prenons sa ligne du tour $22$ ; pour VET, celle du tour $23$. La cible déjà calculée sur la course complète décrit le prochain composé selon la définition de `target.py` : prochain arrêt strictement après ce dernier tour visible.

Nous ne recalculons donc pas la cible à partir de l’historique tronqué : il ne contient plus les événements nécessaires pour la déterminer.

Enfin, `result_status` sert uniquement à construire le masque d’évaluation scored. Il ne sera pas donné au modèle comme feature. Nous utilisons ici `FINISHED` du module (`traget.py`) fourni, qui inclut également le statut `Lapped`.
</details>

In [26]:
# Prendre la dernière ligne visible de chaque voiture retenue
snapshot = (
    visible_selected
    .sort_values(["driver_code", "lap_number"])
    .drop_duplicates("driver_code", keep="last")
    .copy()
)

# Identifier les observations évaluables selon le module fourni
from target import FINISHED

snapshot["scored"] = (
    snapshot["result_status"].str.match(FINISHED, na=False)
    & snapshot["next_compound"].notna()
)

display(
    snapshot[
        [
            "driver_code",
            "lap_number",
            "compound",
            "next_compound",
            "scored",
        ]
    ]
)

print("Voitures retenues :", len(snapshot))
print("Observations évaluables :", int(snapshot["scored"].sum()))

,driver_code,lap_number,compound,next_compound,scored
21,ALO,22,SOFT,HARD,True
79,BOT,22,SOFT,MEDIUM,True
157,GRO,22,SOFT,<NA>,False
181,HAM,22,HARD,NONE,True
238,HAR,21,HARD,SOFT,True
296,HUL,22,SOFT,HARD,True
354,LEC,22,HARD,SOFT,True
434,OCO,22,SOFT,HARD,True
492,PER,22,SOFT,HARD,True
550,RAI,22,HARD,NONE,True


Voitures retenues : 16
Observations évaluables : 15


<details>
<summary>Nous retrouvons bien 16 voitures retenues, dont 15 évaluables: </summary>

Chaque ligne représente maintenant une voiture à notre instant de coupure $T=2\,056{,}9$ secondes.

| Pilote | Information visible | Cible à prédire |
|---|---|---|
| ALO | En `SOFT` au dernier tour visible | `HARD` au prochain arrêt |
| HAM | En `HARD` | `NONE` : aucun nouvel arrêt selon la définition de la cible |
| HAR | En `HARD` | `SOFT` au prochain arrêt |
| GRO | En `SOFT`, encore en course à $(T)$ | Cible manquante ; observation non évaluée |

GRO illustre la distinction essentielle : être présent à la coupure ne signifie pas terminer la course. Nous conservons son état pour décrire le peloton visible, mais sa ligne ne contribuera pas à la perte supervisée.

Pour les voitures évaluables, le modèle devra produire :
$$
p_{d,k}=P(Y_d=k\mid X_d(T)),
$$
où $X_d(T)$ résume les informations autorisées à la coupure, et $Y_d$ correspond à `next_compound`.
</details> 


> La coupure au tour $23$ du leader produit un instant commun $T=2\,056{,}9$ secondes. Selon notre convention de sélection, $16$ voitures sont retenues. Leur dernier tour visible varie entre $21$ et $23$. Parmi elles, $15$ disposent d’un résultat final admissible et d’une cible connue. Les colonnes `next_compound` et `scored` servent respectivement de cible et de masque d’évaluation ; elles ne constituent pas des variables explicatives.

Nous allons maintenant tirer le tour de coupure au hasard, au lieu de fixer `cutoff_lap` $ = 23$.

In [27]:
# Initialiser le générateur pour rendre le tirage reproductible
random_seed = 2026
rng = np.random.default_rng(random_seed)

# Tirer uniformément un tour parmi les tours admissibles
random_cutoff_lap = int(rng.choice(candidate_laps))

# Déterminer le premier instant connu de fin de ce tour
candidate_times = race.loc[
    race["lap_number"].eq(random_cutoff_lap),
    "race_time_s",
].dropna()

assert not candidate_times.empty, "Aucun temps disponible pour le tour tiré."

random_cutoff_time = candidate_times.min()

print("Tour de coupure tiré :", random_cutoff_lap)
print(
    "Avancement du leader (%) :",
    round(100 * random_cutoff_lap / scheduled_laps, 2),
)
print("Instant de coupure (s) :", random_cutoff_time)

Tour de coupure tiré : 28
Avancement du leader (%) : 48.28
Instant de coupure (s) : 2612.42


Nous allons maintenant appliquer les mêmes étapes qu’au tour 23 :

**Intuition :** nous photographions la même course plus tard. Certaines voitures peuvent avoir changé de pneus ou abandonné entre les deux coupures ; leur état et leur cible peuvent donc changer.

Nous conservons la même convention de sélection : un tour chronométré après $T$. Sa limite reste la même pour les voitures qui abandonnent sans terminer un nouveau tour.

In [28]:
# Conserver les tours terminés avant ou à la nouvelle coupure
random_visible_laps = race.loc[
    race["race_time_s"].notna()
    & race["race_time_s"].le(random_cutoff_time)
].copy()

# Appliquer notre convention de sélection des voitures
last_known_times = race.groupby("driver_code")["race_time_s"].max()

random_retained_drivers = last_known_times.index[
    last_known_times.gt(random_cutoff_time).fillna(False)
]

random_visible_selected = random_visible_laps.loc[
    random_visible_laps["driver_code"].isin(random_retained_drivers)
].copy()

# Construire une ligne par voiture retenue
random_snapshot = (
    random_visible_selected
    .sort_values(["driver_code", "lap_number"])
    .drop_duplicates("driver_code", keep="last")
    .copy()
)

# Déterminer quelles observations contribuent à l'évaluation
random_snapshot["scored"] = (
    random_snapshot["result_status"].str.match(FINISHED, na=False)
    & random_snapshot["next_compound"].notna()
)

display(
    random_snapshot[
        ["driver_code", "lap_number", "compound", "next_compound", "scored"]
    ]
)

print("Voitures retenues :", len(random_snapshot))
print("Observations évaluables :", int(random_snapshot["scored"].sum()))

,driver_code,lap_number,compound,next_compound,scored
26,ALO,27,HARD,NONE,True
84,BOT,27,MEDIUM,NONE,True
186,HAM,27,HARD,NONE,True
243,HAR,26,SOFT,NONE,True
301,HUL,27,HARD,NONE,True
359,LEC,27,HARD,NONE,True
439,OCO,27,HARD,NONE,True
497,PER,27,HARD,NONE,True
555,RAI,27,HARD,NONE,True
613,RIC,27,HARD,NONE,True


Voitures retenues : 15
Observations évaluables : 15


<details>
<summary> Cette nouvelle coupure montre que la cible dépend du moment où l’on observe la course: </summary>

Nous avons maintenant $15$ voitures retenues, toutes évaluables. Comparons quelques pilotes :

| Pilote | Au cutoff du tour $23$ : pneu → cible | Au cutoff du tour $28$ : pneu → cible |
|---|---|---|
| ALO | `SOFT` → `HARD` | `HARD` → `NONE` |
| BOT | `SOFT` → `MEDIUM` | `MEDIUM` → `NONE` |
| HAR | `HARD` → `SOFT` | `SOFT` → `NONE` |
| STR | `MEDIUM` → `HARD` | `HARD` → `SOFT` |


**Intuition :** au premier instant, nous cherchions le composé du prochain arrêt. Au second, cet arrêt est déjà passé pour ces pilotes : nous cherchons désormais celui de l’arrêt suivant, s’il existe.

Pour STR, par exemple, le prochain composé était `HARD`. Maintenant qu’il roule en `HARD`, sa nouvelle cible est `SOFT`.

GRO n’est plus retenu : son dernier temps connu, $2\,192{,}878$ s, est antérieur à la nouvelle coupure $2\,612{,}42$ s.

La distribution des cibles est désormais :
$$
\widehat P(Y=\text{NONE})=\frac{14}{15}\approx93{,}33\%,
\qquad
\widehat P(Y=\text{SOFT})=\frac{1}{15}\approx6{,}67\%.
$$
Ces proportions décrivent cette course à cette coupure, pas toutes les courses.


</details>

>Déplacer la coupure modifie la population retenue, l’état visible des voitures et les cibles à prédire. Dans cet exemple, plusieurs arrêts ont eu lieu entre les deux coupures : au tour $28$ du leader, $14$ des $15$ voitures retenues ont pour cible `NONE`. Cela motive l’étude de la distribution de la cible selon l’avancement de la course.

>Nous allons regrouper les étapes déjà vérifiées dans une fonction, pour pouvoir les appliquer à chaque course.
>
>Elle renverra deux tables :
>- `visible` : l’historique visible des voitures retenues, nécessaire pour construire les features ;
>- `snapshot` : leur dernière ligne visible, avec la cible et les informations de coupure.
>
>**Intuition :** la fonction applique une coupure donnée. Le tirage aléatoire reste à l’extérieur : nous pouvons ainsi reproduire une coupure précise ou en tirer une nouvelle sans changer la logique.

In [29]:
def build_race_snapshot(race, cutoff_lap):
    """Construit une coupure et une ligne par voiture retenue."""

    # Vérifier que l'entrée correspond à une seule course
    if race.empty or race["race_id"].nunique() != 1:
        raise ValueError("Une seule course non vide est attendue.")

    # Déterminer l'instant du premier passage au tour choisi
    cutoff_times = race.loc[
        race["lap_number"].eq(cutoff_lap),
        "race_time_s",
    ].dropna()

    if cutoff_times.empty:
        raise ValueError("Aucun temps connu pour le tour choisi.")

    cutoff_time = cutoff_times.min()

    # Sélectionner les tours terminés avant ou à T
    visible = race.loc[
        race["race_time_s"].notna()
        & race["race_time_s"].le(cutoff_time)
    ].copy()

    # Appliquer notre convention : un tour chronométré existe après T
    last_known_times = race.groupby("driver_code")["race_time_s"].max()
    retained_drivers = last_known_times.index[
        last_known_times.gt(cutoff_time).fillna(False)
    ]

    visible = visible.loc[
        visible["driver_code"].isin(retained_drivers)
    ].copy()

    # Prendre la dernière ligne visible de chaque voiture retenue
    snapshot = (
        visible
        .sort_values(["driver_code", "lap_number"])
        .drop_duplicates("driver_code", keep="last")
        .copy()
    )

    # Conserver les informations permettant de retrouver la coupure
    snapshot["cutoff_lap"] = cutoff_lap
    snapshot["cutoff_time_s"] = cutoff_time

    # Identifier les observations évaluables sans modifier la cible
    snapshot["scored"] = (
        snapshot["result_status"].str.match(FINISHED, na=False)
        & snapshot["next_compound"].notna()
    )

    return visible, snapshot

Avant de l’utiliser sur les $95$ courses, vérifions qu’elle reproduit exactement notre calcul manuel au tour $28$ :

In [30]:
# Reproduire la coupure déjà calculée manuellement
visible_check, snapshot_check = build_race_snapshot(
    race,
    random_cutoff_lap,
)

# Comparer les colonnes communes aux deux résultats
pd.testing.assert_frame_equal(
    snapshot_check[random_snapshot.columns]
    .sort_values("driver_code")
    .reset_index(drop=True),
    random_snapshot
    .sort_values("driver_code")
    .reset_index(drop=True),
)

print("La fonction reproduit exactement notre snapshot manuel.")
print("Voitures retenues :", len(snapshot_check))

La fonction reproduit exactement notre snapshot manuel.
Voitures retenues : 15


Nous pouvons maintenant construire une coupure aléatoire par course, pour préparer la suite de l’EDA.

Pour chaque course $r$, de distance prévue $D_r$, nous tirons :
$$
L_r\sim\mathcal U\left(
\{\lceil0{,}30D_r\rceil,\ldots,\lfloor0{,}50D_r\rfloor\}
\right).
$$
Chaque course reçoit un tirage, mais peut fournir un nombre différent de voitures retenues.

In [31]:
# Initialiser une seule fois le générateur pour toutes les courses
rng = np.random.default_rng(2026)

snapshot_parts = []
cutoff_records = []

# Parcourir les courses dans un ordre reproductible
for race_id, race_table in train_data.groupby("race_id", sort=True):

    # Vérifier la distance prévue avant de définir les tours admissibles
    if (
        race_table["scheduled_laps"].isna().any()
        or race_table["scheduled_laps"].nunique() != 1
    ):
        raise ValueError(f"Distance prévue absente ou contradictoire : {race_id}")

    scheduled_laps = int(race_table["scheduled_laps"].iloc[0])

    candidate_laps = np.arange(
        int(np.ceil(cutoff_low * scheduled_laps)),
        int(np.floor(cutoff_high * scheduled_laps)) + 1,
    )

    if len(candidate_laps) == 0:
        raise ValueError(f"Aucun tour admissible : {race_id}")

    # Effectuer un tirage uniforme pour cette course
    sampled_lap = int(rng.choice(candidate_laps))

    # Construire le snapshot avec la fonction vérifiée
    try:
        visible, race_snapshot = build_race_snapshot(
            race_table,
            sampled_lap,
        )
    except ValueError as error:
        raise ValueError(
            f"Échec pour {race_id}, tour {sampled_lap} : {error}"
        ) from error

    if race_snapshot.empty:
        raise ValueError(f"Aucune voiture retenue : {race_id}")

    snapshot_parts.append(race_snapshot)

    # Conserver une ligne de contrôle par course
    cutoff_records.append({
        "race_id": race_id,
        "scheduled_laps": scheduled_laps,
        "cutoff_lap": sampled_lap,
        "cutoff_progress": sampled_lap / scheduled_laps,
        "cutoff_time_s": race_snapshot["cutoff_time_s"].iloc[0],
        "retained_cars": len(race_snapshot),
        "scored_cars": int(race_snapshot["scored"].sum()),
    })

# Assembler les voitures et les informations sur les coupures
eda_snapshots = pd.concat(snapshot_parts, ignore_index=True)
cutoff_summary = pd.DataFrame(cutoff_records)

# Vérifier qu'une voiture n'apparaît qu'une fois par course
assert not eda_snapshots.duplicated(
    ["race_id", "driver_code"]
).any()

assert cutoff_summary["cutoff_progress"].between(
    cutoff_low, cutoff_high
).all()

print("Courses représentées :", len(cutoff_summary))
print("Voitures retenues :", len(eda_snapshots))
print("Observations évaluables :", int(eda_snapshots["scored"].sum()))

display(cutoff_summary.head())

Courses représentées : 95
Voitures retenues : 1759
Observations évaluables : 1609


,race_id,scheduled_laps,cutoff_lap,cutoff_progress,cutoff_time_s,retained_cars,scored_cars
0,2018_01,58,28,0.482759,2612.420,15,15
1,2018_04,51,17,0.333333,2122.045,17,13
2,2018_05,66,20,0.303030,1933.695,17,14
3,2018_07,70,30,0.428571,2449.805,18,17
4,2018_09,71,27,0.380282,1914.578,18,14


Avant d’étudier la distribution des cibles, réservons les courses de validation. Nous pourrons ainsi poursuivre l’EDA qui guidera nos choix sur les seules courses d’entraînement.

Nous conservons les voitures non évaluables dans `eda_train` et `eda_valid` pour décrire le peloton. Seules les lignes évaluables serviront à calculer la perte supervisée.

In [32]:
from sklearn.model_selection import GroupShuffleSplit # sert à séparer les courses en ensembles d'entraînement et de validation, en conservant toutes les voitures d'une même course dans le même ensemble.


# Séparer les courses, en conservant toutes leurs voitures ensemble
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=2026,
)

train_idx, valid_idx = next(
    splitter.split(
        eda_snapshots,
        groups=eda_snapshots["race_id"],
    )
)

eda_train = eda_snapshots.iloc[train_idx].copy()
eda_valid = eda_snapshots.iloc[valid_idx].copy()

# Vérifier qu'aucune course n'appartient aux deux ensembles
train_races = set(eda_train["race_id"])
valid_races = set(eda_valid["race_id"])

assert train_races.isdisjoint(valid_races)

# Identifier les observations utilisables pour l'apprentissage et l'évaluation
scored_train = eda_train.loc[eda_train["scored"]].copy()
scored_valid = eda_valid.loc[eda_valid["scored"]].copy()

print(
    f"Entraînement : {len(train_races)} courses, "
    f"{len(scored_train)} observations évaluables"
)
print(
    f"Validation : {len(valid_races)} courses, "
    f"{len(scored_valid)} observations évaluables"
)

Entraînement : 76 courses, 1296 observations évaluables
Validation : 19 courses, 313 observations évaluables


Nous poursuivons maintenant l’EDA sur `scored_train` uniquement. Les $313$ observations de validation serviront ensuite à évaluer les modèles.

**Commençons par une question simple : quelles classes devons-nous prédire, et avec quelle fréquence ?**

Pour chaque classe $k$, calculons :
$$
n_k=\sum_{i=1}^{1296}\mathbf{1}\{Y_i=k\},
\qquad
\widehat p_k=\frac{n_k}{1296}.
$$
**Intuition :** $\widehat p_k$ est la proportion de voitures dont le prochain composé est $k$, dans notre population d’entraînement aux coupures simulées.

In [33]:
from target import CLASSES

# Compter les cibles dans les seules observations d'entraînement évaluables
class_counts = (
    scored_train["next_compound"]
    .value_counts()
    .reindex(CLASSES, fill_value=0)
)

# Mesurer aussi le nombre de courses où chaque classe apparaît
class_race_counts = (
    scored_train.groupby("next_compound")["race_id"]
    .nunique()
    .reindex(CLASSES, fill_value=0)
)

target_summary = pd.DataFrame({
    "observations": class_counts,
    "percentage": 100 * class_counts / len(scored_train),
    "courses": class_race_counts,
})

target_summary.index.name = "next_compound"

display(target_summary.round(2))

,observations,percentage,courses
next_compound,,,
SOFT,190,14.66,49
MEDIUM,305,23.53,68
HARD,469,36.19,61
WET_OR_INTER,35,2.7,2
NONE,297,22.92,44


La colonne `courses` est particulièrement importante pour la pluie : $20$ voitures passant aux pneus pluie dans une même course représentent un seul contexte météorologique partagé, pas $20$ situations indépendantes.

La classe `HARD` est la plus fréquente, tandis que les $35$ cibles `WET_OR_INTER` sont concentrées dans seulement $2$ courses.

<details>
<summary> Trois enseignements :</summary>

- HARD représente $36,19 \%$ des observations, mais apparaît dans 61 courses. `MEDIUM`, moins fréquent en nombre de voitures, apparaît dans davantage de courses : $68$.
- `NONE` représente $22,92 \%$ : ne plus s’arrêter est une situation importante, sans être majoritaire.
- `WET_OR_INTER` représente $2,70 \%$, avec seulement deux contextes de course représentés. Cela limite notre capacité à apprendre une relation qui se généralise à d’autres courses.

Attention : « $2$ courses avec une cible pluie » ne signifie pas « seulement $2$ courses où il pleut ». La cible décrit le prochain pneu monté, pas la météo actuelle.

Pour la classe pluie, nous avons :
$$
\widehat P(Y=\text{WET\_OR\_INTER})
=\frac{35}{1296}\approx2{,}70\%.
$$
C’est une fréquence globale. Notre objectif sera ensuite d’estimer comment cette probabilité change selon la météo, le pneu actuel et les autres informations visibles.
</details>


Avant cela, regardons précisément comment ces 35 observations se répartissent :

**Intuition : ce tableau nous dira si ces cibles concernent presque tout le peloton dans chaque course, ou seulement quelques voitures. Nous ne modifions pas encore les features ni la pondération des classes.**

In [34]:
# Examiner les courses d'entraînement avec au moins une cible pluie
wet_targets = scored_train.loc[
    scored_train["next_compound"].eq("WET_OR_INTER")
]

wet_by_race = (
    wet_targets.groupby("race_id")
    .agg(
        wet_targets=("driver_code", "size"),
        cutoff_lap=("cutoff_lap", "first"),
    )
)

# Rapporter les cibles pluie à toutes les voitures évaluables de la course
wet_by_race["scored_cars"] = (
    scored_train.groupby("race_id").size()
)

wet_by_race["wet_target_percentage"] = (
    100 * wet_by_race["wet_targets"] / wet_by_race["scored_cars"]
)

display(wet_by_race.round(2))

,wet_targets,cutoff_lap,scored_cars,wet_target_percentage
race_id,,,,
2021_16,17,22,20,85.0
2024_12,18,18,18,100.0


<details>
<summary>Les 35 cibles pluie sont concentrées dans deux situations collectives : </summary>

| Course | Coupure du leader | Cibles pluie / voitures évaluables |
|---|---:|---:|
| `2021_16` | Tour 22 | 17 / 20, soit 85 % |
| `2024_12` | Tour 18 | 18 / 18, soit 100 % |

Les voitures d’une même course partagent les conditions météo. Nous avons donc 35 exemples de voitures, mais seulement deux contextes de course avec une cible pluie, à ces coupures.

Cela explique pourquoi une validation par course est exigeante : le modèle doit généraliser à une autre situation, plutôt que retrouver des voitures ayant partagé la météo de son entraînement.


**La prochaine distinction importante est :** ces voitures roulent-elles déjà en pneus pluie, ou vont-elles passer des pneus secs aux pneus pluie ?

Ce sont deux transitions différentes :
$$
\text{pneu sec}\rightarrow\text{pneu pluie},
\qquad
\text{pneu pluie}\rightarrow\text{pneu pluie}.
$$

**Intuition :** si les exemples concernent surtout des voitures déjà en pneus pluie, ils enseignent peu au modèle sur le moment où une voiture en pneus secs doit changer de type de pneu. C’est pourquoi le nombre total de cibles pluie ne suffit pas à mesurer la diversité des exemples disponibles.
</details>

In [35]:
# Décrire le pneu actuel des voitures dont la cible est un pneu pluie
wet_transition_summary = (
    wet_targets
    .groupby(["race_id", "compound"], dropna=False)
    .size()
    .rename("observations")
    .reset_index()
)

display(wet_transition_summary)

,race_id,compound,observations
0,2021_16,INTERMEDIATE,17
1,2024_12,HARD,1
2,2024_12,MEDIUM,16
3,2024_12,SOFT,1


<details>
<summary>Nous avons deux situations bien distinctes : </summary>

| Course | Pneu actuel | Prochain pneu | Observations |
|---|---|---|---:|
| `2021_16` | `INTERMEDIATE` | `WET_OR_INTER` | $17$ |
| `2024_12` | Pneus secs : $1$ `HARD`, $16$ `MEDIUM`, $1$ `SOFT` | `WET_OR_INTER` | $18$ |


Pour 2021_16, les voitures utilisent déjà des pneus intermédiaires. La cible regroupée `WET_OR_INTER` ne permet pas de savoir ici si elles remontent des intermédiaires ou passent aux pneus pluie.

Pour 2024_12, les $18$ voitures passent d’un composé sec à un composé de pluie.

Notre entraînement ne contient donc qu’une seule course illustrant la transition sec → pluie, aux coupures choisies. Cela ne signifie pas que cette transition n’existe nulle part ailleurs dans les historiques complets.

Mathématiquement, parmi les cibles pluie :
$$
\widehat P(\text{pneu actuel sec}\mid Y=\text{WET\_OR\_INTER})
=\frac{18}{35}\approx51{,}43\%.
$$
Attention au sens : ce n’est pas la probabilité de passer aux pneus pluie lorsqu’on roule en pneus secs. Pour cette dernière, le dénominateur doit inclure toutes les voitures actuellement en pneus secs.


</details>

Regardons maintenant la pluie actuelle pour ces $35$ observations :

**Intuition :** nous voulons distinguer « rouler encore en pneus secs alors que la pluie est déjà détectée » de « rouler en pneus secs avant une éventuelle arrivée de pluie ». Ici, `rainfall` décrit le dernier tour visible de chaque voiture ; nous examinerons les informations oracle séparément.

In [36]:
# Croiser le pneu actuel et la pluie au dernier tour visible
wet_weather_summary = (
    wet_targets
    .groupby(
        ["race_id", "compound", "rainfall"],
        dropna=False,
    )
    .size()
    .rename("observations")
    .reset_index()
)

display(wet_weather_summary)

,race_id,compound,rainfall,observations
0,2021_16,INTERMEDIATE,False,17
1,2024_12,HARD,False,1
2,2024_12,MEDIUM,False,15
3,2024_12,MEDIUM,True,1
4,2024_12,SOFT,False,1


<details>
<summary>Deux observations : </summary>

- 2021_16 : les $17$ voitures roulent déjà en intermédiaires, sans pluie détectée sur leur dernier tour visible. Cela ne prouve pas que la piste est sèche : elle peut notamment rester humide après la pluie.
- 2024_12 : les $18$ voitures roulent encore en pneus secs. Une seule a `rainfall`=True, mais toutes ont une cible pluie.

La différence entre pilotes d’une même course mérite attention : leurs derniers tours visibles ne se terminent pas au même instant. Le tableau ne suffit cependant pas à expliquer précisément cette différence de météo.

Nous observons donc :
$$
\widehat P(\texttt{rainfall=True}\mid Y=\text{WET\_OR\_INTER})
=\frac{1}{35}\approx2{,}86\%.
$$
Cela ne signifie pas que la pluie actuelle est inutile. Nous avons sélectionné uniquement les cibles pluie. Pour mesurer son association avec la cible, il faut aussi examiner les autres voitures.
</details>


Calculons maintenant la distribution des cinq cibles selon `rainfall`, sur tout l’entraînement évaluable :

In [37]:
# Compter chaque cible selon la pluie au dernier tour visible
rain_target_counts = pd.crosstab(
    scored_train["rainfall"],
    scored_train["next_compound"],
).reindex(
    index=[False, True],
    columns=CLASSES,
    fill_value=0,
)

# Calculer les proportions conditionnelles à la météo observée
rain_target_pct = (
    rain_target_counts.div(
        rain_target_counts.sum(axis=1).replace(0, np.nan),
        axis=0,
    ) * 100
)

# Mesurer le nombre de voitures et de courses derrière chaque situation
rain_support = (
    scored_train.groupby("rainfall")
    .agg(
        observations=("driver_code", "size"),
        courses=("race_id", "nunique"),
    )
    .reindex([False, True], fill_value=0)
)

display(rain_support)
display(rain_target_counts)
display(rain_target_pct.round(2))

,observations,courses
rainfall,,
False,1262,74
True,34,3


next_compound,SOFT,MEDIUM,HARD,WET_OR_INTER,NONE
rainfall,,,,,
False,188,299,457,34,284
True,2,6,12,1,13


next_compound,SOFT,MEDIUM,HARD,WET_OR_INTER,NONE
rainfall,,,,,
False,14.90,23.69,36.21,2.69,22.50
True,5.88,17.65,35.29,2.94,38.24


<details>
<summary> Voici ce que nous observons :</summary>

| Pluie au dernier tour visible | Observations | Cibles pluie | Proportion |
|---|---:|---:|---:|
| `False` | $1 262$ | $34$ | $2,69 \%$ |
| `True` | $34$ | $1$ | $2,94 \%$ |

Sur les snapshots d’entraînement, la fréquence d’une cible pluie est de $2,94 \%$ lorsque la pluie est détectée au dernier tour visible, contre $2,69 \%$ sinon. Cette comparaison descriptive ne met pas en évidence de forte différence marginale. Toutefois, les observations sous pluie sont rares et concentrées dans trois courses. Elle ne suffit donc pas à écarter cette variable, dont l’intérêt peut dépendre du pneu actuel, de l’historique météo et des informations oracle.


</details>


La prochaine étape sera d’examiner la pluie future autorisée par l’exercice. Nous distinguerons bien la pluie au tour $L+14$ de la présence de pluie à un moment quelconque des $14$ prochains tours.

<details>
<summary> Commençons par la signification exacte de rainfall_Lplus14 : </summary>

 pour une ligne au tour $L$, elle renseigne la pluie au tour $L+14$, pas sur toute la période intermédiaire.

Par exemple :
- au tour $22$, elle décrit le tour $36$ ;
- `False` signifie absence de pluie détectée au tour $36$, pas nécessairement aux tours $23$ à $35$ ;
- une valeur manquante signifie que cette information n’est pas disponible.

Cette information future est explicitement autorisée par l’exercice.

</details>

Nous allons examiner son association avec la cible sur les mêmes $1 296$ snapshots d’entraînement, en distinguant les valeurs manquantes.



>> En notant $F_i$ la valeur oracle de la ligne $i$, nous estimons :
>> $$\widehat P(Y=k\mid F=f) = \frac{\sum_i \mathbf{1}\{Y_i=k,\ F_i=f\}} {\sum_i \mathbf{1}\{F_i=f\}}.$$
>>
>> **Intuition :** parmi les voitures pour lesquelles de la pluie est annoncée au tour $L+14$, quelle proportion monte ensuite des pneus pluie ?

In [38]:
# Reprendre explicitement notre population d'entraînement évaluable
scored_train_snapshots = eda_train.loc[
    eda_train["scored"]
].copy()

# Distinguer absence de pluie, pluie et information indisponible
oracle_rain_state = (
    scored_train_snapshots["rainfall_Lplus14"]
    .astype("boolean")
    .astype("string")
    .fillna("UNKNOWN")
)

oracle_states = ["False", "True", "UNKNOWN"]

# Compter les cibles pour chaque état de l'information oracle
oracle_target_counts = pd.crosstab(
    oracle_rain_state,
    scored_train_snapshots["next_compound"],
).reindex(
    index=oracle_states,
    columns=CLASSES,
    fill_value=0,
)

oracle_target_counts.index.name = "rain_at_Lplus14"

# Calculer les pourcentages conditionnels
oracle_target_pct = (
    oracle_target_counts.div(
        oracle_target_counts.sum(axis=1).replace(0, np.nan),
        axis=0,
    ) * 100
)

# Mesurer les effectifs et le nombre de courses représentées
oracle_support = (
    scored_train_snapshots
    .assign(oracle_rain_state=oracle_rain_state)
    .groupby("oracle_rain_state")
    .agg(
        observations=("driver_code", "size"),
        courses=("race_id", "nunique"),
    )
    .reindex(oracle_states, fill_value=0)
)

display(oracle_support)
display(oracle_target_counts)
display(oracle_target_pct.round(2))

,observations,courses
oracle_rain_state,,
False,1245,73
True,51,3
UNKNOWN,0,0


next_compound,SOFT,MEDIUM,HARD,WET_OR_INTER,NONE
rain_at_Lplus14,,,,,
False,188,299,457,17,284
True,2,6,12,18,13
UNKNOWN,0,0,0,0,0


next_compound,SOFT,MEDIUM,HARD,WET_OR_INTER,NONE
rain_at_Lplus14,,,,,
False,15.10,24.02,36.71,1.37,22.81
True,3.92,11.76,23.53,35.29,25.49
UNKNOWN,NaN,NaN,NaN,NaN,NaN


<details>
<summary> L’association avec la cible pluie est ici beaucoup plus marquée qu’avec la pluie actuelle.</summary>

En notant $F=\texttt{rainfall\_Lplus14}$ :
$$
\widehat P(Y=\text{WET\_OR\_INTER}\mid F=1)
=\frac{18}{51}\approx35{,}29\%,
$$
contre :
$$
\widehat P(Y=\text{WET\_OR\_INTER}\mid F=0)
=\frac{17}{1245}\approx1{,}37\%.
$$
**Intuition :** savoir qu’il pleuvra au tour $L+14$ semble apporter une information utile sur le prochain composé. Cependant, cela ne détermine pas automatiquement la cible :
- $33$ des $51$ voitures avec $F$=`True` ont une autre cible : leur prochain arrêt peut notamment précéder la pluie ;
- $17$ voitures avec $F$ = `False` ont une cible pluie : l’absence de pluie à cet instant précis ne décrit ni toute la météo intermédiaire ni l’état de la piste.
Ce sont des explications possibles, à vérifier dans les historiques.

**Restons prudents :** les $51$ observations avec $F$ = `True` proviennent de seulement $3$ courses. Nous observons une association dans l’entraînement, pas encore une amélioration prédictive démontrée sur d’autres courses.

Enfin, les `NaN` de la ligne `UNKNOWN` sont normaux : aucune observation n’appartient à ce groupe, donc ses proportions correspondent à $0/0$.
</details>

>La fréquence d’une cible pluie atteint $35,29 \%$ lorsque l’oracle annonce de la pluie au tour $L+14$, contre $1,37 \%$ sinon. Cette association motive l’utilisation de l’information oracle, mais sa généralisation reste à évaluer par course compte tenu du faible nombre de situations représentées.

Pour identifier ces trois courses et vérifier la concentration du signal :

In [39]:
# Décrire les courses pour lesquelles l'oracle annonce de la pluie
oracle_rain_by_race = (
    scored_train_snapshots.loc[
        scored_train_snapshots["rainfall_Lplus14"].eq(True).fillna(False)
    ]
    .groupby("race_id")
    .agg(
        observations=("driver_code", "size"),
        wet_targets=(
            "next_compound",
            lambda values: values.eq("WET_OR_INTER").sum(),
        ),
    )
)

oracle_rain_by_race["wet_target_percentage"] = (
    100
    * oracle_rain_by_race["wet_targets"]
    / oracle_rain_by_race["observations"]
)

display(oracle_rain_by_race.round(2))

,observations,wet_targets,wet_target_percentage
race_id,,,
2018_05,14,0,0.0
2019_06,19,0,0.0
2024_12,18,18,100.0


<details>
<summary> L’association observée repose entièrement sur une seule course : 2024_12 </summary>

Parmi les voitures avec `rainfall_Lplus14` = `True` :
- 2018_05 : $0$ cible pluie sur $14$ voitures ;
- 2019_06 : $0$ sur $19$ ;
- 2024_12 : $18$ sur $18$.
Le taux global de $35,29 \%$ est donc :
$$
\frac{0+0+18}{14+19+18}
=\frac{18}{51}
\approx35{,}29\%.
$$

**Intuition :** ce pourcentage mélange trois situations très différentes. Il ne signifie pas que, dans chacune de ces courses, environ un tiers des voitures choisit des pneus pluie.

**Cela confirme deux choses :**

1. La pluie au tour $L+14$ ne suffit pas à déterminer le prochain pneu. Il faut également considérer le moment du prochain arrêt, le pneu actuel et l’évolution des conditions.
2. Notre exemple de transition sec → pluie est concentré dans une seule course. Si cette course est absente d’un pli d’apprentissage, le modèle ne disposera plus de ce type de transition dans les snapshots utilisés.

Nous conservons donc l’oracle comme variable candidate, mais ce tableau ne démontre pas encore sa capacité à généraliser.

</details>

>L’association globale entre pluie annoncée à $L+14$ et cible pluie est portée par la course 2024_12. Les deux autres courses présentant cette information oracle ne comportent aucune cible pluie aux coupures retenues. Cette hétérogénéité justifie une validation par course et invite à décrire l’évolution de la météo plutôt qu’un unique instant futur.

<details>
<summary> La suite consiste à reconstruire la pluie annoncée sur les 14 prochains tours à partir des lignes déjà visibles. </summary>

Par exemple, lorsque le leader vient de terminer le tour $28$ :

| Ligne visible au tour… | Son oracle `Lplus14` décrit le tour… |
|---:|---:|
| $15$ | $29$ |
| $16$ | $30$ |
| … | … |
| $28$ | $42$ |

Nous pouvons ainsi obtenir une séquence pour les tours $29$ à $42$, sous réserve que les valeurs soient disponibles. Cela permettra de distinguer une pluie annoncée prochainement d’une pluie annoncée seulement en fin d’horizon, sans consulter les lignes futures cachées.


Construisons cette séquence sur notre exemple 2018_01, coupé au tour $28$ du leader, avant de généraliser.

Notons $L=28$ et $h\in\{1,\ldots,14\}$ le nombre de tours après celui du leader. Pour connaître la pluie au tour $L+h$, nous lisons l’oracle de la ligne :
$$
\ell=L+h-14,
\qquad\text{car}\qquad \ell+14=L+h.
$$
Par exemple, pour le tour suivant ($h=1$) :
$$
\ell=28+1-14=15.
$$

**Pourquoi vérifier la concordance ?** Plusieurs pilotes fournissent une ligne pour le même numéro de tour. Nous vérifions leurs valeurs oracle avant d’en retenir une. Si aucune valeur n’est disponible, nous gardons `NA` : inconnu ne signifie pas absence de pluie.
</details> 

L’oracle du tour $15$ renseigne donc le tour $29$. Nous n’utilisons aucune ligne du tour $29$ lui-même.

In [40]:
# Reprendre les tours terminés avant la coupure de notre exemple
example_race_id = "2018_01"

example_cutoff = cutoff_summary.loc[
    cutoff_summary["race_id"].eq(example_race_id)
].iloc[0]

leader_lap = int(example_cutoff["cutoff_lap"])
example_cutoff_time = example_cutoff["cutoff_time_s"]

example_visible = train_data.loc[
    train_data["race_id"].eq(example_race_id)
    & train_data["race_time_s"].notna()
    & train_data["race_time_s"].le(example_cutoff_time)
].copy()

# Identifier les 14 tours sources nécessaires
source_laps = pd.Index(
    range(leader_lap - 13, leader_lap + 1),
    name="source_lap",
)

oracle_source = example_visible.loc[
    example_visible["lap_number"].isin(source_laps)
]

# Vérifier que les valeurs connues concordent entre pilotes au même tour
known_values_per_lap = (
    oracle_source.groupby("lap_number")["rainfall_Lplus14"]
    .nunique(dropna=True)
)

assert known_values_per_lap.le(1).all(), (
    "Des valeurs oracle contradictoires existent pour un même tour."
)

# Extraire la valeur connue de chaque tour, en conservant les absences
rain_by_source_lap = (
    oracle_source.groupby("lap_number")["rainfall_Lplus14"]
    .first()
    .reindex(source_laps)
    .astype("boolean")
)

# Présenter la correspondance entre tour source et tour futur
future_rain_timeline = pd.DataFrame({
    "offset": range(1, 15),
    "source_lap": source_laps.to_numpy(),
    "future_lap": source_laps.to_numpy() + 14,
    "rain": rain_by_source_lap.array,
})

display(future_rain_timeline)

,offset,source_lap,future_lap,rain
0,1,15,29,False
1,2,16,30,False
2,3,17,31,False
3,4,18,32,False
4,5,19,33,False
5,6,20,34,False
6,7,21,35,False
7,8,22,36,False
8,9,23,37,False
9,10,24,38,False


<details>
<summary>Les 14 valeurs sont connues et toutes valent False : </summary>

Aucune pluie n’est annoncée aux tours $29$ à $42$. Cela ne signifie pas nécessairement que la piste est sèche, ni qu’il ne pleuvra pas après le tour $42$.

Nous pouvons maintenant résumer cette séquence par trois features :

| Feature | Sens | Valeur ici |
|---|---|---:|
| `future_rain_known_laps` | Nombre de tours futurs renseignés | $14$ |
| `future_rain_laps` | Nombre de tours renseignés avec pluie | $0$ |
| `future_rain_first` | Délai jusqu’au premier tour avec pluie | $15$, par convention |

**Pourquoi 15 ?** Notre horizon s’arrête à $14$. La valeur $15$ signifie donc « aucune pluie dans l’horizon entièrement connu », et non « il pleuvra au quinzième tour ».

</details>

Pour les données incomplètes, soyons précis : si une valeur inconnue précède la première pluie détectée, nous ne connaissons pas le véritable premier tour de pluie.

**Intuition :** le nombre de tours de pluie décrit sa présence dans l’horizon ; le premier décalage décrit sa proximité ; le nombre de tours connus permet de distinguer « zéro pluie sur $14$ tours renseignés » de « zéro pluie sur seulement quelques tours renseignés ».

In [41]:
# Extraire la séquence avec les décalages comme index
rain = future_rain_timeline.set_index("offset")["rain"].astype("boolean")

# Compter les tours connus et les tours avec pluie observée dans l'oracle
future_rain_known_laps = int(rain.notna().sum())
future_rain_laps = rain.sum(min_count=1)

# Identifier le premier décalage connu avec pluie
rain_offsets = rain.index[rain.fillna(False)]

if len(rain_offsets) > 0:
    first_rain_offset = int(rain_offsets.min())

    # Le premier événement est certain seulement si tout ce qui précède est connu
    preceding_values = rain.loc[rain.index < first_rain_offset]

    future_rain_first = (
        first_rain_offset
        if preceding_values.notna().all()
        else np.nan
    )

elif rain.notna().all():
    # Aucun événement dans un horizon entièrement connu
    future_rain_first = 15

else:
    # L'absence de pluie ne peut pas être établie sur tout l'horizon
    future_rain_first = np.nan

rain_features = pd.DataFrame([{
    "future_rain_laps": future_rain_laps,
    "future_rain_known_laps": future_rain_known_laps,
    "future_rain_first": future_rain_first,
}])

display(rain_features)

,future_rain_laps,future_rain_known_laps,future_rain_first
0,0,14,15


Examinons maintenant 2024_12, la course où les $18$ voitures évaluables avaient une cible pluie. Nous reconstruisons uniquement la séquence oracle disponible à sa coupure, sans consulter les lignes futures.

**La question est maintenant :** la pluie est-elle annoncée dès le prochain tour, ou plus tard dans l’horizon ? Le tableau permettra de le lire directement avant de calculer les trois features.

In [42]:
# Sélectionner un exemple avec des transitions des pneus secs vers la pluie
wet_race_id = "2024_12"

wet_cutoff = cutoff_summary.loc[
    cutoff_summary["race_id"].eq(wet_race_id)
].iloc[0]

wet_leader_lap = int(wet_cutoff["cutoff_lap"])
wet_cutoff_time = wet_cutoff["cutoff_time_s"]

# Conserver uniquement les lignes visibles à cet instant
wet_visible = train_data.loc[
    train_data["race_id"].eq(wet_race_id)
    & train_data["race_time_s"].notna()
    & train_data["race_time_s"].le(wet_cutoff_time)
].copy()

# Identifier les tours sources qui décrivent les 14 prochains tours
wet_source_laps = pd.Index(
    range(wet_leader_lap - 13, wet_leader_lap + 1),
    name="source_lap",
)

wet_oracle_source = wet_visible.loc[
    wet_visible["lap_number"].isin(wet_source_laps)
]

# Vérifier la concordance des valeurs connues entre pilotes
assert (
    wet_oracle_source.groupby("lap_number")["rainfall_Lplus14"]
    .nunique(dropna=True)
    .le(1)
    .all()
), "Des valeurs oracle contradictoires existent pour un même tour."

# Extraire la séquence en conservant les éventuelles valeurs manquantes
wet_rain = (
    wet_oracle_source.groupby("lap_number")["rainfall_Lplus14"]
    .first()
    .reindex(wet_source_laps)
    .astype("boolean")
)

wet_rain_timeline = pd.DataFrame({
    "offset": range(1, 15),
    "source_lap": wet_source_laps.to_numpy(),
    "future_lap": wet_source_laps.to_numpy() + 14,
    "rain": wet_rain.array,
})

print("Tour du leader à la coupure :", wet_leader_lap)
display(wet_rain_timeline)

Tour du leader à la coupure : 18


,offset,source_lap,future_lap,rain
0,1,5,19,True
1,2,6,20,True
2,3,7,21,True
3,4,8,22,True
4,5,9,23,True
5,6,10,24,True
6,7,11,25,True
7,8,12,26,True
8,9,13,27,True
9,10,14,28,True


<details>
<summary> La pluie est annoncée dès le prochain tour du leader, puis sur chacun des 14 tours de l’horizon : du tour 19 au tour 32. </summary>

Nous pouvons donc lire directement nos trois features :

| Feature | Valeur | Interprétation |
|---|---:|---|
| `future_rain_laps` | $14$ | Les $14$ tours renseignés annoncent de la pluie |
| `future_rain_known_laps` | $14$ | L’horizon est entièrement renseigné |
| `future_rain_first` | $1$ | Le premier tour avec pluie est le prochain |

Mathématiquement, si $R_h$ indique la pluie au décalage $h$ :
$$
\sum_{h=1}^{14}R_h=14,
\qquad
\min\{h:R_h=1\}=1.
$$
Cela complète la pluie actuelle : pour cette course, $17$ des $18$ voitures évaluables avaient `rainfall` = `False` au dernier tour visible, mais l’oracle annonce de la pluie dès le prochain tour du leader.

**Nous distinguons ainsi :**
- l’état observé au dernier tour visible de chaque voiture ;
- les conditions futures autorisées, sur un horizon commun défini par rapport au leader.

**Attention :** `future_rain_first` $= 1$ signifie un tour de décalage, pas une minute, et ne prouve pas que la pluie commence précisément à ce moment. Elle peut déjà être présente à la coupure.
</details>


>À la coupure du tour $18$ de `2024_12`, l’oracle indique de la pluie aux tours $19$ à $32$. Les features correspondantes sont donc $(14, 14, 1)$ : quatorze tours avec pluie, quatorze tours connus et une première occurrence au décalage $1$. Cette séquence apporte une information que la seule météo du dernier tour visible ne résume pas.

La prochaine étape sera de regrouper ce calcul dans une fonction réutilisable.

In [43]:
"""
Créons une fonction qui calcule ces trois features pour une course à une coupure donnée.

Elle recevra uniquement les tours visibles et le numéro du tour du leader. Elle vérifiera aussi que les valeurs oracle connues concordent entre pilotes pour un même tour.
"""

def build_rain_features(visible, leader_lap):
    """Résume la pluie oracle sur les 14 prochains tours du leader."""

    # Vérifier que les données concernent une seule course
    if visible.empty or visible["race_id"].nunique() != 1:
        raise ValueError("Une seule course non vide est attendue.")

    # Les tours L-13 à L décrivent les tours futurs L+1 à L+14
    source_laps = pd.Index(
        range(leader_lap - 13, leader_lap + 1),
        name="source_lap",
    )

    oracle_source = visible.loc[
        visible["lap_number"].isin(source_laps)
    ]

    # Vérifier que les valeurs connues concordent pour chaque tour
    known_values = (
        oracle_source.groupby("lap_number")["rainfall_Lplus14"]
        .nunique(dropna=True)
    )

    if known_values.gt(1).any():
        raise ValueError("Valeurs oracle contradictoires pour un même tour.")

    # Extraire les valeurs disponibles sans remplacer les inconnues par False
    rain = (
        oracle_source.groupby("lap_number")["rainfall_Lplus14"]
        .first()
        .reindex(source_laps)
        .astype("boolean")
    )

    rain.index = pd.RangeIndex(1, 15, name="offset")

    # Compter séparément les tours connus et les tours avec pluie
    known_laps = int(rain.notna().sum())
    rain_laps = rain.sum(min_count=1)

    # Rechercher la première pluie dont le décalage est établi
    rain_offsets = rain.index[rain.fillna(False)]

    if len(rain_offsets) > 0:
        first_offset = int(rain_offsets.min())
        preceding_values = rain.loc[rain.index < first_offset]

        first_rain = (
            first_offset
            if preceding_values.notna().all()
            else np.nan
        )

    elif rain.notna().all():
        # Aucun tour avec pluie dans l'horizon entièrement connu
        first_rain = 15

    else:
        # Des inconnues empêchent de conclure sur la première pluie
        first_rain = np.nan

    return pd.DataFrame([{
        "future_rain_laps": rain_laps,
        "future_rain_known_laps": known_laps,
        "future_rain_first": first_rain,
    }])

Pourquoi fournir `leader_lap` explicitement ? L’horizon est défini par la coupure choisie. Nous évitons de le déduire d’une table qui pourrait avoir été filtrée sur quelques pilotes.

Vérifions maintenant les deux exemples étudiés :

In [44]:
# Vérifier la course sans pluie annoncée
dry_check = build_rain_features(example_visible, leader_lap)

# Vérifier la course avec pluie sur tout l'horizon
wet_check = build_rain_features(wet_visible, wet_leader_lap)

comparison = pd.concat(
    [
        dry_check.assign(race_id="2018_01"),
        wet_check.assign(race_id="2024_12"),
    ],
    ignore_index=True,
).set_index("race_id")

display(comparison)

assert dry_check.iloc[0].tolist() == [0, 14, 15]
assert wet_check.iloc[0].tolist() == [14, 14, 1]

print("La fonction reproduit les deux exemples étudiés.")

,future_rain_laps,future_rain_known_laps,future_rain_first
race_id,,,
2018_01,0,14,15
2024_12,14,14,1


La fonction reproduit les deux exemples étudiés.


La fonction renvoie une ligne par course–coupure. Nous associerons ensuite ces mêmes valeurs aux voitures de cette coupure. Elle suppose que son argument visible a déjà été filtré avec `race_time_s` $<=$ `cutoff_time` : elle ne consulte pas elle-même la course complète.

>La fonction reproduit bien nos deux exemples. Appliquons-la maintenant aux $76$ courses d’entraînement, en conservant exactement leurs coupures déjà tirées.

>Nous calculons une ligne de météo future par course, puis nous la joignons aux voitures de cette course. Aucun nouveau tirage n’est nécessaire.

>**Pourquoi** `many_to_one` ? Plusieurs voitures appartiennent à une même course, mais chacune doit recevoir une seule ligne de features météo. Cette vérification empêche une jointure qui dupliquerait accidentellement les observations.

>Le tableau final nous dira combien de courses disposent de $14$ tours futurs renseignés, et combien ont un horizon incomplet. Nous vérifierons cette couverture avant d’interpréter les nouvelles features.

In [45]:
# Reprendre les coupures des seules courses d'entraînement
train_cutoffs = cutoff_summary.loc[
    cutoff_summary["race_id"].isin(eda_train["race_id"].unique())
]

rain_parts = []

for cutoff in train_cutoffs.itertuples(index=False):

    # Lire uniquement les tours terminés avant ou à cette coupure
    visible = train_data.loc[
        train_data["race_id"].eq(cutoff.race_id)
        & train_data["race_time_s"].notna()
        & train_data["race_time_s"].le(cutoff.cutoff_time_s)
    ].copy()

    # Calculer les trois features communes à cette course-coupure
    rain_part = build_rain_features(
        visible,
        int(cutoff.cutoff_lap),
    )

    rain_part["race_id"] = cutoff.race_id
    rain_parts.append(rain_part)

rain_features_train = pd.concat(rain_parts, ignore_index=True)

# Associer les features aux voitures, sans modifier la table initiale
eda_train_with_rain = eda_train.merge(
    rain_features_train,
    on="race_id",
    how="left",
    validate="many_to_one", # # Plusieurs voitures pour une seule ligne météo
)

# Vérifier que la jointure conserve toutes les observations
assert len(rain_features_train) == 76
assert len(eda_train_with_rain) == len(eda_train)

print("Courses avec features pluie :", len(rain_features_train))

# Examiner la disponibilité de l'information, une fois par course
display(
    rain_features_train["future_rain_known_laps"]
    .value_counts()
    .sort_index()
    .rename_axis("known_future_laps")
    .to_frame("courses")
)

Courses avec features pluie : 76


,courses
known_future_laps,
14,76


<details>
<summary> Les 76 courses d’entraînement disposent toutes des 14 valeurs oracle attendues.</summary>

Aucun horizon pluie n’est incomplet à ces coupures.

Cela implique que :
- `future_rain_laps` compte ici la pluie sur l’horizon complet ;
- `future_rain_first` est connu : de $1$ à $14$, ou $15$ si aucune pluie n’est annoncée ;
- `future_rain_known_laps` vaut toujours $14$ : elle ne distingue donc aucune observation dans cet entraînement. Nous la conservons pour contrôler la disponibilité des données.

</details> 

Regardons maintenant comment les séquences se répartissent par course, sans compter plusieurs fois la même météo pour chaque voiture :

**Intuition :** deux courses peuvent annoncer le même nombre de tours de pluie, mais avec un délai différent. Trois tours de pluie à partir du prochain tour ne décrivent pas la même situation que trois tours de pluie seulement en fin d’horizon.

Ces deux features résument donc combien de tours sont concernés et quand survient la première pluie, sans décrire à elles seules toute la séquence.

In [46]:
# Compter les courses présentant chaque résumé de pluie future
rain_pattern_summary = (
    rain_features_train
    .groupby(
        ["future_rain_laps", "future_rain_first"],
        dropna=False,
    )
    .size()
    .rename("courses")
    .reset_index()
    .sort_values(["future_rain_laps", "future_rain_first"])
)

display(rain_pattern_summary)

,future_rain_laps,future_rain_first,courses
0,0,15,71
1,3,1,1
2,7,1,1
3,13,1,1
4,14,1,2


<details>
<summary> Sur les 76 courses d’entraînement, 5 annoncent de la pluie dans les 14 prochains tours, contre 71 sans pluie annoncée dans cet horizon.</summary>

$$
\frac{5}{76}\times100\approx6{,}58\%.
$$
Pour ces cinq courses, le prochain tour du leader est déjà annoncé pluvieux : `future_rain_first` $= 1$. Le nombre total de tours avec pluie varie :$ 3, 7, 13$ ou $14$.

**Attention :** $3$ tours avec pluie ne signifie pas nécessairement les trois premiers tours consécutifs. Nos features donnent le nombre total et la première occurrence, mais pas toutes les interruptions éventuelles.

Ce tableau révèle également une redondance dans cet échantillon :
$$
\texttt{future\_rain\_first}=
\begin{cases}
15 & \text{si }\texttt{future\_rain\_laps}=0,\\
1 & \text{si }\texttt{future\_rain\_laps}>0.
\end{cases}
$$

Ici, le délai jusqu’à la première pluie n’apporte donc aucune distinction supplémentaire entre les courses annonçant de la pluie. Ce constat concerne nos coupures d’entraînement ; ce n’est pas une propriété générale de la météo.
</details>

Regardons maintenant quelles sont ces cinq courses et leurs cibles :

L’objectif est de vérifier si l’horizon complet repère des situations que la seule valeur à $L+14$ ne montrait pas, puis d’examiner leur association avec le prochain pneu.

In [47]:
# Compter les observations évaluables et les cibles pluie par course
rain_course_targets = (
    eda_train_with_rain.loc[eda_train_with_rain["scored"]]
    .groupby("race_id")
    .agg(
        scored_cars=("driver_code", "size"),
        wet_targets=(
            "next_compound",
            lambda values: values.eq("WET_OR_INTER").sum(),
        ),
    )
    .reset_index()
)

# Associer les cibles aux résumés météo des courses avec pluie annoncée
rain_course_diagnostics = (
    rain_features_train.loc[
        rain_features_train["future_rain_laps"].gt(0)
    ]
    .merge(
        rain_course_targets,
        on="race_id",
        how="left",
        validate="one_to_one",
    )
)

display(rain_course_diagnostics)

,future_rain_laps,future_rain_known_laps,future_rain_first,race_id,scored_cars,wet_targets
0,13,14,1,2018_05,14,0
1,14,14,1,2019_06,19,0
2,7,14,1,2024_09,15,0
3,14,14,1,2024_12,18,18
4,3,14,1,2026_12,16,0


<details>
<summary>L’horizon complet repère cinq courses avec de la pluie annoncée, mais une seule comporte des cibles pluie à la coupure : 2024_12. </summary>

Le contraste le plus instructif est :
| Course | Tours avec pluie annoncée | Cibles pluie |
|---|---:|---:|
| `2019_06` | $14$ sur $14$ | $0$ sur $19$ |
| `2024_12` | $14$ sur $14$ | $18$ sur $18$ |

Ces deux courses ont exactement les mêmes trois features oracle, mais des cibles très différentes. Ces features ne suffisent donc pas à expliquer le prochain composé.

Cela n’implique pas qu’elles soient inutiles : elles peuvent devenir informatives lorsqu’on les combine au pneu actuel, à son âge, à l’historique des arrêts et aux conditions déjà observées. Nous devrons le vérifier par validation.

Sur les cinq courses réunies :
$$
\widehat P\!\left(
Y=\text{WET\_OR\_INTER}
\mid \texttt{future\_rain\_laps}>0
\right)
=
\frac{18}{14+19+15+18+16}
=\frac{18}{82}
\approx21{,}95\%.
$$
Mais ce taux reste entièrement porté par une seule course.

</details>

Pour comprendre le contraste, regardons le pneu actuel et le prochain composé dans ces cinq courses :

**Intuition :** connaître la météo future ne dit pas à quel moment aura lieu le prochain arrêt. La cible peut concerner un arrêt avant, pendant ou après cet horizon. Nous allons donc examiner les transitions observées sans supposer que « pluie annoncée » implique automatiquement « prochain pneu pluie ».

In [48]:
# Examiner les transitions de pneus lorsque l'oracle annonce de la pluie
rain_transition_summary = (
    eda_train_with_rain.loc[
        eda_train_with_rain["scored"]
        & eda_train_with_rain["future_rain_laps"].gt(0)
    ]
    .groupby(
        ["race_id", "compound", "next_compound"],
        dropna=False,
    )
    .size()
    .rename("observations")
    .reset_index()
)

display(rain_transition_summary)

,race_id,compound,next_compound,observations
0,2018_05,HARD,HARD,3
1,2018_05,HARD,MEDIUM,1
2,2018_05,MEDIUM,HARD,8
3,2018_05,MEDIUM,NONE,1
4,2018_05,SOFT,HARD,1
5,2019_06,HARD,NONE,7
6,2019_06,MEDIUM,MEDIUM,1
7,2019_06,MEDIUM,NONE,5
8,2019_06,MEDIUM,SOFT,2
9,2019_06,SOFT,MEDIUM,4


<details>
<summary> Ce tableau distingue trois situations malgré la présence de pluie annoncée. </summary>

| Situation | Courses | Ce que nous observons |
|---|---|---|
| Sec → sec ou aucun nouvel arrêt | `2018_05`, `2019_06`, `2026_12` | Aucun prochain composé pluie |
| Intermédiaire → sec | `2024_09` | $4$ voitures passent en `HARD` et $11$ en `MEDIUM` |
| Sec → pluie | `2024_12` | Les $18$ voitures passent en `WET_OR_INTER` |

Le cas `2024_09` est particulièrement instructif : les voitures sont déjà en intermédiaires, l’oracle annonce $7$ tours avec pluie sur $14$, mais leur prochain composé est sec. Ce n’est pas nécessairement contradictoire : le prochain arrêt peut survenir après une évolution des conditions. Le tableau ne donne toutefois ni son instant ni l’état de la piste à cet instant.

De même, pour les trois courses où les pneus restent secs, nous ne pouvons pas conclure que la pluie était faible : `rainfall` ne mesure pas son intensité.

L’idée pour le modèle est donc d’estimer une probabilité conditionnelle combinant plusieurs informations :
$$
P\!\left(
Y=k\mid
\text{pneu actuel},
\text{pluie actuelle},
\text{pluie future},
\text{âge du pneu},
\ldots
\right),
$$
plutôt que d’appliquer une règle « pluie future ⇒ pneu pluie ».

</details>

>Aux coupures étudiées, la présence de pluie dans l’horizon oracle coexiste avec des transitions sec–sec, intermédiaire–sec et sec–pluie. La météo future ne détermine donc pas seule le prochain composé. Nous conservons les résumés de pluie comme variables candidates à combiner avec l’état de la voiture, tout en soulignant que la transition sec–pluie n’est représentée que par une course d’entraînement.

Nous avons suffisamment exploré ce point pour justifier les features de pluie future. La prochaine étape sera de définir proprement la pluie actuelle et son historique visible, afin de compléter ces informations sans consulter le futur caché.

<details>
<summary>Commençons par une seule variable :</summary>

`rain_now`, la pluie renseignée au dernier tour visible de chaque voiture.

Pour le pilote $d$, notons $L_d(T)$ son dernier tour visible à la coupure. Nous définissons :
$$
\texttt{rain\_now}_d
=\texttt{rainfall}_{d,L_d(T)}.
$$

**Intuition :** c’est la dernière information météo disponible sur la ligne de cette voiture. Ce n’est pas nécessairement une mesure prise exactement à $T$, puisque les voitures terminent leurs tours à des instants différents.

Nous avons déjà sélectionné cette ligne dans `eda_train_with_rain`. Il suffit donc de reprendre sa colonne `rainfall` dans le code. 

Nous attendons les effectifs déjà observés : $1 262$ False et $34$ True, sans valeur manquante à la sortie.

Pourquoi garder cette variable en plus de l’oracle ? Elles décrivent des moments différents :

| Variable | Information |
|---|---|
| `rain_now` | Dernière pluie observée pour la voiture |
| `future_rain_laps` | Nombre de tours avec pluie dans l’horizon futur renseigné |
| `future_rain_first` | Premier décalage futur avec pluie, lorsqu’il est déterminable |

Le nom `rain_now` rend son rôle explicite, mais n’apporte aucune information supplémentaire par rapport à rainfall. Dans les features du modèle, nous utiliserons une seule des deux colonnes.
</details> 

In [49]:
# Conserver la dernière observation météo visible de chaque voiture
eda_train_with_rain["rain_now"] = (
    eda_train_with_rain["rainfall"].astype("boolean")
)

# Contrôler sa disponibilité dans la population évaluable
rain_now_summary = (
    eda_train_with_rain.loc[
        eda_train_with_rain["scored"],
        "rain_now",
    ]
    .value_counts(dropna=False)
    .rename_axis("rain_now")
    .to_frame("observations")
)

display(rain_now_summary)

,observations
rain_now,
False,1262
True,34


<details>
<summary>Passons à l’historique avec une variable candidate : </summary>

`rain_share_so_far`, la proportion des tours visibles renseignés pendant lesquels de la pluie a été détectée.

Pour une voiture $d$, notons $\mathcal V_d(T)$ ses tours visibles et $R_{d,\ell}$ l’indicateur de pluie :
$$
\texttt{rain\_share\_so\_far}_d
=
\frac{
\sum_{\ell\in\mathcal V_d(T)}
\mathbf 1\{R_{d,\ell}=\mathrm{True}\}
}{
\sum_{\ell\in\mathcal V_d(T)}
\mathbf 1\{R_{d,\ell}\text{ est renseigné}\}
}.
$$
**Intuition :** une voiture peut avoir `rain_now = False`, mais avoir connu de la pluie auparavant. Cette proportion conserve une trace de cet historique. Elle mesure une part de tours, pas une durée de pluie ni l’humidité de la piste.

Calculons-la d’abord sur 2024_09, où les voitures étaient en intermédiaires à la coupure
</details>

In [50]:
# Sélectionner la coupure déjà tirée pour cette course d'entraînement
history_race_id = "2024_09"

history_cutoff = train_cutoffs.loc[
    train_cutoffs["race_id"].eq(history_race_id)
].iloc[0]

# Conserver uniquement les tours visibles à cette coupure
history_visible = train_data.loc[
    train_data["race_id"].eq(history_race_id)
    & train_data["race_time_s"].notna()
    & train_data["race_time_s"].le(history_cutoff["cutoff_time_s"])
].copy()

# Résumer l'historique de pluie par voiture
rain_history = (
    history_visible.groupby("driver_code")
    .agg(
        visible_laps=("lap_number", "size"),
        rain_known_laps=("rainfall", "count"),
        rain_observed_laps=(
            "rainfall",
            lambda values: values.sum(min_count=1),
        ),
    )
)

# Ne pas interpréter une absence totale d'information comme zéro pluie
rain_history["rain_share_so_far"] = (
    rain_history["rain_observed_laps"]
    / rain_history["rain_known_laps"].replace(0, np.nan)
)

# Comparer l'historique et la dernière observation des voitures retenues
history_comparison = (
    eda_train_with_rain.loc[
        eda_train_with_rain["race_id"].eq(history_race_id),
        ["driver_code", "lap_number", "compound", "rain_now"],
    ]
    .merge(
        rain_history,
        left_on="driver_code",
        right_index=True,
        how="left",
        validate="one_to_one",
    )
)

display(history_comparison)

,driver_code,lap_number,compound,rain_now,visible_laps,rain_known_laps,rain_observed_laps,rain_share_so_far
972,ALB,27,INTERMEDIATE,False,27,27,9,0.333333
973,ALO,27,INTERMEDIATE,False,27,27,9,0.333333
974,BOT,27,INTERMEDIATE,False,27,27,9,0.333333
975,GAS,27,INTERMEDIATE,False,27,27,9,0.333333
976,HAM,27,INTERMEDIATE,False,27,27,9,0.333333
977,HUL,27,INTERMEDIATE,False,27,27,9,0.333333
978,LEC,27,INTERMEDIATE,False,27,27,9,0.333333
979,MAG,27,INTERMEDIATE,False,27,27,9,0.333333
980,NOR,27,INTERMEDIATE,False,27,27,9,0.333333
981,OCO,27,INTERMEDIATE,False,27,27,9,0.333333


Nous distinguons donc :
- `rain_now=False` : aucune pluie détectée au dernier tour visible ;
- `rain_share_so_far`$≈0,33$ : de la pluie a été détectée sur environ un tiers des tours visibles ;
- `compound=INTERMEDIATE` : les voitures utilisent encore des intermédiaires.

Ces observations peuvent coexister. Elles ne permettent toutefois pas de mesurer directement l’humidité de la piste.

Le tableau contient $19$ voitures retenues, alors que notre analyse précédente de cette course portait sur $15$ voitures évaluables : ici, nous décrivons aussi les voitures dont la cible n’est pas évaluée.

**Il nous manque maintenant la chronologie : quand ces neuf tours avec pluie ont-ils eu lieu ? Une pluie ancienne et une pluie récente peuvent donner exactement la même proportion.**

Examinons un pilote, sans supposer que sa séquence représente automatiquement tous les autres :
 >Cette lecture nous permettra de justifier ensuite une éventuelle mesure de pluie récente, plutôt que d’ajouter arbitrairement des variables.

In [51]:
# Examiner la chronologie visible d'un pilote de cette course
example_driver = "ALB"

driver_rain_history = (
    history_visible.loc[
        history_visible["driver_code"].eq(example_driver),
        ["lap_number", "rainfall", "compound"],
    ]
    .sort_values("lap_number")
)

display(driver_rain_history)

,lap_number,rainfall,compound
73087,1,True,INTERMEDIATE
73088,2,True,INTERMEDIATE
73089,3,True,INTERMEDIATE
73090,4,True,INTERMEDIATE
73091,5,True,INTERMEDIATE
73092,6,True,INTERMEDIATE
73093,7,True,INTERMEDIATE
73094,8,True,INTERMEDIATE
73095,9,True,INTERMEDIATE
73096,10,False,INTERMEDIATE


Pour ALB, les neuf tours avec pluie sont les tours $1$ à $9$. Les tours $10$ à $27$ indiquent ensuite False.

À la coupure, nous avons donc :
- `rain_now = False` ;
- `rain_share_so_far` $= 9/27 ≈ 33,33 \%$ ;
- $18$ tours consécutifs renseignés sans pluie, tout en restant en intermédiaires.

Cela montre la limite de la proportion globale : elle ne distingue pas neuf tours pluvieux au début de neuf tours pluvieux juste avant la coupure.

<details>
<summary> Nous pouvons proposer une variable candidate :</summary>

`rain_share_recent`, la proportion de tours avec pluie parmi les $w$ derniers tours, en ne comptant au dénominateur que les valeurs connues.

Pour un dernier tour visible $L$ :
$$
\texttt{rain\_share\_recent}(w)
=
\frac{
\sum_{\ell=L-w+1}^{L}\mathbf 1\{R_\ell=\mathrm{True}\}
}{
\sum_{\ell=L-w+1}^{L}\mathbf 1\{R_\ell\text{ est connu}\}
}.
$$
Le paramètre $w$ règle la mémoire de cette variable. Une petite fenêtre réagit rapidement ; une grande fenêtre conserve davantage d’historique. Prenons $(w=5)$ comme premier choix illustratif, sans prétendre qu’il est optimal.

**NB**: Nous comparerons éventuellement plusieurs fenêtres par validation croisée à l’intérieur des courses d’entraînement. Et cette variable reste un résumé de la pluie détectée : elle ne mesure pas directement le séchage de la piste.
</details>
 

In [52]:
# Choisir une fenêtre illustrative de cinq tours
recent_window = 5

last_visible_lap = int(driver_rain_history["lap_number"].max())

# Sélectionner les numéros de tour de la fenêtre récente
recent_history = driver_rain_history.loc[
    driver_rain_history["lap_number"].between(
        last_visible_lap - recent_window + 1,
        last_visible_lap,
    )
]

# Calculer la proportion sur les valeurs renseignées uniquement
recent_rain = recent_history["rainfall"].astype("boolean")
recent_known_laps = int(recent_rain.notna().sum())

rain_share_recent = (
    float(recent_rain.sum() / recent_known_laps)
    if recent_known_laps > 0
    else np.nan
)

display(recent_history)
print("Tours récents renseignés :", recent_known_laps)
print("Proportion récente de tours avec pluie :", rain_share_recent)

,lap_number,rainfall,compound
73109,23,False,INTERMEDIATE
73110,24,False,INTERMEDIATE
73111,25,False,INTERMEDIATE
73112,26,False,INTERMEDIATE
73113,27,False,INTERMEDIATE


Tours récents renseignés : 5
Proportion récente de tours avec pluie : 0.0


Le résultat est bien celui attendu : les cinq derniers tours visibles sont renseignés et sans pluie.

Pour ALB, nos trois descriptions du passé sont donc :
| Variable | Valeur | Interprétation |
|---|---:|---|
| `rain_now` | `False` | Pas de pluie détectée au dernier tour visible |
| `rain_share_so_far` | $0,3333$ | Pluie sur $9$ des $27$ tours visibles |
| `rain_share_recent` | $0$ | Pas de pluie sur les $5$ derniers tours |

L’historique global conserve la trace de la pluie ancienne ; la fenêtre récente indique qu’elle n’a plus été détectée sur les derniers tours. Les deux proportions décrivent des périodes différentes.

Nous pouvons maintenant regrouper ces calculs dans une fonction par pilote, avec une fenêtre réglable :

In [53]:
def build_rain_history(visible, recent_window=5):
    """Résume la pluie actuelle, passée et récente par pilote."""

    if visible.empty or visible["race_id"].nunique() != 1:
        raise ValueError("Une seule course non vide est attendue.")

    if not isinstance(recent_window, int) or recent_window < 1:
        raise ValueError("La fenêtre doit être un entier strictement positif.")

    records = []

    for driver_code, driver_laps in visible.groupby("driver_code"):
        ordered = driver_laps.sort_values("lap_number")
        last_row = ordered.iloc[-1]
        last_lap = int(last_row["lap_number"])

        # Calculer la proportion sur tout l'historique renseigné
        rain = ordered["rainfall"].astype("boolean")
        known_laps = int(rain.notna().sum())

        share_so_far = (
            float(rain.sum() / known_laps)
            if known_laps > 0
            else np.nan
        )

        # Utiliser les numéros de tour, pas seulement les dernières lignes
        recent_rain = ordered.loc[
            ordered["lap_number"].between(
                last_lap - recent_window + 1,
                last_lap,
            ),
            "rainfall",
        ].astype("boolean")

        recent_known_laps = int(recent_rain.notna().sum())

        share_recent = (
            float(recent_rain.sum() / recent_known_laps)
            if recent_known_laps > 0
            else np.nan
        )

        records.append({
            "driver_code": driver_code,
            "rain_now": last_row["rainfall"],
            "rain_share_so_far": share_so_far,
            "rain_share_recent": share_recent,
            # Conserver les effectifs pour contrôler la couverture
            "rain_known_laps": known_laps,
            "rain_recent_known_laps": recent_known_laps,
        })

    result = pd.DataFrame(records).set_index("driver_code")
    result["rain_now"] = result["rain_now"].astype("boolean")

    return result

Vérifions notre exemple :

In [54]:
rain_history_check = build_rain_history(
    history_visible,
    recent_window=5,
)

display(rain_history_check.loc[["ALB"]])

,rain_now,rain_share_so_far,rain_share_recent,rain_known_laps,rain_recent_known_laps
driver_code,,,,,
ALB,False,0.333333,0.0,27,5


Cette fois, les résultats sont propres à chaque voiture. La jointure utilisera donc les deux identifiants `race_id` et `driver_code`. 

**Pourquoi contrôler cette couverture ?** Une proportion de pluie égale à zéro sur $5$ tours renseignés est mieux documentée que zéro sur un seul tour renseigné.

Nous avons désormais une table `train_weather` combinant la météo actuelle, son historique et les trois résumés oracle.

In [55]:
# Conserver la même fenêtre pour toutes les voitures
recent_window = 5
history_parts = []

for cutoff in train_cutoffs.itertuples(index=False):

    # Ne consulter que l'historique visible à cette coupure
    visible = train_data.loc[
        train_data["race_id"].eq(cutoff.race_id)
        & train_data["race_time_s"].notna()
        & train_data["race_time_s"].le(cutoff.cutoff_time_s)
    ].copy()

    history_part = build_rain_history(
        visible,
        recent_window=recent_window,
    ).reset_index()

    history_part["race_id"] = cutoff.race_id
    history_parts.append(history_part)

rain_history_train = pd.concat(history_parts, ignore_index=True)

# Remplacer la colonne rain_now déjà créée pour éviter les doublons
history_columns = [
    "rain_now",
    "rain_share_so_far",
    "rain_share_recent",
    "rain_known_laps",
    "rain_recent_known_laps",
]

train_weather = (
    eda_train_with_rain
    .drop(columns=history_columns, errors="ignore")
    .merge(
        rain_history_train,
        on=["race_id", "driver_code"],
        how="left",
        validate="one_to_one",
    )
)

# Vérifier que chaque voiture retenue a reçu son historique
assert len(train_weather) == len(eda_train_with_rain)
assert train_weather["rain_known_laps"].notna().all()

# Contrôler la couverture récente sur les observations évaluables
recent_coverage = (
    train_weather.loc[
        train_weather["scored"],
        "rain_recent_known_laps",
    ]
    .value_counts()
    .sort_index()
    .rename_axis("known_recent_laps")
    .to_frame("observations")
)

display(recent_coverage)

,observations
known_recent_laps,
5,1296


Le compteur `rain_recent_known_laps` vaut toujours 5 dans cet échantillon : il sert au contrôle de disponibilité, mais ne distingue aucune observation ici.

Avant de terminer le bloc météo, vérifions ce que l’historique récent apporte par rapport à la dernière observation :

**Intuition :** si nous trouvons `rain_now=False` avec `rain_share_recent > 0`, l’historique nous apprend qu’il a plu récemment alors que la dernière observation ne le montre plus. Inversement, `rain_now=True` avec une faible proportion récente décrit une pluie peu présente dans les cinq derniers tours.

Ce contrôle nous permettra de voir si les deux variables décrivent effectivement des situations différentes dans notre entraînement.

In [56]:
# Comparer la pluie actuelle et la proportion de pluie récente
weather_comparison = (
    train_weather.loc[train_weather["scored"]]
    .groupby(
        ["rain_now", "rain_share_recent"],
        dropna=False,
    )
    .agg(
        observations=("driver_code", "size"),
        courses=("race_id", "nunique"),
    )
    .reset_index()
)

display(weather_comparison)

,rain_now,rain_share_recent,observations,courses
0,False,0.0,1246,73
1,False,0.2,15,1
2,False,0.4,1,1
3,True,0.2,1,1
4,True,1.0,33,2


<details>
<summary> L’historique récent apporte bien une information différente de la pluie actuelle dans certaines situations.</summary>

| Situation | Observations | Interprétation |
|---|---:|---|
| `False`, proportion $0$ | $1 246$ | Aucun des cinq derniers tours n’indique de pluie |
| `False`, proportion $0,2$ | $15$ | Un des quatre tours précédents indique de la pluie |
| `False`, proportion $0,4$ | $1$ | Deux des quatre tours précédents indiquent de la pluie |
| `True`, proportion $0,2$ | $1$ | Seul le dernier tour indique de la pluie |
| `True`, proportion $1$ | $33$ | Les cinq derniers tours indiquent de la pluie |

Pour 16 voitures, `rain_now=False` masque donc une pluie récente, que rain_share_recent permet de retrouver.

À l’inverse, l’unique observation avec `rain_now=True` et `rain_share_recent=0.2` décrit une apparition de pluie dans la séquence observée : les quatre tours précédents valent `False`, puis le dernier vaut `True`. Cela ne donne pas l’instant exact du début de la pluie.

Notre bloc météo contient désormais six variables candidates :
- `rain_now`
- `rain_share_so_far`
- `rain_share_recent`
- `future_rain_laps`
- `future_rain_first`
- `future_rain_known_laps`
Les compteurs `rain_known_laps` et `rain_recent_known_laps` restent disponibles pour les contrôles. Nous avons également documenté que `future_rain_known_laps` est constant ici et que `future_rain_first` est redondant avec la présence de pluie future dans cet échantillon.
</details>

Nous pouvons maintenant passer au bloc « état de la voiture et des pneus », en commençant par `compound` et `tyre_life_laps` au dernier tour visible.

> - `compound` décrit le type de pneu utilisé : `SOFT`, `MEDIUM`, `HARD`, `INTERMEDIATE` ou `WET`.
>- `tyre_life_laps` décrit son âge en tours. Il peut différer du nombre de tours depuis le dernier arrêt, notamment si un train de pneus déjà utilisé a été monté.

**Intuition :** le même composé peut correspondre à des situations différentes selon son âge. Nous étudierons leur lien avec la cible après avoir vérifié leur disponibilité et leurs distributions.

Ces deux colonnes sont déjà présentes dans `train_weather`, puisque cette table conserve la dernière ligne visible de chaque voiture. Nous n’avons pas besoin de les recalculer.

Pourquoi afficher `known_tyre_ages` ? Les statistiques ignorent les âges manquants. Ce compteur permet donc de vérifier sur combien d’observations elles reposent.

In [57]:
# Examiner uniquement les observations d'entraînement évaluables
scored_car_state = train_weather.loc[
    train_weather["scored"]
].copy()

# Contrôler la disponibilité des deux variables
state_missing = pd.DataFrame({
    "missing_count": scored_car_state[
        ["compound", "tyre_life_laps"]
    ].isna().sum(),

    "missing_percentage": 100 * scored_car_state[
        ["compound", "tyre_life_laps"]
    ].isna().mean(),
})

display(state_missing.round(2))

# Décrire l'âge des pneus pour chaque composé actuel
tyre_age_summary = (
    scored_car_state
    .groupby("compound", dropna=False)
    .agg(
        observations=("driver_code", "size"),
        courses=("race_id", "nunique"),
        known_tyre_ages=("tyre_life_laps", "count"),
        min_tyre_age=("tyre_life_laps", "min"),
        median_tyre_age=("tyre_life_laps", "median"),
        max_tyre_age=("tyre_life_laps", "max"),
    )
)

display(tyre_age_summary)

,missing_count,missing_percentage
compound,0,0.0
tyre_life_laps,0,0.0


,observations,courses,known_tyre_ages,min_tyre_age,median_tyre_age,max_tyre_age
compound,,,,,,
HARD,630,67,630,1,10.0,36
INTERMEDIATE,67,5,67,1,21.0,30
MEDIUM,520,68,520,1,18.0,43
SOFT,79,29,79,1,21.0,41


Les deux variables sont renseignées pour les $1 296$ observations évaluables. Aucune imputation n’est nécessaire pour elles dans cet échantillon.

Nous observons notamment :
- `HARD` : $630$ voitures, âge médian de $10$ tours ;
- `MEDIUM` : $520$ voitures, âge médian de $18$ tours ;
- `SOFT` : $79$ voitures, âge médian de $21$ tours ;
- `INTERMEDIATE` : $67$ voitures, réparties sur $5$ courses ;
- aucun pneu `WET` actuellement monté dans cet échantillon.

**Attention à l’interprétation :** ces âges décrivent les pneus présents aux coupures choisies, pas leur durée de vie maximale. Les `HARD` plus jeunes pourraient notamment avoir été montés récemment, mais nous vérifierons cela avec l’historique des arrêts. Un `SOFT` âgé de $41$ tours ne signifie pas non plus que tous les `SOFT` peuvent tenir aussi longtemps.

**Passons maintenant à la question : que monte-t-on ensuite selon le pneu actuel ?**

Pour un composé actuel $c$ et une cible $k$, nous estimons :
$$
\widehat P(Y=k\mid C=c)
=
\frac{\#\{i:C_i=c,\ Y_i=k\}}
{\#\{i:C_i=c\}}.
$$

**Intuition :** chaque ligne répond à une question comme : « Parmi les voitures actuellement en `HARD`, quelle proportion ne s’arrêtera plus, passera en `SOFT`, etc. ? »

La ligne `WET` aura des pourcentages manquants puisqu’aucune observation ne permet de les estimer. Nous gardons aussi les effectifs : une proportion calculée sur $79$ voitures ne dispose pas du même support qu’une proportion calculée sur $630$.

In [58]:
# Compter les transitions entre composé actuel et prochain composé
compound_order = ["SOFT", "MEDIUM", "HARD", "INTERMEDIATE", "WET"]

transition_counts = pd.crosstab(
    scored_car_state["compound"],
    scored_car_state["next_compound"],
).reindex(
    index=compound_order,
    columns=CLASSES,
    fill_value=0,
)

# Calculer les proportions pour chaque composé actuel
transition_pct = (
    transition_counts.div(
        transition_counts.sum(axis=1).replace(0, np.nan),
        axis=0,
    ) * 100
)

display(transition_counts)
display(transition_pct.round(2))

next_compound,SOFT,MEDIUM,HARD,WET_OR_INTER,NONE
compound,,,,,
SOFT,15,46,13,1,4
MEDIUM,87,42,319,16,56
HARD,83,183,131,1,232
INTERMEDIATE,5,34,6,17,5
WET,0,0,0,0,0


next_compound,SOFT,MEDIUM,HARD,WET_OR_INTER,NONE
compound,,,,,
SOFT,18.99,58.23,16.46,1.27,5.06
MEDIUM,16.73,8.08,61.35,3.08,10.77
HARD,13.17,29.05,20.79,0.16,36.83
INTERMEDIATE,7.46,50.75,8.96,25.37,7.46
WET,NaN,NaN,NaN,NaN,NaN


<details>
<summary> Le tableau montre que la distribution du prochain composé varie selon le pneu actuel.</summary>

Les lignes indiquent le pneu à la coupure ; les colonnes indiquent la cible.

Prenons la ligne `HARD.` Parmi les $630$ voitures actuellement en `HARD` :
- $232$ ne s’arrêtent plus : cible `NONE`, soit $36,83 \%$ ;
- $183$ montent des `MEDIUM` au prochain arrêt : $29,05 \%$ ;
- $131$ remontent un train de `HARD `: $20,79 \%$ ;
- $83$ passent aux `SOFT` : $13,17 \%$ ;
- $1$ passe aux pneus pluie : $0,16 \%$.

Par exemple :
$$
\widehat P(Y=\text{NONE}\mid C=\text{HARD})
=\frac{232}{630}
\approx36{,}83\%.
$$
`NONE` est la cible la plus fréquente pour les `HARD`, mais elle n’est pas majoritaire : les $398$ autres voitures ont un nouvel arrêt.

Les principales tendances sont :

| Pneu actuel | Cible la plus fréquente | Proportion |
|---|---|---:|
| `SOFT` | `MEDIUM` | $58,23 \%$ |
| `MEDIUM` | `HARD` | $61,35 \%$ |
| `HARD` | `NONE` | $36,83 \%$ |
| `INTERMEDIATE` | `MEDIUM` | $50,75 \%$ |

La ligne `WET` contient des `NaN` parce qu’aucune voiture n’utilise ce composé à nos coupures : nous ne pouvons pas estimer les proportions correspondantes.

Pourquoi conserver `compound` comme feature ? Sans cette information, la fréquence globale de `NONE` est de $22,92 \%$. En connaissant le pneu actuel, elle devient $5,06 \%$ pour `SOFT`, $10,77 \%$ pour `MEDIUM` ou $36,83 \%$ pour `HARD`. Cette variable distingue donc des situations différentes dans l’entraînement.

Cela reste une association, pas une preuve que le composé provoque ces différences : son âge, les arrêts déjà effectués et les tours restants interviennent aussi.

</details>

La prochaine question sera : pour un même composé, la fréquence de `NONE` varie-t-elle avec l’âge du pneu ? Nous pourrons ainsi examiner ce qu’apporte `tyre_life_laps` en complément de `compound`.


Étudions maintenant la proportion de `NONE` selon l’âge du pneu, séparément pour chaque composé.

Pour un composé $c$ et une tranche d’âge $B$, nous calculons :
$$
\widehat P(Y=\text{NONE}\mid C=c,\ A\in B)
=
\frac{
\#\{\text{voitures en }c,\ \text{âge dans }B,\ Y=\text{NONE}\}
}{
\#\{\text{voitures en }c,\ \text{âge dans }B\}
}.
$$

Intuition : parmi les voitures en `HARD`, par exemple, nous comparons celles dont les pneus ont $1$ à $5$ tours à celles dont les pneus ont $21$ à $30$ tours.

Nous regroupons les âges pour éviter un tableau trop dispersé. Ces tranches servent à l’EDA ; elles ne remplacent pas la variable numérique dans le modèle.

In [59]:
# Définir des tranches d'âge pour faciliter la lecture
age_analysis = scored_car_state.copy()

age_analysis["tyre_age_group"] = pd.cut(
    age_analysis["tyre_life_laps"].astype(float),
    bins=[0, 5, 10, 15, 20, 30, np.inf],
    labels=["0–5", "6–10", "11–15", "16–20", "21–30", "31+"],
    include_lowest=True,
)

# Identifier les voitures qui ne s'arrêtent plus après la coupure
age_analysis["no_further_stop"] = (
    age_analysis["next_compound"].eq("NONE")
)

# Calculer les effectifs et la fréquence de NONE par composé et âge
age_target_summary = (
    age_analysis
    .groupby(["compound", "tyre_age_group"], observed=True)
    .agg(
        observations=("driver_code", "size"),
        courses=("race_id", "nunique"),
        none_count=("no_further_stop", "sum"),
    )
)

age_target_summary["none_percentage"] = (
    100
    * age_target_summary["none_count"]
    / age_target_summary["observations"]
)

display(age_target_summary.round(2))

observations  courses  none_count  \
compound     tyre_age_group                                      
HARD         0–5                      177       48          97   
             6–10                     162       41          69   
             11–15                    112       34          36   
             16–20                     91       37          12   
             21–30                     77       28          11   
             31+                       11        2           7   
INTERMEDIATE 0–5                       12        1           0   
             6–10                       1        1           0   
             11–15                     18        2           3   
             21–30                     36        3           2   
MEDIUM       0–5                       68       27          30   
             6–10                      56       25           2   
             11–15                     56       16           1   
             16–20                    141       27           8   
             21–30                    175       42          10   
             31+                       24        7           5   
SOFT         0–5                        6        5           2   
             6–10                      11        9           0   
             11–15                      6        5           0   
             16–20                     16        7           0   
             21–30                     32       13           2   
             31+                        8        4           0   

                             none_percentage  
compound     tyre_age_group                   
HARD         0–5                        54.8  
             6–10                      42.59  
             11–15                     32.14  
             16–20                     13.19  
             21–30                     14.29  
             31+                       63.64  
INTERMEDIATE 0–5                         0.0  
             6–10                        0.0  
             11–15                     16.67  
             21–30                      5.56  
MEDIUM       0–5                       44.12  
             6–10                       3.57  
             11–15                      1.79  
             16–20                      5.67  
             21–30                      5.71  
             31+                       20.83  
SOFT         0–5                       33.33  
             6–10                        0.0  
             11–15                       0.0  
             16–20                       0.0  
             21–30                      6.25  
             31+                         0.0

L’âge du pneu distingue bien des situations différentes, même à composé identique. Mais la relation n’est pas systématiquement décroissante.

<details>
<summary> Pour les HARD, la proportion de NONE passe de :</summary>

- $54,80 \%$ pour les pneus âgés de $0$ à $5$ tours ;
- à $32,14 \%$ pour $11$ à $15$ tours ;
- puis $13,19 \%$ pour $16$ à $20$ tours.
Par exemple :
$$
\widehat P(Y=\text{NONE}\mid C=\text{HARD},\,A\in[0,5])
=\frac{97}{177}\approx54{,}80\%.
$$
**Intuition :** un pneu jeune peut avoir été monté lors d’un arrêt récent et permettre de poursuivre jusqu’à l’arrivée. C’est une explication possible, que l’historique des arrêts et les tours restants permettront de préciser.

Pour les `MEDIUM`, le contraste est également marqué : $44,12 \%$ de `NONE` pour $0–5$ tours, contre $3,57 \%$ pour $6–10$ tours.

Il faut cependant éviter deux conclusions trop rapides :

- Les `HARD` de $31$ tours ou plus ont un taux de `NONE` de $63,64 \%$, mais cela concerne seulement $11$ voitures dans $2$ courses. Ce résultat ne démontre pas qu’un pneu plus vieux favorise l’absence d’arrêt.
- Les groupes `SOFT` et `INTERMEDIATE` sont souvent petits. Un taux observé de $0 \%$ ne signifie pas que l’événement est impossible.
</details>

>À composé identique, la fréquence de `NONE` varie avec l’âge du pneu, ce qui motive l’utilisation conjointe de `compound` et `tyre_life_laps.` Cette relation n’est toutefois pas monotone et dépend probablement du contexte de course. Les groupes peu représentés doivent être interprétés avec prudence.

**Il manque maintenant une information essentielle : combien de tours reste-t-il à parcourir ?**

Un pneu âgé de $15$ tours avec $10$ tours restants ne pose pas le même problème qu’un pneu du même âge avec $40$ tours restants. Cela motive l’examen de :
$$
\texttt{projected\_tyre\_age}
=
\texttt{tyre\_life\_laps}
+
\texttt{laps\_remaining}.
$$
Cette variable décrit l’âge projeté du pneu si la voiture effectue tous les tours restants indiqués sans le changer. Avant de la calculer, nous vérifierons comment `laps_remaining` est défini dans les données.

On vérifie d’abord si `laps_remaining` correspond à :
$$
R_i=D_i-L_i,
$$
où $D_i$ est le nombre de tours prévus et $L_i$ le dernier tour visible de la voiture.

**Intuition :** pour une course prévue sur $58$ tours, une voiture ayant terminé son tour $22$ aurait alors :
$$
R_i=58-22=36\text{ tours restants}.
$$
Il s’agit d’une distance théorique jusqu’au nombre de tours prévu, pas nécessairement du nombre de tours que la voiture effectuera réellement.

**Pourquoi ce contrôle ?** Nous voulons comprendre ce que mesure la colonne avant de l’utiliser. Si l’égalité est vérifiée, notre projection sera :
$$
A_i^{\mathrm{proj}}=A_i+(D_i-L_i).
$$
Ce sera une hypothèse de poursuite sans changement de pneus, pas une prédiction de leur durée de vie maximale.

In [60]:
# Comparer la colonne fournie au nombre de tours prévus moins le tour courant
remaining_audit = scored_car_state[
    [
        "race_id",
        "driver_code",
        "scheduled_laps",
        "lap_number",
        "laps_remaining",
    ]
].copy()

remaining_audit["expected_laps_remaining"] = (
    remaining_audit["scheduled_laps"]
    - remaining_audit["lap_number"]
)

# Comparer seulement les lignes où les deux valeurs sont connues
checkable = remaining_audit[
    ["laps_remaining", "expected_laps_remaining"]
].notna().all(axis=1)

remaining_audit["difference"] = (
    remaining_audit["laps_remaining"]
    - remaining_audit["expected_laps_remaining"]
)

mismatch = (
    checkable
    & remaining_audit["difference"].ne(0).fillna(False)
)

print("Observations examinées :", len(remaining_audit))
print("Valeurs laps_remaining manquantes :",
      int(remaining_audit["laps_remaining"].isna().sum()))
print("Comparaisons possibles :", int(checkable.sum()))
print("Différences avec scheduled_laps - lap_number :",
      int(mismatch.sum()))

display(remaining_audit.head())

# Afficher les éventuelles différences pour les examiner
if mismatch.any():
    display(remaining_audit.loc[mismatch])

Observations examinées : 1296
Valeurs laps_remaining manquantes : 0
Comparaisons possibles : 1296
Différences avec scheduled_laps - lap_number : 0


,race_id,driver_code,scheduled_laps,lap_number,laps_remaining,expected_laps_remaining,difference
0,2018_01,ALO,58,27,31,31,0
1,2018_01,BOT,58,27,31,31,0
2,2018_01,HAM,58,27,31,31,0
3,2018_01,HAR,58,26,32,32,0
4,2018_01,HUL,58,27,31,31,0


L’égalité est vérifiée sur les 1 296 observations, sans valeur manquante :
$$
\texttt{laps\_remaining}
=\texttt{scheduled\_laps}-\texttt{lap\_number}.
$$
Pour ALO, par exemple, $58-27=31$. Nous pouvons donc construire :
$$
\underbrace{A_i^{\mathrm{proj}}}_{\text{âge projeté}}
=
\underbrace{A_i}_{\text{âge actuel}}
+
\underbrace{R_i}_{\text{tours restants}}.
$$
Intuition : si un pneu a déjà $8$ tours et que la voiture doit encore parcourir $31$ tours, il atteindrait $39$ tours d’utilisation si elle conservait ce train jusqu’à cette distance prévue. Cela ne dit pas si le pneu peut effectivement tenir.


**NB:** Point mathématique important : cette variable n’ajoute aucune information indépendante lorsque l’âge actuel et les tours restants sont déjà disponibles. Elle est leur somme exacte.

Elle peut néanmoins faciliter certaines séparations pour un modèle à arbres. Pour la régression logistique, la LDA ou la QDA, nous éviterons d’inclure automatiquement les trois colonnes ensemble : leur dépendance exacte peut rendre la représentation redondante, notamment pour les matrices de covariance.

Nous la construisons donc comme variable candidate, puis nous choisirons la représentation adaptée au modèle.

In [61]:
# Calculer l'âge projeté en supposant aucun nouveau changement de pneus
train_weather["projected_tyre_age"] = (
    train_weather["tyre_life_laps"]
    + train_weather["laps_remaining"]
)

# Actualiser la table utilisée pour l'analyse des voitures évaluables
scored_car_state = train_weather.loc[
    train_weather["scored"]
].copy()

# Examiner quelques exemples
display(
    scored_car_state[
        [
            "driver_code",
            "compound",
            "tyre_life_laps",
            "laps_remaining",
            "projected_tyre_age",
            "next_compound",
        ]
    ].head(10)
)

,driver_code,compound,tyre_life_laps,laps_remaining,projected_tyre_age,next_compound
0,ALO,HARD,1,31,32,NONE
1,BOT,MEDIUM,2,31,33,NONE
2,HAM,HARD,8,31,39,NONE
3,HAR,SOFT,4,32,36,NONE
4,HUL,HARD,3,31,34,NONE
5,LEC,HARD,7,31,38,NONE
6,OCO,HARD,4,31,35,NONE
7,PER,HARD,3,31,34,NONE
8,RAI,HARD,9,31,40,NONE
9,RIC,HARD,1,31,32,NONE


La colonne `next_compound=NONE` est la réponse connue grâce à la course complète. Elle n’intervient pas dans le calcul de la feature. Et la projection n’est pas nécessairement l’âge exact à l’arrivée, notamment pour une voiture terminant avec un tour de retard.

Ces dix exemples viennent de la même course. Pour examiner l’association sur tout l’entraînement, comparons la fréquence de `NONE` selon le composé et l’âge projeté

**La question est :** à composé identique, l’absence de nouvel arrêt est-elle moins fréquente lorsque l’âge projeté est élevé ? Nous allons le vérifier sans supposer que la relation sera systématique. Ces tranches servent seulement à lire les résultats ; la feature reste numérique.

In [62]:
# Regrouper les âges projetés pour l'analyse descriptive
projected_analysis = scored_car_state.copy()

projected_analysis["projected_age_group"] = pd.cut(
    projected_analysis["projected_tyre_age"].astype(float),
    bins=[0, 30, 40, 50, 60, np.inf],
    labels=["0–30", "31–40", "41–50", "51–60", "61+"],
    include_lowest=True,
)

projected_analysis["no_further_stop"] = (
    projected_analysis["next_compound"].eq("NONE")
)

# Conserver les effectifs pour interpréter les pourcentages
projected_summary = (
    projected_analysis
    .groupby(["compound", "projected_age_group"], observed=True)
    .agg(
        observations=("driver_code", "size"),
        courses=("race_id", "nunique"),
        none_count=("no_further_stop", "sum"),
    )
)

projected_summary["none_percentage"] = (
    100
    * projected_summary["none_count"]
    / projected_summary["observations"]
)

display(projected_summary.round(2))

observations  courses  none_count  \
compound     projected_age_group                                      
HARD         0–30                            7        2           6   
             31–40                         127       26          93   
             41–50                         240       51          84   
             51–60                         190       46          34   
             61+                            66       21          15   
INTERMEDIATE 31–40                           2        1           0   
             41–50                          29        3           3   
             51–60                          21        2           2   
             61+                            15        2           0   
MEDIUM       31–40                          52       15          23   
             41–50                         103       31          15   
             51–60                         223       40           6   
             61+                           142       27          12   
SOFT         0–30                            1        1           1   
             31–40                           3        3           1   
             41–50                          19        8           0   
             51–60                          34       14           2   
             61+                            22       13           0   

                                  none_percentage  
compound     projected_age_group                   
HARD         0–30                           85.71  
             31–40                          73.23  
             41–50                           35.0  
             51–60                          17.89  
             61+                            22.73  
INTERMEDIATE 31–40                            0.0  
             41–50                          10.34  
             51–60                           9.52  
             61+                              0.0  
MEDIUM       31–40                          44.23  
             41–50                          14.56  
             51–60                           2.69  
             61+                             8.45  
SOFT         0–30                           100.0  
             31–40                          33.33  
             41–50                            0.0  
             51–60                           5.88  
             61+                              0.0

<details>
<summary>Nous observons une tendance nette pour les HARD et MEDIUM : </summary>

 Plus l’âge projeté augmente, moins `NONE` est fréquent, jusqu’à la tranche $51–60$ tours.

 | Âge projeté | `NONE` pour `HARD` | `NONE` pour `MEDIUM` |
|---|---:|---:|
| $31–40$ tours | $73,23 \%$ | $44,23\%$ |
| $41–50$ tours | $35,00 \%$ | $14,56 \%$ |
| $51–60$ tours | $17,89 \%$ | $2,69 \%$ |
| $61$ tours ou plus | $22,73 \%$ | $8,45 \%$ |


Par exemple :
$$
\widehat P(Y=\text{NONE}\mid C=\text{HARD},\,A^{\mathrm{proj}}\in[31,40])
=\frac{93}{127}\approx73{,}23\%.
$$

**Intuition :** lorsqu’un train de pneus devrait atteindre un âge élevé pour couvrir la distance restante, un nouvel arrêt est généralement plus fréquent dans nos observations.

Mais la relation n’est pas strictement décroissante : les proportions remontent dans la tranche $61+$. Les circuits, la longueur des tours et les stratégies diffèrent entre courses. Un âge exprimé en tours ne mesure donc pas une usure physique comparable partout.

Deux précautions supplémentaires :
- Le $100 %$ des `SOFT` entre $0$ et $30$ tours repose sur une seule voiture : ce n’est pas une règle générale.
- Les `INTERMEDIATE` sont représentés dans peu de courses ; leurs choix dépendent aussi de l’évolution des conditions météo.

</details>


>L’âge projeté est associé à la fréquence d’absence de nouvel arrêt, particulièrement pour les `HARD` et `MEDIUM`. Cette association justifie son examen comme variable candidate, sans définir un seuil universel de durée de vie des pneus. Il s’agit d’une combinaison déterministe de l’âge actuel et des tours restants ; son intérêt prédictif devra être évalué selon le modèle.
Nous avons maintenant justifié quatre variables d’état candidates : `compound`, `tyre_life_laps`, `laps_remaining` et `projected_tyre_age`.

**L’historique des arrêts :** combien d’arrêts la voiture a-t-elle déjà effectués, et depuis combien de tours roule-t-elle depuis le dernier ? Cela permettra notamment de distinguer l’âge du pneu de la durée du relais actuel.

Commençons par `n_stops` : Le nombre d’arrêts officiels déjà observés pour une voiture à la coupure.

Pour le pilote $d$, nous définissons :
$$
N_d(T)=
\sum_{\ell\in\mathcal V_d(T)}
\mathbf 1_{\{\texttt{official\_stop}_{d,\ell}=\mathrm{True}\}},
$$
où $\mathcal V_d(T)$ contient uniquement ses tours visibles.

**Intuition :** nous comptons les arrêts dans l’historique disponible. Nous ne les déduisons pas des changements de composé : une voiture peut s’arrêter et remonter le même type de pneu.

Un arrêt signalé sur le dernier tour visible est inclus dans ce compteur. Selon `target.py`, cet arrêt est déjà visible et ne compte pas comme le prochain arrêt à prédire.

In [63]:
"""
Ce compteur mesure les arrêts présents dans les tours visibles.
"""
# Reprendre la coupure de notre course d'exemple
stop_race_id = "2018_01"

stop_cutoff = train_cutoffs.loc[
    train_cutoffs["race_id"].eq(stop_race_id)
].iloc[0]

# Conserver uniquement les tours terminés avant ou à la coupure
stop_visible = train_data.loc[
    train_data["race_id"].eq(stop_race_id)
    & train_data["race_time_s"].notna()
    & train_data["race_time_s"].le(stop_cutoff["cutoff_time_s"])
].copy()

# Vérifier que chaque ligne possède un indicateur d'arrêt
assert stop_visible["official_stop"].notna().all(), (
    "L'indicateur d'arrêt contient des valeurs manquantes."
)

# Compter les arrêts officiels observés par pilote
stop_counts = (
    stop_visible.groupby("driver_code")["official_stop"]
    .sum()
    .astype(int)
    .rename("n_stops")
)

# Associer le compteur aux voitures retenues à cette coupure
stop_comparison = (
    train_weather.loc[
        train_weather["race_id"].eq(stop_race_id),
        [
            "driver_code",
            "lap_number",
            "compound",
            "tyre_life_laps",
            "official_stop",
            "next_compound",
        ],
    ]
    .merge(
        stop_counts,
        left_on="driver_code",
        right_index=True,
        how="left",
        validate="one_to_one",
    )
)

display(stop_comparison)

,driver_code,lap_number,compound,tyre_life_laps,official_stop,next_compound,n_stops
0,ALO,27,HARD,1,False,NONE,1
1,BOT,27,MEDIUM,2,False,NONE,1
2,HAM,27,HARD,8,False,NONE,1
3,HAR,26,SOFT,4,False,NONE,2
4,HUL,27,HARD,3,False,NONE,1
5,LEC,27,HARD,7,True,NONE,2
6,OCO,27,HARD,4,False,NONE,1
7,PER,27,HARD,3,False,NONE,1
8,RAI,27,HARD,9,False,NONE,1
9,RIC,27,HARD,1,False,NONE,1


toutes les voitures retenues ont au moins un arrêt observé : une pour la plupart, deux pour HAR et LEC.

Le cas LEC est important :
- son dernier tour visible est le $27$ ;
- `official_stop=True` : l’arrêt de ce tour est inclus dans `n_stops=2` ;
- `next_compound=NONE` signifie aucun nouvel arrêt strictement après le tour $27$, conformément à `target.py`.
Son âge de pneu, $7$ tours, ne doit donc pas être automatiquement remis à zéro : cette ligne ne nous fournit pas nécessairement le pneu monté après l’arrêt.

Calculons maintenant `laps_since_stop`. Si $L_d$ est le dernier tour visible et $S_d$ le dernier tour avec arrêt observé :
$$
\texttt{laps\_since\_stop}_d=L_d-S_d.
$$
Sans arrêt observé, nous utilisons $0$ comme origine conventionnelle, ce qui donne le numéro du dernier tour visible. Cela représente alors les tours depuis le départ, sous réserve d’un historique complet.

In [64]:
# Identifier le dernier tour avec arrêt dans l'historique visible
last_stop_lap = (
    stop_visible.loc[stop_visible["official_stop"].astype(bool)]
    .groupby("driver_code")["lap_number"]
    .max()
)

# Ajouter le dernier arrêt à notre tableau de contrôle
stop_comparison["last_stop_lap"] = (
    stop_comparison["driver_code"].map(last_stop_lap)
)

# Sans arrêt observé, prendre le départ comme origine conventionnelle
stop_comparison["laps_since_stop"] = (
    stop_comparison["lap_number"]
    - stop_comparison["last_stop_lap"].fillna(0)
).astype(int)

display(
    stop_comparison[
        [
            "driver_code",
            "lap_number",
            "last_stop_lap",
            "n_stops",
            "laps_since_stop",
            "tyre_life_laps",
        ]
    ]
)

,driver_code,lap_number,last_stop_lap,n_stops,laps_since_stop,tyre_life_laps
0,ALO,27,26,1,1,1
1,BOT,27,25,1,2,2
2,HAM,27,19,1,8,8
3,HAR,26,22,2,4,4
4,HUL,27,24,1,3,3
5,LEC,27,27,2,0,7
6,OCO,27,23,1,4,4
7,PER,27,24,1,3,3
8,RAI,27,18,1,9,9
9,RIC,27,26,1,1,1


Le calcul donne bien le résultat attendu. Pour HAM :
$$
\texttt{laps\_since\_stop}=27-19=8,
$$
ce qui correspond ici à son âge de pneu de 8 tours.

LEC illustre pourquoi les deux variables ne sont pas interchangeables :
$$
\texttt{laps\_since\_stop}=27-27=0,
\qquad
\texttt{tyre\_life\_laps}=7.
$$
L’arrêt est enregistré sur son dernier tour visible, mais l’âge du pneu de cette ligne reste de $7$ tours. Nous conservons ces deux informations telles quelles.

Pour rendre cette situation explicite, ajoutons `stopped_on_last_lap`, qui reprend l’indicateur d’arrêt du dernier tour visible :

>Nos trois variables d’arrêt répondent ainsi à des questions différentes :
>| Variable | Question |
>|---|---|
>| `n_stops` | Combien d’arrêts officiels avons-nous observés ? |
>| `laps_since_stop` | Combien de tours séparent le dernier tour visible du dernier arrêt observé ? |
>| `stopped_on_last_lap` | Un arrêt est-il enregistré précisément sur le dernier tour visible ? |

In [65]:
# Indiquer si un arrêt est enregistré sur le dernier tour visible
stop_comparison["stopped_on_last_lap"] = (
    stop_comparison["official_stop"].astype(int)
)

display(
    stop_comparison[
        [
            "driver_code",
            "n_stops",
            "laps_since_stop",
            "stopped_on_last_lap",
            "tyre_life_laps",
        ]
    ]
)

,driver_code,n_stops,laps_since_stop,stopped_on_last_lap,tyre_life_laps
0,ALO,1,1,0,1
1,BOT,1,2,0,2
2,HAM,1,8,0,8
3,HAR,2,4,0,4
4,HUL,1,3,0,3
5,LEC,2,0,1,7
6,OCO,1,4,0,4
7,PER,1,3,0,3
8,RAI,1,9,0,9
9,RIC,1,1,0,1


>Regroupons maintenant les trois calculs dans une fonction qui reçoit uniquement l’historique visible d’une course.

In [66]:
def build_stop_history(visible):
    """Construit les trois variables d'arrêts observés par pilote."""

    if visible.empty or visible["race_id"].nunique() != 1:
        raise ValueError("Une seule course non vide est attendue.")

    if visible["official_stop"].isna().any():
        raise ValueError("L'indicateur d'arrêt contient des valeurs manquantes.")

    if visible.duplicated(["driver_code", "lap_number"]).any():
        raise ValueError("Plusieurs lignes existent pour un même pilote-tour.")

    ordered = visible.sort_values(["driver_code", "lap_number"])

    # Conserver exactement la dernière ligne visible de chaque pilote
    last_rows = (
        ordered
        .drop_duplicates("driver_code", keep="last")
        .set_index("driver_code")
    )

    features = pd.DataFrame(index=last_rows.index)

    # Compter tous les arrêts visibles, y compris sur le dernier tour
    features["n_stops"] = (
        ordered.groupby("driver_code")["official_stop"]
        .sum()
        .astype(int)
    )

    # Identifier le dernier tour avec arrêt observé
    last_stop_lap = (
        ordered.loc[ordered["official_stop"].astype(bool)]
        .groupby("driver_code")["lap_number"]
        .max()
    )

    # Sans arrêt observé, utiliser le départ comme origine conventionnelle
    features["laps_since_stop"] = (
        last_rows["lap_number"]
        - last_stop_lap.reindex(last_rows.index).fillna(0)
    ).astype(int)

    features["stopped_on_last_lap"] = (
        last_rows["official_stop"].astype(int)
    )

    return features

In [67]:
stop_features_check = build_stop_history(stop_visible)

stop_columns = [
    "n_stops",
    "laps_since_stop",
    "stopped_on_last_lap",
]

expected_stops = (
    stop_comparison
    .set_index("driver_code")[stop_columns]
    .sort_index()
)

pd.testing.assert_frame_equal(
    stop_features_check.loc[expected_stops.index, stop_columns],
    expected_stops,
)

print("La fonction reproduit les trois variables d'arrêts de notre exemple.")

La fonction reproduit les trois variables d'arrêts de notre exemple.


Appliquons-la maintenant aux 76 courses d’entraînement, avec leurs coupures déjà fixées.

**Intuition :** nous allons voir combien de voitures n’ont encore aucun arrêt observé, combien en ont déjà un, deux, etc. Nous examinerons ensuite comment la cible varie selon cet historique.

In [68]:
stop_parts = []

for cutoff in train_cutoffs.itertuples(index=False):

    # Conserver uniquement les tours visibles à la coupure
    visible = train_data.loc[
        train_data["race_id"].eq(cutoff.race_id)
        & train_data["race_time_s"].notna()
        & train_data["race_time_s"].le(cutoff.cutoff_time_s)
    ].copy()

    # Calculer les trois variables d'arrêts par pilote
    stop_part = build_stop_history(visible).reset_index()
    stop_part["race_id"] = cutoff.race_id
    stop_parts.append(stop_part)

stop_features_train = pd.concat(stop_parts, ignore_index=True)

stop_columns = [
    "n_stops",
    "laps_since_stop",
    "stopped_on_last_lap",
]

# Ajouter les variables aux voitures déjà retenues
train_features = (
    train_weather
    .drop(columns=stop_columns, errors="ignore")
    .merge(
        stop_features_train,
        on=["race_id", "driver_code"],
        how="left",
        validate="one_to_one",
    )
)

# Vérifier que la jointure conserve les lignes et renseigne les trois variables
assert len(train_features) == len(train_weather)
assert train_features[stop_columns].notna().all().all()

# Décrire le nombre d'arrêts déjà observés dans la population évaluable
stop_support = (
    train_features.loc[train_features["scored"]]
    .groupby("n_stops")
    .agg(
        observations=("driver_code", "size"),
        courses=("race_id", "nunique"),
    )
)

display(stop_support)

,observations,courses
n_stops,,
0,411,57
1,754,72
2,95,30
3,31,6
4,5,3


<details>
<summary> La majorité des voitures ont déjà un arrêt observé à la coupure : 754 sur 1 296, soit 58,18 %.</summary>

| Arrêts observés | Voitures–coupures | Proportion |
|---:|---:|---:|
| $0$ |$411$ | $31,71 \%$ |
| $1$ | $754$ | $58,18 \%$ |
| $2$ | $95$ | $7,33 \%$ |
| $3$ | $31$ | $2,39 \%$ |
| $4$ | $5$ | $0,39 \%$ |

La colonne `courses` compte les courses représentées dans chaque groupe. Une même course peut apparaître dans plusieurs groupes, car ses pilotes n’ont pas forcément effectué autant d’arrêts.
</details>

La question suivante est : la proportion de `NONE` varie-t-elle selon le nombre d’arrêts déjà observés ?
$$
\widehat P(Y=\text{NONE}\mid N=n)
=
\frac{\#\{i:N_i=n,\ Y_i=\text{NONE}\}}
{\#\{i:N_i=n\}}.
$$
**Intuition :** avoir déjà effectué un arrêt peut correspondre à une stratégie différente de celle d’une voiture encore sur son premier relais. Mais plusieurs arrêts peuvent aussi refléter une course perturbée : nous ne supposons donc pas que davantage d’arrêts implique systématiquement moins d’arrêts futurs.

In [69]:
# Examiner la cible selon le nombre d'arrêts déjà observés
stop_target_analysis = train_features.loc[
    train_features["scored"]
].copy()

stop_target_analysis["no_further_stop"] = (
    stop_target_analysis["next_compound"].eq("NONE")
)

stop_target_summary = (
    stop_target_analysis.groupby("n_stops")
    .agg(
        observations=("driver_code", "size"),
        courses=("race_id", "nunique"),
        none_count=("no_further_stop", "sum"),
    )
)

stop_target_summary["none_percentage"] = (
    100
    * stop_target_summary["none_count"]
    / stop_target_summary["observations"]
)

display(stop_target_summary.round(2))

,observations,courses,none_count,none_percentage
n_stops,,,,
0,411,57,2,0.49
1,754,72,266,35.28
2,95,30,19,20.0
3,31,6,10,32.26
4,5,3,0,0.0


<details>
<summary>Le contraste est marqué entre les voitures sans arrêt observé et celles qui en ont déjà effectué un. </summary>

$$
\widehat P(Y=\text{NONE}\mid N=0)
=\frac{2}{411}\approx0{,}49\%,
$$
contre :
$$
\widehat P(Y=\text{NONE}\mid N=1)
=\frac{266}{754}\approx35{,}28\%.
$$

**Intuition :** aux coupures étudiées, presque toutes les voitures sans arrêt observé ont encore un arrêt à effectuer. Parmi celles ayant déjà un arrêt, environ un tiers ne s’arrêtera plus.

Mais la relation n’est pas monotone : le taux passe ensuite à $20 \%$ pour deux arrêts, puis $32,26 \%$ pour trois. Le nombre d’arrêts résume une partie de la stratégie, sans décrire à lui seul le pneu actuel, la météo ou la distance restante.

Le $0 \%$ pour quatre arrêts repose seulement sur cinq observations : il ne signifie pas qu’un nouvel arrêt est toujours nécessaire.
</details>


>Le nombre d’arrêts observés est associé à la cible : `NONE` représente $0,49 \%$ des observations sans arrêt, contre $35,28 \%$ après un arrêt. Cette association motive l’utilisation de `n_stops`, sans supposer une relation monotone ni causale.

Examinons les deux cas `n_stops=0` et `next_compound=NONE`, avant de chercher une explication :

« Aucun arrêt observé » n’est pas automatiquement « aucun changement de pneus ». Un historique incomplet ou un changement non compté comme arrêt officiel peut notamment produire une différence. Ce sont des possibilités à vérifier, pas encore des conclusions sur ces deux voitures.

In [70]:
# Examiner les voitures sans arrêt observé dont la cible est NONE
no_stop_none = train_features.loc[
    train_features["scored"]
    & train_features["n_stops"].eq(0)
    & train_features["next_compound"].eq("NONE"),
    [
        "race_id",
        "driver_code",
        "lap_number",
        "compound",
        "tyre_life_laps",
        "laps_remaining",
        "rain_now",
        "n_stops",
        "result_status",
    ],
]

display(no_stop_none)

,race_id,driver_code,lap_number,compound,tyre_life_laps,laps_remaining,rain_now,n_stops,result_status
256,2019_21,GAS,16,HARD,15,39,False,0,+2 Laps
483,2021_16,OCO,21,INTERMEDIATE,21,37,False,0,+1 Lap


Les deux cas sont GAS dans 2019_21 et OCO dans 2021_16. Ils ont tous deux un résultat final admissible, mais leurs situations diffèrent.
- OCO roule en intermédiaires âgés de $21$ tours, au tour $21$. Cela est compatible avec un même train utilisé depuis le départ, sans le prouver à lui seul.
- GAS roule en `HARD` âgés de $15$ tours, au tour $16$. Le décalage entre l’âge du pneu et le numéro du tour mérite un examen de son historique.

Vérifions, pour chacun, les changements de composé, les baisses d’âge du pneu et les arrêts enregistrés:
Nous consultons ici la course complète pour comprendre la cible et la qualité des données. Les éventuels événements après la coupure ne seront pas utilisés pour calculer les features.

In [71]:
# Examiner les historiques complets uniquement pour diagnostiquer ces deux cas
diagnostic_parts = []

for case in no_stop_none.itertuples(index=False):
    car_history = (
        train_data.loc[
            train_data["race_id"].eq(case.race_id)
            & train_data["driver_code"].eq(case.driver_code)
        ]
        .sort_values("lap_number")
        .copy()
    )

    # Comparer chaque ligne à la précédente disponible
    previous_compound = car_history["compound"].shift(1)
    previous_tyre_age = car_history["tyre_life_laps"].shift(1)

    car_history["compound_changed"] = (
        car_history["compound"].notna()
        & previous_compound.notna()
        & car_history["compound"].ne(previous_compound)
    ).fillna(False)

    car_history["tyre_age_decreased"] = (
        car_history["tyre_life_laps"]
        .lt(previous_tyre_age)
        .fillna(False)
    )

    # Afficher les extrémités de l'historique et les événements à examiner
    event_mask = (
        car_history["official_stop"].fillna(False).astype(bool)
        | car_history["compound_changed"]
        | car_history["tyre_age_decreased"]
    )

    event_mask.iloc[0] = True
    event_mask.iloc[-1] = True

    diagnostic_parts.append(
        car_history.loc[
            event_mask,
            [
                "race_id",
                "driver_code",
                "lap_number",
                "compound",
                "tyre_life_laps",
                "official_stop",
                "compound_changed",
                "tyre_age_decreased",
                "next_compound",
            ],
        ]
    )

display(pd.concat(diagnostic_parts, ignore_index=True))

,race_id,driver_code,lap_number,compound,tyre_life_laps,official_stop,compound_changed,tyre_age_decreased,next_compound
0,2019_21,GAS,1,MEDIUM,1,False,False,False,NONE
1,2019_21,GAS,2,HARD,1,False,True,False,NONE
2,2019_21,GAS,53,HARD,52,False,False,False,NONE
3,2021_16,OCO,1,INTERMEDIATE,1,False,False,False,NONE
4,2021_16,OCO,57,INTERMEDIATE,57,False,False,False,NONE


Un changement de composé apparaît donc dans les données sans arrêt officiel enregistré. Nous ne pouvons pas en déterminer la cause avec ce tableau seul. Nous conservons `n_stops=0` : notre compteur doit rester fidèle à l’indicateur fourni.

La cible `NONE` est également cohérente avec le fonctionnement de `target.py`, qui recherche les arrêts officiels, et non les changements de composé.

Les deux observations sans arrêt visible et de cible `NONE` correspondent à des situations distinctes. L’historique d’OCO est compatible avec un même train d’intermédiaires conservé jusqu’à l’arrivée. Celui de GAS présente un changement `MEDIUM → HARD` sans arrêt officiel enregistré. Nous conservons la définition de `n_stops` fondée sur `official_stop` et décrivons séparément les composés observés.

Commençons par trois indicateurs décrivant les composés secs déjà observés avant la coupure :
- `used_soft`
- `used_medium`
- `used_hard`

Pour un pilote $d$ et un composé $c$ :
$$
\texttt{used}_{d,c}
=
\mathbf 1_{\left\{
\exists\,\ell\in\mathcal V_d(T):
\texttt{compound}_{d,\ell}=c
\right\}}.
$$
**Intuition :** l’indicateur vaut $1$ si nous avons vu ce composé au moins une fois dans l’historique visible, même si la voiture utilise maintenant un autre pneu. Il ne compte pas le nombre de trains utilisés.

Vérifions les deux cas que nous venons d’étudier :

In [72]:
dry_compounds = ["SOFT", "MEDIUM", "HARD"]
tyre_usage_examples = []

for case in no_stop_none.itertuples(index=False):

    # Retrouver l'instant de coupure de cette course
    case_cutoff_time = train_cutoffs.loc[
        train_cutoffs["race_id"].eq(case.race_id),
        "cutoff_time_s",
    ].iloc[0]

    # Utiliser uniquement les tours visibles de cette voiture
    car_visible = train_data.loc[
        train_data["race_id"].eq(case.race_id)
        & train_data["driver_code"].eq(case.driver_code)
        & train_data["race_time_s"].notna()
        & train_data["race_time_s"].le(case_cutoff_time)
    ]

    usage = {
        "race_id": case.race_id,
        "driver_code": case.driver_code,
    }

    # Indiquer si chaque composé sec apparaît dans l'historique
    for compound in dry_compounds:
        usage[f"used_{compound.lower()}"] = int(
            car_visible["compound"].eq(compound).fillna(False).any()
        )

    tyre_usage_examples.append(usage)

display(pd.DataFrame(tyre_usage_examples))

,race_id,driver_code,used_soft,used_medium,used_hard
0,2019_21,GAS,0,1,1
1,2021_16,OCO,0,0,0


Nous retrouvons bien les résultats attendus : `MEDIUM` et `HARD` ont été observés pour GAS, tandis qu’aucun composé sec n’a été observé pour OCO.

Complétons cet historique avec deux variables :
$$
\texttt{n\_dry\_used}
=\texttt{used\_soft}+\texttt{used\_medium}+\texttt{used\_hard},
$$
et `used_wet`, qui vaut $1$ si au moins un tour visible indique `INTERMEDIATE` ou `WET`.

Intuition : `n_dry_used` compte les types de gommes sèches distincts, pas les arrêts ni les trains de pneus. Deux trains de `HARD` ne comptent que pour un type.

In [73]:
# Reprendre les indicateurs calculés pour nos deux exemples
tyre_usage_check = pd.DataFrame(tyre_usage_examples)

# Compter les composés secs distincts observés
tyre_usage_check["n_dry_used"] = tyre_usage_check[
    ["used_soft", "used_medium", "used_hard"]
].sum(axis=1)

wet_usage = []

for case in tyre_usage_check.itertuples(index=False):

    # Retrouver la coupure de la course
    case_cutoff_time = train_cutoffs.loc[
        train_cutoffs["race_id"].eq(case.race_id),
        "cutoff_time_s",
    ].iloc[0]

    # Lire uniquement les composés visibles avant ou à la coupure
    car_visible = train_data.loc[
        train_data["race_id"].eq(case.race_id)
        & train_data["driver_code"].eq(case.driver_code)
        & train_data["race_time_s"].notna()
        & train_data["race_time_s"].le(case_cutoff_time)
    ]

    wet_usage.append(
        int(car_visible["compound"].isin(["INTERMEDIATE", "WET"]).any())
    )

tyre_usage_check["used_wet"] = wet_usage

display(tyre_usage_check)

,race_id,driver_code,used_soft,used_medium,used_hard,n_dry_used,used_wet
0,2019_21,GAS,0,1,1,2,0
1,2021_16,OCO,0,0,0,0,1


Regroupons maintenant ces calculs dans une fonction réutilisable :

In [74]:
def build_tyre_usage(visible):
    """Décrit les types de pneus observés par pilote avant la coupure."""

    if visible.empty or visible["race_id"].nunique() != 1:
        raise ValueError("Une seule course non vide est attendue.")

    records = []

    for driver_code, car_visible in visible.groupby("driver_code"):
        compounds = car_visible["compound"]

        record = {"driver_code": driver_code}

        # Repérer chaque composé sec dans l'historique visible
        for compound in ["SOFT", "MEDIUM", "HARD"]:
            record[f"used_{compound.lower()}"] = int(
                compounds.eq(compound).fillna(False).any()
            )

        # Compter les types de composés secs distincts
        record["n_dry_used"] = (
            record["used_soft"]
            + record["used_medium"]
            + record["used_hard"]
        )

        # Repérer l'utilisation d'intermédiaires ou de pneus pluie
        record["used_wet"] = int(
            compounds.isin(["INTERMEDIATE", "WET"]).any()
        )

        records.append(record)

    return pd.DataFrame(records).set_index("driver_code")

In [75]:
usage_columns = [
    "used_soft",
    "used_medium",
    "used_hard",
    "n_dry_used",
    "used_wet",
]

for case in tyre_usage_check.itertuples(index=False):
    cutoff_time = train_cutoffs.loc[
        train_cutoffs["race_id"].eq(case.race_id),
        "cutoff_time_s",
    ].iloc[0]

    visible = train_data.loc[
        train_data["race_id"].eq(case.race_id)
        & train_data["race_time_s"].notna()
        & train_data["race_time_s"].le(cutoff_time)
    ]

    calculated = build_tyre_usage(visible)

    expected = tyre_usage_check.loc[
        tyre_usage_check["race_id"].eq(case.race_id)
        & tyre_usage_check["driver_code"].eq(case.driver_code),
    ].set_index("driver_code")[usage_columns]

    pd.testing.assert_frame_equal(
        calculated.loc[expected.index, usage_columns],
        expected,
    )

print("La fonction reproduit les cinq indicateurs des deux exemples.")

La fonction reproduit les cinq indicateurs des deux exemples.


Appliquons-la maintenant aux $76$ courses d’entraînement, aux mêmes coupures que les autres features:

Le tableau distinguera notamment :
- un seul composé sec observé, sans pneu pluie ;
- deux ou trois composés secs observés ;
- des pneus intermédiaires ou pluie observés, avec ou sans composés secs.
Nous examinerons d’abord ces effectifs, puis leur association avec la cible `NONE`. Le nombre de types de pneus utilisés et le nombre d’arrêts restent deux informations distinctes.

In [76]:
usage_parts = []

for cutoff in train_cutoffs.itertuples(index=False):

    # Conserver uniquement les tours visibles à cette coupure
    visible = train_data.loc[
        train_data["race_id"].eq(cutoff.race_id)
        & train_data["race_time_s"].notna()
        & train_data["race_time_s"].le(cutoff.cutoff_time_s)
    ].copy()

    # Calculer l'historique des composés pour chaque pilote
    usage_part = build_tyre_usage(visible).reset_index()
    usage_part["race_id"] = cutoff.race_id
    usage_parts.append(usage_part)

tyre_usage_train = pd.concat(usage_parts, ignore_index=True)

# Ajouter les cinq variables aux voitures déjà retenues
previous_row_count = len(train_features)

train_features = (
    train_features
    .drop(columns=usage_columns, errors="ignore")
    .merge(
        tyre_usage_train,
        on=["race_id", "driver_code"],
        how="left",
        validate="one_to_one",
    )
)

# Contrôler la jointure
assert len(train_features) == previous_row_count
assert train_features[usage_columns].notna().all().all()

# Décrire les historiques dans la population évaluable
usage_summary = (
    train_features.loc[train_features["scored"]]
    .groupby(["n_dry_used", "used_wet"])
    .agg(
        observations=("driver_code", "size"),
        courses=("race_id", "nunique"),
    )
)

display(usage_summary)

observations  courses
n_dry_used used_wet                       
0          1                   63        5
1          0                  498       64
           1                   27        4
2          0                  695       69
           1                    7        1
3          0                    6        5

<details>
<summary>Les six groupes totalisent bien nos 1 296 observations. Voici comment lire ce tableau : </summary>

| Composés secs distincts | Pneu pluie déjà observé | Observations | Interprétation |
|---:|---:|---:|---|
| $0$ | Oui | $63$ | Uniquement des intermédiaires ou pneus pluie observés |
| $1$ | Non | $498$ | Un seul type de gomme sèche observé |
| $1$ | Oui | $27$ | Un type sec et au moins un type pluie observés |
| $2$ | Non | $695$ | Deux types de gommes sèches observés |
| $2$ | Oui | $7$ | Deux types secs et au moins un type pluie observés |
| $3$ | Non | $6$ | `SOFT`, `MEDIUM` et `HARD` ont tous été observés |

Le groupe principal est celui des deux composés secs sans pneu pluie : $695$ voitures. Mais cela ne signifie pas forcément deux arrêts : une voiture peut commencer en `MEDIUM`, puis monter des `HARD` lors d’un seul arrêt.

</details>

Examinons maintenant la proportion de `NONE` dans chaque groupe :

Nous estimons ici :
$$
\widehat P(Y=\text{NONE}\mid
\texttt{n\_dry\_used}=n,\ \texttt{used\_wet}=w).
$$
**Intuition :** nous voulons savoir si les voitures ayant déjà utilisé plusieurs types de pneus présentent une fréquence différente d’absence de nouvel arrêt. Nous gardons les effectifs et les courses visibles, car les groupes de $6$ ou $7$ voitures seront particulièrement fragiles à interpréter.

In [77]:
# Examiner les cibles selon l'historique des types de pneus
usage_analysis = train_features.loc[
    train_features["scored"]
].copy()

usage_analysis["no_further_stop"] = (
    usage_analysis["next_compound"].eq("NONE")
)

usage_target_summary = (
    usage_analysis
    .groupby(["n_dry_used", "used_wet"])
    .agg(
        observations=("driver_code", "size"),
        courses=("race_id", "nunique"),
        none_count=("no_further_stop", "sum"),
    )
)

usage_target_summary["none_percentage"] = (
    100
    * usage_target_summary["none_count"]
    / usage_target_summary["observations"]
)

display(usage_target_summary.round(2))

observations  courses  none_count  none_percentage
n_dry_used used_wet                                                    
0          1                   63        5           5             7.94
1          0                  498       64          23             4.62
           1                   27        4          11            40.74
2          0                  695       69         252            36.26
           1                    7        1           4            57.14
3          0                    6        5           2            33.33

<details>
<summary> L’historique des composés distingue nettement certaines situations. </summary>

Le contraste principal concerne les voitures sans pneu pluie observé :
$$
\widehat P(Y=\text{NONE}\mid n_{\mathrm{dry}}=1,\ \texttt{used\_wet}=0)
=\frac{23}{498}\approx4{,}62\%,
$$
contre :
$$
\widehat P(Y=\text{NONE}\mid n_{\mathrm{dry}}=2,\ \texttt{used\_wet}=0)
=\frac{252}{695}\approx36{,}26\%.
$$

**Intuition :** les voitures ayant déjà utilisé deux composés secs sont plus souvent dans une situation où aucun nouvel arrêt ne sera effectué. Mais cet historique reflète aussi les arrêts passés, le pneu actuel et la stratégie : ce tableau ne démontre pas un effet causal.

Autre contraste : avec un seul composé sec observé, la proportion de NONE est de :
- $4,62 \%$ sans pneu pluie observé ;
- $40,74 \%$ avec un pneu pluie déjà observé.
Cela justifie de conserver `used_wet` séparément de `n_dry_used`. Le second résultat repose toutefois sur seulement $27$ voitures dans $4$ courses.

Enfin, les $57,14 \%$ du groupe « deux composés secs et pneu pluie » correspondent à $4$ voitures sur $7$, dans une seule course : nous ne pouvons pas en tirer une règle générale.

</details>
 


>La fréquence de `NONE` varie selon le nombre de composés secs distincts et l’utilisation passée de pneus intermédiaires ou pluie. Ces variables décrivent des aspects complémentaires de la stratégie observée. Elles sont conservées comme candidates, sans imposer de règle déterministe à partir de ces proportions.

Nous avons justifié les cinq variables d’usage des pneus. La prochaine information à examiner sera le composé de départ, `start_compound` : deux voitures peuvent avoir utilisé les mêmes composés, mais dans un ordre différent. Par exemple, `MEDIUM → HARD` et `HARD → MEDIUM` donnent les mêmes indicateurs d’usage, mais pas le même composé de départ.


**Intuition :** les indicateurs `used_medium=1` et `used_hard=1` ne disent pas par quel composé la voiture a commencé. `start_compound` conserve cette information.

Nous adoptons une règle prudente :
$$
\texttt{start\_compound}_d=
\begin{cases}
\text{composé observé au tour 1}, & \text{si disponible},\\
\text{NA}, & \text{sinon}.
\end{cases}
$$
Si la première ligne disponible est au tour $3$, nous ne supposons pas que son composé était celui du départ. De même, un composé manquant au tour $1$ reste inconnu.

Vérifions cette variable sur notre exemple 2018_01, avec `stop_visible` qui contient déjà les tours visibles :

Nous créons ainsi une variable catégorielle et un indicateur d’absence. Pour l’instant, nous gardons les catégories d’origine, notamment `INTERMEDIATE` et `WET`.

Précision : il s’agit du composé enregistré au premier tour, utilisé comme représentation du composé de départ. Un changement pendant ce premier tour pourrait limiter cette interprétation.

In [78]:
# Identifier la première ligne visible de chaque pilote
first_visible_rows = (
    stop_visible
    .sort_values(["driver_code", "lap_number"])
    .drop_duplicates("driver_code", keep="first")
    .set_index("driver_code")
)

# Retenir le composé uniquement si cette première ligne correspond au tour 1
start_compound = (
    first_visible_rows["compound"]
    .where(first_visible_rows["lap_number"].eq(1))
    .astype("string")
    .rename("start_compound")
)

# Comparer le composé de départ au composé actuel des voitures retenues
start_comparison = (
    train_features.loc[
        train_features["race_id"].eq("2018_01"),
        ["driver_code", "compound", "n_stops"],
    ]
    .merge(
        start_compound,
        left_on="driver_code",
        right_index=True,
        how="left",
        validate="one_to_one",
    )
)

# Signaler explicitement les composés de départ inconnus
start_comparison["start_compound_missing"] = (
    start_comparison["start_compound"].isna().astype(int)
)

display(start_comparison)

,driver_code,compound,n_stops,start_compound,start_compound_missing
0,ALO,HARD,1,SOFT,0
1,BOT,MEDIUM,1,SOFT,0
2,HAM,HARD,1,SOFT,0
3,HAR,SOFT,2,<NA>,1
4,HUL,HARD,1,SOFT,0
5,LEC,HARD,2,MEDIUM,0
6,OCO,HARD,1,SOFT,0
7,PER,HARD,1,SOFT,0
8,RAI,HARD,1,SOFT,0
9,RIC,HARD,1,MEDIUM,0


Le résultat distingue bien le composé du premier tour du composé actuel. Par exemple :
- ALO : `SOFT` au premier tour, `HARD` à la coupure ;
- BOT : `SOFT` au premier tour, `MEDIUM` à la coupure ;
- LEC : `MEDIUM` au premier tour, `HARD` à la coupure.

Pour HAR, `start_compound` reste inconnu et `start_compound_missing=1`. Nous ne le remplaçons pas par son composé actuel : ses deux arrêts observés ne permettent pas de retrouver automatiquement son pneu de départ.

Regroupons ce calcul dans une fonction :

In [79]:
def build_start_compound(visible):
    """Extrait le composé du premier tour et signale son absence."""

    if visible.empty or visible["race_id"].nunique() != 1:
        raise ValueError("Une seule course non vide est attendue.")

    if visible.duplicated(["driver_code", "lap_number"]).any():
        raise ValueError("Plusieurs lignes existent pour un même pilote-tour.")

    # Conserver la première ligne, même si son composé est manquant
    first_rows = (
        visible
        .sort_values(["driver_code", "lap_number"])
        .drop_duplicates("driver_code", keep="first")
        .set_index("driver_code")
    )

    features = pd.DataFrame(index=first_rows.index)

    # Ne pas utiliser un tour ultérieur pour inventer le composé de départ
    features["start_compound"] = (
        first_rows["compound"]
        .where(first_rows["lap_number"].eq(1))
        .astype("string")
    )

    features["start_compound_missing"] = (
        features["start_compound"].isna().astype(int)
    )

    return features

In [80]:
start_features_check = build_start_compound(stop_visible)

start_columns = [
    "start_compound",
    "start_compound_missing",
]

expected_start = (
    start_comparison
    .set_index("driver_code")[start_columns]
    .sort_index()
)

pd.testing.assert_frame_equal(
    start_features_check.loc[expected_start.index, start_columns],
    expected_start,
)

print("La fonction reproduit les deux variables de composé de départ.")

La fonction reproduit les deux variables de composé de départ.


Appliquons-la aux $76$ courses d’entraînement, puis vérifions combien de composés de départ restent inconnus.

- `dropna=False` permet de garder les composés inconnus dans le tableau. Leur absence fait partie du diagnostic : nous ne les supprimons pas et ne leur attribuons pas arbitrairement un composé.

In [81]:
start_parts = []

for cutoff in train_cutoffs.itertuples(index=False):

    # Conserver uniquement les tours visibles à la coupure
    visible = train_data.loc[
        train_data["race_id"].eq(cutoff.race_id)
        & train_data["race_time_s"].notna()
        & train_data["race_time_s"].le(cutoff.cutoff_time_s)
    ].copy()

    # Extraire le composé du premier tour pour chaque pilote
    start_part = build_start_compound(visible).reset_index()
    start_part["race_id"] = cutoff.race_id
    start_parts.append(start_part)

start_features_train = pd.concat(start_parts, ignore_index=True)

start_columns = [
    "start_compound",
    "start_compound_missing",
]

# Ajouter les deux variables sans dupliquer les voitures
previous_row_count = len(train_features)

train_features = (
    train_features
    .drop(columns=start_columns, errors="ignore")
    .merge(
        start_features_train,
        on=["race_id", "driver_code"],
        how="left",
        validate="one_to_one",
    )
)

# Vérifier la jointure et la cohérence de l'indicateur
assert len(train_features) == previous_row_count
assert train_features["start_compound_missing"].notna().all()
assert (
    train_features["start_compound_missing"]
    .eq(train_features["start_compound"].isna().astype(int))
    .all()
)

# Décrire les composés de départ dans la population évaluable
start_summary = (
    train_features.loc[train_features["scored"]]
    .groupby("start_compound", dropna=False)
    .agg(
        observations=("driver_code", "size"),
        courses=("race_id", "nunique"),
    )
)

display(start_summary)

,observations,courses
start_compound,,
HARD,118,44
INTERMEDIATE,91,6
MEDIUM,734,70
SOFT,326,56
WET,2,1
<NA>,25,5


Les effectifs totalisent bien $1 296$ observations. Le composé du premier tour est connu pour $1 271$ voitures, et inconnu pour $25$, réparties sur $5$ courses :
$$
\frac{25}{1296}\times100\approx1{,}93\%.
$$
Le `MEDIUM` est le composé de départ le plus représenté : $734$ observations. À l’inverse, `WET` ne concerne que $2$ observations dans une seule course : ce groupe sera trop petit pour tirer des conclusions solides à lui seul.

Avant de traiter les $25$ inconnues, distinguons deux causes possibles :
1. aucun tour $1$ n’est présent dans l’historique visible ;
2. le tour $1$ est présent, mais son composé est manquant.

**Intuition :** si notre première observation est au tour $3$, son pneu ne renseigne pas forcément le départ. Si le tour $1$ est présent avec un composé inconnu, regarder le tour $2$ reviendrait également à faire une supposition.

Nous conservons donc les `NA` à ce stade. Ce contrôle permettra de documenter précisément leur origine dans l’historique visible, sans modifier les données.

In [82]:
# Identifier les voitures évaluables dont le composé de départ est inconnu
missing_start_cars = train_features.loc[
    train_features["scored"]
    & train_features["start_compound_missing"].eq(1),
    ["race_id", "driver_code"],
]

start_missing_records = []

for car in missing_start_cars.itertuples(index=False):

    cutoff_time = train_cutoffs.loc[
        train_cutoffs["race_id"].eq(car.race_id),
        "cutoff_time_s",
    ].iloc[0]

    # Retrouver uniquement l'historique visible de cette voiture
    car_visible = train_data.loc[
        train_data["race_id"].eq(car.race_id)
        & train_data["driver_code"].eq(car.driver_code)
        & train_data["race_time_s"].notna()
        & train_data["race_time_s"].le(cutoff_time)
    ].sort_values("lap_number")

    first_row = car_visible.iloc[0]

    start_missing_records.append({
        "race_id": car.race_id,
        "driver_code": car.driver_code,
        "first_visible_lap": first_row["lap_number"],
        "first_visible_compound": first_row["compound"],
        "missing_reason": (
            "FIRST_LAP_NOT_VISIBLE"
            if first_row["lap_number"] != 1
            else "FIRST_LAP_COMPOUND_MISSING"
        ),
    })

start_missing_audit = pd.DataFrame(start_missing_records)

display(start_missing_audit)
display(
    start_missing_audit.groupby("missing_reason")
    .agg(
        observations=("driver_code", "size"),
        courses=("race_id", "nunique"),
    )
)

,race_id,driver_code,first_visible_lap,first_visible_compound,missing_reason
0,2018_01,HAR,1,NaN,FIRST_LAP_COMPOUND_MISSING
1,2018_04,ALO,1,NaN,FIRST_LAP_COMPOUND_MISSING
2,2018_04,ERI,1,NaN,FIRST_LAP_COMPOUND_MISSING
3,2018_04,HAR,1,NaN,FIRST_LAP_COMPOUND_MISSING
4,2018_04,MAG,1,NaN,FIRST_LAP_COMPOUND_MISSING
5,2018_04,RAI,1,NaN,FIRST_LAP_COMPOUND_MISSING
6,2018_05,HAR,1,NaN,FIRST_LAP_COMPOUND_MISSING
7,2018_05,SIR,1,NaN,FIRST_LAP_COMPOUND_MISSING
8,2018_07,ERI,1,NaN,FIRST_LAP_COMPOUND_MISSING
9,2018_07,VAN,1,NaN,FIRST_LAP_COMPOUND_MISSING


,observations,courses
missing_reason,,
FIRST_LAP_COMPOUND_MISSING,25,5


Les $25$ valeurs manquantes ont toutes la même origine : le tour $1$ est visible, mais son composé est absent. Aucune ne provient ici d’un historique commençant après le premier tour.

Nous observons également une concentration :
- les $25$ cas appartiennent à $5$ courses de $2018$ ;
- $15$ concernent $2018\_13$, soit $60 \%$ des cas.
Cela suggère une absence liée à certains contextes de collecte, mais ne suffit pas à identifier sa cause ni à établir son mécanisme statistique.

Notre décision reste :
- conserver `start_compound = NA` ;
- conserver `start_compound_missing = 1` ;
- ne pas supprimer ces voitures ;
- ne pas remplacer automatiquement le composé initial par celui du tour suivant.
Lors du prétraitement, nous pourrons encoder ce composé inconnu avec une catégorie `UNKNOWN`. Ce sera une manière de représenter l’absence, pas de reconstituer le pneu réellement utilisé.

**Une conséquence** concerne aussi `used_soft`, `used_medium`, `used_hard` et `n_dry_used` : si un composé n’a été utilisé qu’au premier tour et que celui-ci est inconnu, notre historique ne pourra pas le compter. Ces variables décrivent donc bien les composés observés, pas nécessairement tous les composés réellement utilisés.

**Peloton à la coupure :** quels pneus les autres voitures utilisent-elles et combien ont déjà effectué un arrêt ?

Pour une course $r$, notons $\mathcal D_r(T)$ les voitures retenues. La part utilisant le composé $c$ est :
$$
\texttt{field\_on}_{r,c}
=
\frac{
\sum_{d\in\mathcal D_r(T)}
\mathbf 1\{\texttt{compound}_d=c\}
}{
|\mathcal D_r(T)|
}.
$$

**Intuition :** si 12 des 15 voitures retenues sont en `HARD`, chaque voiture reçoit `field_on_hard` $= 12/15 = 0.8$. Cette variable décrit le contexte collectif.

Nous utilisons toutes les voitures retenues, y compris celles qui abandonneront ensuite : leur résultat final ne doit pas servir à reconstruire le peloton visible.

In [83]:
# Examiner le peloton de notre course d'exemple
field_example = train_features.loc[
    train_features["race_id"].eq("2018_01")
].copy()

# Regrouper intermédiaires et pneus pluie pour ce résumé collectif
field_compound = field_example["compound"].replace({
    "INTERMEDIATE": "WET_OR_INTER",
    "WET": "WET_OR_INTER",
})

field_classes = ["SOFT", "MEDIUM", "HARD", "WET_OR_INTER"]

field_counts = (
    field_compound.value_counts()
    .reindex(field_classes, fill_value=0)
)

field_summary = pd.DataFrame({
    "cars": field_counts,
    "share": field_counts / len(field_example),
})

field_summary.index.name = "compound"

print("Voitures retenues :", len(field_example))
print("Composés inconnus :", int(field_compound.isna().sum()))
display(field_summary)

Voitures retenues : 15
Composés inconnus : 0


,cars,share
compound,,
SOFT,1,0.066667
MEDIUM,2,0.133333
HARD,12,0.800000
WET_OR_INTER,0,0.000000


Complétons ce contexte par deux variables :
$$
\texttt{field\_share\_stopped}
=\frac{1}{N}\sum_{d=1}^{N}\mathbf 1\{n_{\mathrm{stops},d}>0\},
$$
$$
\texttt{field\_mean\_stops}
=\frac{1}{N}\sum_{d=1}^{N}n_{\mathrm{stops},d}.
$$
- La première indique la proportion de voitures ayant au moins un arrêt observé.
- La seconde indique le nombre moyen d’arrêts observés par voiture.

In [84]:
# Résumer les arrêts des voitures retenues, sans filtrer sur scored
field_stop_summary = pd.DataFrame([{
    "retained_cars": len(field_example),
    "cars_with_stop": int(field_example["n_stops"].gt(0).sum()),
    "total_observed_stops": int(field_example["n_stops"].sum()),
    "field_share_stopped": field_example["n_stops"].gt(0).mean(),
    "field_mean_stops": field_example["n_stops"].mean(),
}])

display(field_stop_summary)

,retained_cars,cars_with_stop,total_observed_stops,field_share_stopped,field_mean_stops
0,15,15,17,1.0,1.133333


Regroupons maintenant les six variables du peloton dans une fonction. Elle reçoit une table contenant une ligne par voiture retenue, sans filtrage sur `scored`.

In [85]:
def build_field_features(snapshot):
    """Résume les pneus actuels et les arrêts du peloton retenu."""

    if snapshot.empty or snapshot["race_id"].nunique() != 1:
        raise ValueError("Une seule course non vide est attendue.")

    if snapshot["driver_code"].duplicated().any():
        raise ValueError("Une seule ligne par pilote est attendue.")

    if snapshot["n_stops"].isna().any():
        raise ValueError("Le nombre d'arrêts doit être renseigné.")

    # Regrouper les deux types de pneus pluie
    compound_now = snapshot["compound"].replace({
        "INTERMEDIATE": "WET_OR_INTER",
        "WET": "WET_OR_INTER",
    })

    values = {}

    # Garder toutes les voitures retenues au dénominateur
    for compound in ["SOFT", "MEDIUM", "HARD", "WET_OR_INTER"]:
        values[f"field_on_{compound.lower()}"] = (
            compound_now.eq(compound).fillna(False).sum()
            / len(snapshot)
        )

    # Distinguer la présence d'un arrêt du nombre d'arrêts
    values["field_share_stopped"] = snapshot["n_stops"].gt(0).mean()
    values["field_mean_stops"] = snapshot["n_stops"].mean()

    return pd.DataFrame([values])

In [86]:
field_features_check = build_field_features(field_example)

expected_values = [
    1 / 15,   # SOFT
    2 / 15,   # MEDIUM
    12 / 15,  # HARD
    0,        # WET_OR_INTER
    1,        # Part des voitures avec au moins un arrêt
    17 / 15,  # Nombre moyen d'arrêts
]

np.testing.assert_allclose(
    field_features_check.iloc[0].to_numpy(dtype=float),
    expected_values,
)

display(field_features_check)
print("La fonction reproduit les six variables du peloton.")

,field_on_soft,field_on_medium,field_on_hard,field_on_wet_or_inter,field_share_stopped,field_mean_stops
0,0.066667,0.133333,0.8,0.0,1.0,1.133333


La fonction reproduit les six variables du peloton.


La fonction renvoie une ligne par course–coupure. Nous associerons ensuite cette ligne à toutes les voitures concernées, avec une jointure `many_to_one`.

**À noter pour la sélection finale : lorsque tous les composés sont connus, les quatre proportions somment à $1$. Cette dépendance devra être prise en compte pour les modèles sensibles aux redondances, notamment LDA et QDA.**

Appliquons-la aux $76$ courses d’entraînement, puis associons les résultats à chaque voiture.

Nous utilisons toutes les voitures retenues, sans filtrer sur `scored`, pour décrire le peloton.

Pourquoi résumer `field_features_train` plutôt que `train_features` ? Les valeurs du peloton sont répétées pour chaque voiture après la jointure. Ici, nous voulons donner le même poids à chaque course pour examiner leur distribution.

La jointure sur `race_id` convient parce que nous avons une seule coupure par course. Si nous ajoutons plusieurs coupures, il faudra également utiliser un identifiant de coupure.

In [87]:
field_parts = []

# Une seule coupure par course dans notre table actuelle
for race_id, race_snapshot in train_features.groupby("race_id"):
    field_part = build_field_features(race_snapshot)
    field_part["race_id"] = race_id
    field_parts.append(field_part)

field_features_train = pd.concat(field_parts, ignore_index=True)

field_columns = [
    "field_on_soft",
    "field_on_medium",
    "field_on_hard",
    "field_on_wet_or_inter",
    "field_share_stopped",
    "field_mean_stops",
]

# Associer une ligne de contexte à plusieurs voitures de la même course
previous_row_count = len(train_features)

train_features = (
    train_features
    .drop(columns=field_columns, errors="ignore")
    .merge(
        field_features_train,
        on="race_id",
        how="left",
        validate="many_to_one",
    )
)

# Vérifier la couverture et l'absence de duplication
assert len(field_features_train) == 76
assert len(train_features) == previous_row_count
assert train_features[field_columns].notna().all().all()

print("Courses avec variables de peloton :", len(field_features_train))

# Décrire les variables une fois par course, sans répéter chaque voiture
display(
    field_features_train[field_columns]
    .agg(["min", "median", "max"])
    .round(3)
)

Courses avec variables de peloton : 76


,field_on_soft,field_on_medium,field_on_hard,field_on_wet_or_inter,field_share_stopped,field_mean_stops
min,0.000,0.000,0.000,0.0,0.050,0.050
median,0.000,0.372,0.513,0.0,0.778,0.784
max,0.562,0.955,1.000,1.0,1.000,3.062


<details>
<summary> Quelques lectures utiles :</summary>

- `field_on_hard` atteint $1$ : à certaines coupures, toutes les voitures retenues roulent en `HARD`.
- `field_on_wet_or_inter` atteint aussi $1$ : à d’autres coupures, toutes utilisent des intermédiaires ou pneus pluie.
- La médiane de `field_on_soft` vaut $0$ : au moins la moitié des coupures ne présentent aucune voiture retenue en SOFT.
- `field_share_stopped` varie de $5 \%$ à $100 \%$ : les coupures capturent des états très différents de l’avancement des arrêts.

Les médianes sont calculées colonne par colonne. Elles ne décrivent donc pas nécessairement une même course et les quatre médianes de proportions n’ont pas à totaliser $1$.

Le maximum de `field_mean_stops` $= 3,062$ mérite un contrôle : il signifie qu’à une coupure, les voitures retenues ont effectué en moyenne un peu plus de trois arrêts officiels. Ce n’est pas impossible par définition, mais examinons les courses concernées avant de l’interpréter.

**Intuition :** une moyenne élevée peut refléter une stratégie collective ou un contexte particulier, mais ce résumé seul ne permet pas d’en attribuer la cause à la pluie ou à une neutralisation. Nous identifions d’abord les courses et leurs coupures.
</details>

In [88]:
# Identifier les courses avec le plus d'arrêts moyens à la coupure
high_stop_courses = (
    field_features_train[
        ["race_id", "field_share_stopped", "field_mean_stops"]
    ]
    .merge(
        train_cutoffs[
            ["race_id", "cutoff_lap", "cutoff_progress", "retained_cars"]
        ],
        on="race_id",
        validate="one_to_one",
    )
    .sort_values("field_mean_stops", ascending=False)
)

display(high_stop_courses.head(5))

,race_id,field_share_stopped,field_mean_stops,cutoff_lap,cutoff_progress,retained_cars
48,2023_20,1.0,3.062500,35,0.492958,16
24,2021_11,1.0,2.714286,27,0.385714,14
30,2021_21,1.0,2.437500,17,0.340000,16
29,2021_19,1.0,2.400000,24,0.338028,20
18,2020_15,1.0,1.944444,24,0.421053,18


La moyenne maximale concerne 2023_20, à la coupure du tour $35$, soit $49,3 \%$ de la distance prévue.

Les $16$ voitures retenues ont toutes au moins un arrêt observé. Leur nombre total d’arrêts est :
$$
16\times3{,}0625=49.
$$
Cela ne signifie pas que chaque voiture a effectué trois arrêts : $49$ est la somme des compteurs individuels.

Vérifions leur répartition et la correspondance avec les lignes `official_stop` visibles :

In [89]:
# Examiner la course présentant la moyenne d'arrêts la plus élevée
audit_race_id = "2023_20"

audit_cars = train_features.loc[
    train_features["race_id"].eq(audit_race_id),
    ["driver_code", "lap_number", "n_stops"],
].copy()

audit_cutoff_time = train_cutoffs.loc[
    train_cutoffs["race_id"].eq(audit_race_id),
    "cutoff_time_s",
].iloc[0]

# Retrouver les arrêts visibles des seules voitures retenues
observed_stops = train_data.loc[
    train_data["race_id"].eq(audit_race_id)
    & train_data["driver_code"].isin(audit_cars["driver_code"])
    & train_data["race_time_s"].notna()
    & train_data["race_time_s"].le(audit_cutoff_time)
    & train_data["official_stop"].fillna(False).astype(bool),
    ["driver_code", "lap_number"],
]

# Vérifier les compteurs individuels
direct_counts = (
    observed_stops.groupby("driver_code")
    .size()
    .reindex(audit_cars["driver_code"], fill_value=0)
)

assert np.array_equal(
    direct_counts.to_numpy(),
    audit_cars["n_stops"].to_numpy(),
)

display(
    audit_cars.groupby("n_stops")
    .size()
    .rename("cars")
    .to_frame()
)

print("Total d'arrêts visibles :", len(observed_stops))
print("Moyenne par voiture :", audit_cars["n_stops"].mean())

,cars
n_stops,
3,15
4,1


Total d'arrêts visibles : 49
Moyenne par voiture : 3.0625


>Pour 2023_20, à la coupure du tour $35$, les $16$ voitures retenues totalisent $49$ arrêts officiels visibles : quinze voitures en ont trois et une en a quatre. La moyenne de $3,0625$ est conforme aux données sources. Nous la conservons sans correction ; ce contrôle ne permet pas, à lui seul, d’expliquer les arrêts ni de valider leur annotation.

La prochaine information sera complémentaire : quels composés ont été montés après les arrêts déjà observés ?
- `field_on_hard` décrit les voitures actuellement en `HARD` ;
- `field_fitted_hard` décrira la part des arrêts dont le composé monté est connu ayant conduit à monter des `HARD`.
Pour connaître ce composé, nous utiliserons le tour $S+1$ après un arrêt au tour $S$, uniquement si ce tour suivant est déjà visible à la coupure.

Vérifions cela sur 2018_01, avec l’historique `stop_visible` déjà construit.

Pour chaque arrêt au tour $S$, nous cherchons le composé du tour exact $S+1$ parmi les lignes visibles. Si cette ligne est absente ou son composé inconnu, nous conservons une valeur manquante.

**Pourquoi une jointure left ?** Pour conserver aussi les arrêts dont le composé monté n’est pas encore connu. Ils restent des arrêts observés, même si nous ne pouvons pas encore les utiliser dans la répartition des pneus montés.

Pour l’arrêt de LEC au tour $27$, nous attendons un composé manquant, puisque son tour $28$ n’est pas visible à cette coupure.

Ce calcul porte ici sur tous les pilotes de `stop_visible`, y compris ceux exclus ensuite de la population à prédire : leurs choix passés, lorsqu’ils sont visibles, font partie de l’information disponible.

In [90]:
# Extraire les arrêts officiels visibles
visible_stops = stop_visible.loc[
    stop_visible["official_stop"].fillna(False).astype(bool),
    ["driver_code", "lap_number"],
].copy()

visible_stops["lap_after_stop"] = visible_stops["lap_number"] + 1

# Préparer les composés des seuls tours visibles
visible_compounds = stop_visible[
    ["driver_code", "lap_number", "compound"]
].rename(columns={
    "lap_number": "lap_after_stop",
    "compound": "fitted_compound",
})

# Chercher le composé au tour exact qui suit chaque arrêt
fitted_stops = visible_stops.merge(
    visible_compounds,
    on=["driver_code", "lap_after_stop"],
    how="left",
    validate="many_to_one",
)

display(
    fitted_stops.sort_values(["driver_code", "lap_number"])
)

,driver_code,lap_number,lap_after_stop,fitted_compound
0,ALO,26,27,HARD
1,BOT,25,26,MEDIUM
2,GRO,24,25,NaN
3,HAM,19,20,HARD
4,HAR,1,2,HARD
5,HAR,22,23,SOFT
6,HUL,24,25,HARD
7,LEC,20,21,HARD
8,LEC,27,28,NaN
9,MAG,22,23,NaN


>Nous gardons ces valeurs manquantes : elles ne signifient ni NONE, ni absence de changement de pneus.

>Les 19 arrêts diffèrent des 17 comptés précédemment parce que nous incluons ici les arrêts visibles de GRO et MAG, même s’ils ne font plus partie des voitures retenues à cette coupure.

Calculons maintenant la répartition parmi les 16 arrêts dont le composé est connu :

In [91]:
# Regrouper les intermédiaires et les pneus pluie
fitted_classes = ["SOFT", "MEDIUM", "HARD", "WET_OR_INTER"]

fitted_compounds = fitted_stops["fitted_compound"].replace({
    "INTERMEDIATE": "WET_OR_INTER",
    "WET": "WET_OR_INTER",
})

# Retenir les arrêts dont le composé est identifiable
known_fitted = fitted_compounds.loc[
    fitted_compounds.isin(fitted_classes)
]

fitted_counts = (
    known_fitted.value_counts()
    .reindex(fitted_classes, fill_value=0)
)

fitted_summary = pd.DataFrame({
    "stops": fitted_counts,
    "share": (
        fitted_counts / len(known_fitted)
        if len(known_fitted) > 0
        else np.nan
    ),
})

print("Arrêts visibles :", len(fitted_stops))
print("Arrêts avec composé connu :", len(known_fitted))
display(fitted_summary)

Arrêts visibles : 19
Arrêts avec composé connu : 16


,stops,share
fitted_compound,,
SOFT,1,0.0625
MEDIUM,2,0.1250
HARD,13,0.8125
WET_OR_INTER,0,0.0000


>La répartition est confirmée : $13$ `HARD`, $2$ `MEDIUM` et $1$ `SOFT` parmi les $16$ arrêts dont le composé est connu.

Regroupons ce calcul dans une fonction produisant cinq variables : les quatre proportions et `field_stops_seen`, leur effectif de référence.

In [92]:
def build_fitted_features(visible):
    """Résume les composés montés après les arrêts déjà observables."""

    if visible.empty or visible["race_id"].nunique() != 1:
        raise ValueError("Une seule course non vide est attendue.")

    if visible["official_stop"].isna().any():
        raise ValueError("L'indicateur d'arrêt contient des valeurs manquantes.")

    if visible.duplicated(["driver_code", "lap_number"]).any():
        raise ValueError("Plusieurs lignes existent pour un même pilote-tour.")

    # Extraire les arrêts visibles
    stops = visible.loc[
        visible["official_stop"].astype(bool),
        ["driver_code", "lap_number"],
    ].copy()

    stops["lap_after_stop"] = stops["lap_number"] + 1

    # Chercher le composé au tour exact suivant, uniquement s'il est visible
    visible_tyres = visible[
        ["driver_code", "lap_number", "compound"]
    ].rename(columns={
        "lap_number": "lap_after_stop",
        "compound": "fitted_compound",
    })

    stops = stops.merge(
        visible_tyres,
        on=["driver_code", "lap_after_stop"],
        how="left",
        validate="many_to_one",
    )

    fitted = stops["fitted_compound"].replace({
        "INTERMEDIATE": "WET_OR_INTER",
        "WET": "WET_OR_INTER",
    })

    tyre_classes = ["SOFT", "MEDIUM", "HARD", "WET_OR_INTER"]
    known_fitted = fitted.loc[fitted.isin(tyre_classes)]

    counts = known_fitted.value_counts().reindex(
        tyre_classes, fill_value=0
    )

    values = {"field_stops_seen": len(known_fitted)}

    for compound in tyre_classes:
        values[f"field_fitted_{compound.lower()}"] = (
            counts[compound] / len(known_fitted)
            if len(known_fitted) > 0
            else np.nan
        )

    return pd.DataFrame([values])

La fonction reçoit uniquement l’historique visible et renvoie une ligne par course–coupure, à partager ensuite entre les voitures retenues.

In [93]:

fitted_features_check = build_fitted_features(stop_visible)

assert fitted_features_check.loc[0, "field_stops_seen"] == 16

np.testing.assert_allclose(
    fitted_features_check[
        [
            "field_fitted_soft",
            "field_fitted_medium",
            "field_fitted_hard",
            "field_fitted_wet_or_inter",
        ]
    ].iloc[0].to_numpy(dtype=float),
    [1 / 16, 2 / 16, 13 / 16, 0],
)

display(fitted_features_check)
print("La fonction reproduit les cinq variables des composés montés.")

,field_stops_seen,field_fitted_soft,field_fitted_medium,field_fitted_hard,field_fitted_wet_or_inter
0,16,0.0625,0.125,0.8125,0.0


La fonction reproduit les cinq variables des composés montés.


Appliquons-la maintenant aux $76$ courses d’entraînement, puis vérifions où les proportions peuvent être calculées.

Nous ne demandons pas que toutes les proportions soient renseignées. Leur absence est justifiée lorsqu’aucun composé après arrêt n’est connu ; dans ce cas, `field_stops_seen=0` explique cette absence.

Ce compteur permettra aussi de distinguer une proportion de `HARD` de $100 \%$ fondée sur un seul arrêt de la même proportion fondée sur vingt arrêts.

In [94]:
fitted_parts = []

for cutoff in train_cutoffs.itertuples(index=False):

    # Utiliser tous les tours visibles, sans consulter le futur caché
    visible = train_data.loc[
        train_data["race_id"].eq(cutoff.race_id)
        & train_data["race_time_s"].notna()
        & train_data["race_time_s"].le(cutoff.cutoff_time_s)
    ].copy()

    fitted_part = build_fitted_features(visible)
    fitted_part["race_id"] = cutoff.race_id
    fitted_parts.append(fitted_part)

fitted_features_train = pd.concat(fitted_parts, ignore_index=True)

fitted_share_columns = [
    "field_fitted_soft",
    "field_fitted_medium",
    "field_fitted_hard",
    "field_fitted_wet_or_inter",
]

fitted_columns = ["field_stops_seen"] + fitted_share_columns

# Associer le contexte de chaque course aux voitures retenues
previous_row_count = len(train_features)

train_features = (
    train_features
    .drop(columns=fitted_columns, errors="ignore")
    .merge(
        fitted_features_train,
        on="race_id",
        how="left",
        validate="many_to_one",
    )
)

assert len(fitted_features_train) == 76
assert len(train_features) == previous_row_count
assert train_features["field_stops_seen"].notna().all()

# Vérifier les proportions selon la disponibilité des composés
has_known_stops = fitted_features_train["field_stops_seen"].gt(0)

assert fitted_features_train.loc[
    has_known_stops, fitted_share_columns
].notna().all().all()

np.testing.assert_allclose(
    fitted_features_train.loc[
        has_known_stops, fitted_share_columns
    ].sum(axis=1),
    1,
)

assert fitted_features_train.loc[
    ~has_known_stops, fitted_share_columns
].isna().all().all()

print("Courses avec au moins un composé après arrêt connu :",
      int(has_known_stops.sum()))
print("Courses sans composé après arrêt connu :",
      int((~has_known_stops).sum()))

display(
    fitted_features_train["field_stops_seen"]
    .agg(["min", "median", "max"])
    .to_frame("known_stops")
)

Courses avec au moins un composé après arrêt connu : 74
Courses sans composé après arrêt connu : 2


,known_stops
min,0.0
median,14.5
max,52.0


Les proportions sont calculables pour 74 courses sur 76. Pour les deux autres, aucun composé après arrêt n’est identifiable dans l’historique visible.

Nous conservons donc :
- `field_stops_seen = 0` ;
- les quatre field_fitted_* à `NA`.
Cela ne signifie pas forcément qu’aucun arrêt n’a eu lieu. Un arrêt peut être visible alors que le tour suivant ou son composé  ne l’est pas encore.

La médiane de $14,5$ est normale : avec $76$ courses, elle est la moyenne des deux valeurs centrales. Elle ne représente pas un « demi-arrêt » réel.

Identifions les deux courses concernées :

Ce contrôle distinguera « aucun arrêt visible » de « arrêt visible, mais composé suivant inconnu ». Dans les deux cas, une répartition des composés montés ne peut pas être estimée ; remplacer les quatre proportions par zéro masquerait cette absence d’information.

      


In [95]:
# Repérer les coupures sans composé identifiable après un arrêt
no_known_fitted = (
    fitted_features_train.loc[
        fitted_features_train["field_stops_seen"].eq(0),
        ["race_id", "field_stops_seen"],
    ]
    .merge(
        train_cutoffs[
            ["race_id", "cutoff_lap", "cutoff_time_s", "retained_cars"]
        ],
        on="race_id",
        validate="one_to_one",
    )
)

# Compter tous les arrêts visibles, même sans composé identifiable après
visible_stop_counts = []

for cutoff in no_known_fitted.itertuples(index=False):
    visible = train_data.loc[
        train_data["race_id"].eq(cutoff.race_id)
        & train_data["race_time_s"].notna()
        & train_data["race_time_s"].le(cutoff.cutoff_time_s)
    ]

    visible_stop_counts.append(
        int(visible["official_stop"].sum())
    )

no_known_fitted["visible_stops"] = visible_stop_counts

display(no_known_fitted)

,race_id,field_stops_seen,cutoff_lap,cutoff_time_s,retained_cars,visible_stops
0,2021_16,0,22,2070.308,20,1
1,2026_03,0,17,1616.566,22,2


Les quatre proportions `field_fitted_*` restent indéfinies, car leur dénominateur est nul. Nous conservons les NA avec field_stops_seen=0.

Ce tableau ne précise pas encore pourquoi le composé est inconnu : le tour $S+1$ peut être absent de l’historique visible, ou présent avec un composé manquant. Pour distinguer les deux :

Nous restons dans les données visibles : ce diagnostic ne cherche pas le composé dans les tours cachés. Il explique seulement pourquoi notre feature est manquante à la coupure.

In [96]:
fitted_missing_details = []

for cutoff in no_known_fitted.itertuples(index=False):
    visible = train_data.loc[
        train_data["race_id"].eq(cutoff.race_id)
        & train_data["race_time_s"].notna()
        & train_data["race_time_s"].le(cutoff.cutoff_time_s)
    ].copy()

    # Repérer les arrêts visibles
    stops = visible.loc[
        visible["official_stop"].astype(bool),
        ["driver_code", "lap_number"],
    ].copy()

    stops["lap_after_stop"] = stops["lap_number"] + 1

    # Identifier explicitement la présence du tour suivant
    following_laps = visible[
        ["driver_code", "lap_number", "compound"]
    ].rename(columns={
        "lap_number": "lap_after_stop",
        "compound": "fitted_compound",
    })

    details = stops.merge(
        following_laps,
        on=["driver_code", "lap_after_stop"],
        how="left",
        validate="many_to_one",
        indicator=True,
    )

    details["next_lap_visible"] = details["_merge"].eq("both")
    details["race_id"] = cutoff.race_id

    fitted_missing_details.append(
        details[
            [
                "race_id",
                "driver_code",
                "lap_number",
                "lap_after_stop",
                "next_lap_visible",
                "fitted_compound",
            ]
        ]
    )

display(pd.concat(fitted_missing_details, ignore_index=True))

,race_id,driver_code,lap_number,lap_after_stop,next_lap_visible,fitted_compound
0,2021_16,RIC,21,22,False,NaN
1,2026_03,BEA,16,17,False,NaN
2,2026_03,NOR,16,17,False,NaN


La cause est identifiée : dans les trois cas, le tour suivant l’arrêt n’est pas présent dans l’historique visible.
- RIC : arrêt au tour $21$, tour $22$ non visible ;
- BEA et NOR : arrêt au tour $16$, tour $17$ non visible.
Nous connaissons donc l’existence de l’arrêt, mais pas encore le composé indiqué au tour suivant. Ce n’est pas un composé manquant sur une ligne visible : c’est la ligne nécessaire qui n’est pas disponible.

>Pour 2021_16 et 2026_03, des arrêts officiels sont visibles, mais aucun tour suivant ces arrêts n’est disponible à la coupure. Les proportions des composés montés sont donc indéfinies. Nous conservons `field_stops_seen=0` et les quatre proportions à `NA`, sans consulter les tours cachés.
>
>Au moment de préparer les modèles, nous traiterons ces `NA` tout en conservant le compteur qui en indique le contexte

**Parmi les autres voitures ayant le même composé de départ, quelle proportion a déjà effectué un arrêt ? Cela décrira un groupe de stratégies potentiellement plus comparable que l’ensemble du peloton.**




Construisons `same_start_share_stopped` : la proportion des autres voitures ayant le même composé de départ qui ont déjà au moins un arrêt observé.

Pour une voiture $d$, notons $G_d$ ce groupe, voiture $d$ comprise, et $B_j=\mathbf 1_{\{n_{\mathrm{stops},j}>0\}}$. Alors :
$$
\texttt{same\_start\_share\_stopped}_d
=
\frac{\sum_{j\in G_d}B_j-B_d}{|G_d|-1}.
$$
Pourquoi retirer la voiture elle-même ? Nous voulons décrire ce que font ses comparables. Son propre historique est déjà représenté par `n_stops`.

Nous conservons aussi :
$$
\texttt{same\_start\_other\_cars}_d=|G_d|-1,
$$
pour savoir sur combien d’autres voitures la proportion repose.

Commençons sur notre exemple :

In [97]:
# Conserver toutes les voitures retenues, sans filtrer sur scored
same_start_example = train_features.loc[
    train_features["race_id"].eq("2018_01"),
    ["driver_code", "start_compound", "n_stops"],
].copy()

same_start_example["has_stopped"] = (
    same_start_example["n_stops"].gt(0).astype(int)
)

# Exclure les composés de départ inconnus du regroupement
groups = same_start_example.groupby("start_compound", dropna=True)

group_size = groups["has_stopped"].transform("size")
group_stops = groups["has_stopped"].transform("sum")

# Retirer chaque voiture de son propre groupe de comparaison
other_cars = group_size - 1
other_stopped = group_stops - same_start_example["has_stopped"]

same_start_example["same_start_other_cars"] = other_cars

# Une proportion nécessite au moins une autre voiture
same_start_example["same_start_share_stopped"] = (
    other_stopped / other_cars.where(other_cars.gt(0))
)

display(
    same_start_example[
        [
            "driver_code",
            "start_compound",
            "n_stops",
            "same_start_other_cars",
            "same_start_share_stopped",
        ]
    ]
)

,driver_code,start_compound,n_stops,same_start_other_cars,same_start_share_stopped
0,ALO,SOFT,1,9.0,1.0
1,BOT,SOFT,1,9.0,1.0
2,HAM,SOFT,1,9.0,1.0
3,HAR,<NA>,2,NaN,NaN
4,HUL,SOFT,1,9.0,1.0
5,LEC,MEDIUM,2,3.0,1.0
6,OCO,SOFT,1,9.0,1.0
7,PER,SOFT,1,9.0,1.0
8,RAI,SOFT,1,9.0,1.0
9,RIC,MEDIUM,1,3.0,1.0


Dans cet exemple, toutes les voitures retenues ont déjà un arrêt. Nous attendons donc une proportion de $1$ pour les groupes connus comportant plusieurs voitures.

Deux cas particuliers


Construisons `same_start_share_stopped` : la proportion des autres voitures ayant le même composé de départ qui ont déjà un arrêt observé.

Pour le pilote $d$, notons :
$$
G_d=\{j\neq d:\texttt{start\_compound}_j
=\texttt{start\_compound}_d\}.
$$
Alors :
$$
\texttt{same\_start\_share\_stopped}_d
=
\frac{\sum_{j\in G_d}\mathbf 1\{n_{\mathrm{stops},j}>0\}}
{|G_d|}.
$$
Intuition : si une voiture a quatre autres voitures parties avec le même composé, et que trois ont déjà un arrêt observé, sa valeur est $3/4=0{,}75$.

Nous excluons la voiture elle-même pour décrire les autres. Nous conservons aussi `same_start_other_cars`, le nombre de voitures sur lequel repose la proportion.

In [98]:
# Examiner les voitures retenues de notre course d'exemple
same_start_example = train_features.loc[
    train_features["race_id"].eq("2018_01"),
    ["driver_code", "start_compound", "n_stops"],
].copy()

same_start_example["has_stopped"] = (
    same_start_example["n_stops"].gt(0).astype(int)
)

# Former les groupes uniquement lorsque le composé de départ est connu
groups = same_start_example.groupby("start_compound")

group_size = groups["has_stopped"].transform("size")
group_stops = groups["has_stopped"].transform("sum")

# Retirer la voiture elle-même du groupe et du nombre de voitures arrêtées
other_cars = group_size - 1
other_stopped = group_stops - same_start_example["has_stopped"]

same_start_example["same_start_other_cars"] = other_cars

same_start_example["same_start_share_stopped"] = (
    other_stopped / other_cars.where(other_cars.gt(0))
)

display(
    same_start_example[
        [
            "driver_code",
            "start_compound",
            "n_stops",
            "same_start_other_cars",
            "same_start_share_stopped",
        ]
    ]
)

,driver_code,start_compound,n_stops,same_start_other_cars,same_start_share_stopped
0,ALO,SOFT,1,9.0,1.0
1,BOT,SOFT,1,9.0,1.0
2,HAM,SOFT,1,9.0,1.0
3,HAR,<NA>,2,NaN,NaN
4,HUL,SOFT,1,9.0,1.0
5,LEC,MEDIUM,2,3.0,1.0
6,OCO,SOFT,1,9.0,1.0
7,PER,SOFT,1,9.0,1.0
8,RAI,SOFT,1,9.0,1.0
9,RIC,MEDIUM,1,3.0,1.0


Dans cet exemple, toutes les voitures retenues ont déjà un arrêt observé : nous attendons donc une proportion de $1$ pour les groupes comparables.

Deux cas restent distincts :
- Composé de départ inconnu : groupe inconnu, donc les deux variables restent `NA`.
- Aucune autre voiture du même groupe : effectif égal à $0$ et proportion `NA`, car nous ne pouvons pas calculer une proportion sur un groupe vide.
Comme pour le peloton, nous utilisons toutes les voitures retenues, sans filtrer sur leur résultat final.

Regroupons ce calcul dans une fonction :

In [99]:
def build_same_start_features(snapshot):
    """Compare chaque voiture aux autres ayant le même composé de départ."""

    if snapshot.empty or snapshot["race_id"].nunique() != 1:
        raise ValueError("Une seule course non vide est attendue.")

    if snapshot["driver_code"].duplicated().any():
        raise ValueError("Une seule ligne par pilote est attendue.")

    if snapshot["n_stops"].isna().any():
        raise ValueError("Le nombre d'arrêts doit être renseigné.")

    data = snapshot.set_index("driver_code")[
        ["start_compound", "n_stops"]
    ].copy()

    data["has_stopped"] = data["n_stops"].gt(0).astype(int)

    # Exclure les composés inconnus de la constitution des groupes
    groups = data.groupby("start_compound", dropna=True)

    group_size = groups["has_stopped"].transform("size")
    group_stops = groups["has_stopped"].transform("sum")

    # Exclure la voiture elle-même
    other_cars = group_size - 1
    other_stopped = group_stops - data["has_stopped"]

    features = pd.DataFrame(index=data.index)
    features["same_start_other_cars"] = other_cars
    features["same_start_share_stopped"] = (
        other_stopped / other_cars.where(other_cars.gt(0))
    )

    return features

In [100]:
same_start_columns = [
    "same_start_other_cars",
    "same_start_share_stopped",
]

same_start_check = build_same_start_features(
    train_features.loc[train_features["race_id"].eq("2018_01")]
)

expected_same_start = (
    same_start_example
    .set_index("driver_code")[same_start_columns]
    .sort_index()
)

pd.testing.assert_frame_equal(
    same_start_check.sort_index(),
    expected_same_start,
)

print("La fonction reproduit les deux variables du groupe de départ.")

La fonction reproduit les deux variables du groupe de départ.


Appliquons-la aux $76$ courses d’entraînement, en utilisant toutes les voitures retenues, sans filtrer sur scored.

In [101]:
same_start_parts = []

for race_id, race_snapshot in train_features.groupby("race_id"):
    features = build_same_start_features(race_snapshot).reset_index()
    features["race_id"] = race_id
    same_start_parts.append(features)

same_start_features_train = pd.concat(
    same_start_parts,
    ignore_index=True,
)

# Associer les deux variables à chaque voiture
previous_row_count = len(train_features)

train_features = (
    train_features
    .drop(columns=same_start_columns, errors="ignore")
    .merge(
        same_start_features_train,
        on=["race_id", "driver_code"],
        how="left",
        validate="one_to_one",
        indicator="_same_start_merge",
    )
)

# Vérifier que chaque voiture a trouvé sa ligne
assert len(train_features) == previous_row_count
assert train_features["_same_start_merge"].eq("both").all()

train_features = train_features.drop(columns="_same_start_merge")

# Distinguer les causes possibles d'une proportion manquante
scored_groups = train_features.loc[train_features["scored"]]

unknown_start = scored_groups["start_compound"].isna()

no_other_car = (
    ~unknown_start
    & scored_groups["same_start_other_cars"].eq(0)
)

comparable_group = (
    ~unknown_start
    & scored_groups["same_start_other_cars"].gt(0)
)

group_coverage = pd.DataFrame({
    "observations": [
        int(unknown_start.sum()),
        int(no_other_car.sum()),
        int(comparable_group.sum()),
    ],
}, index=[
    "UNKNOWN_START",
    "NO_OTHER_CAR",
    "COMPARABLE_GROUP",
])

display(group_coverage)

,observations
UNKNOWN_START,25
NO_OTHER_CAR,26
COMPARABLE_GROUP,1245


>La comparaison avec les autres voitures de même composé de départ est disponible pour $1 245$ observations évaluables. Elle est impossible pour $25$ observations dont le composé initial est inconnu et pour $26$ observations sans autre voiture comparable. Ces absences sont conservées explicitement.

Passons maintenant aux neutralisations, en commençant par les indicateurs actuels `safety_car` et `virtual_sc`, avant de construire leur résumé oracle.

**Intuition :** nous voulons d’abord savoir quelles situations sont représentées et si des valeurs sont inconnues. Nous ne remplacerons pas automatiquement une absence d’information par `False`.

In [103]:
# Examiner les combinaisons observées au dernier tour visible
neutral_state_summary = (
    train_features.loc[train_features["scored"]]
    .groupby(["safety_car", "virtual_sc"], dropna=False)
    .agg(
        observations=("driver_code", "size"),
        courses=("race_id", "nunique"),
    )
    .reset_index()
)

display(neutral_state_summary)

,safety_car,virtual_sc,observations,courses
0,False,False,1212,72
1,False,True,38,5
2,True,False,33,2
3,True,True,13,1


Les deux indicateurs sont renseignés pour les $1 296$ observations. Nous distinguons :

| Indicateurs au dernier tour visible | Observations |
|---|---:|
| Ni safety car ni VSC signalées | $1 212$ |
| VSC seulement | $38$ |
| Safety car seulement | $33$ |
| Les deux signalées | $13$ |

Les $13$ lignes avec deux `True` ne prouvent pas que les deux dispositifs étaient actifs simultanément. Un tour peut couvrir une transition entre statuts ; la convention de construction des colonnes pourrait aussi intervenir. Nous ne corrigeons donc pas ces lignes sans information supplémentaire.

Pour décrire la présence d’au moins un des deux signaux, construisons `neutral_now` :
$$
N_i=\mathbf 1_{\{S_i=\mathrm{True}\ \text{ou}\ V_i=\mathrm{True}\}}.
$$

**Intuition :** nous comptons les lignes où au moins une neutralisation est signalée, sans compter deux fois celles qui portent les deux indicateurs.

Comme `rain_now`, cette variable décrit le dernier tour visible de chaque voiture, pas nécessairement le statut exact à l’instant $T$. Les colonnes d’origine restent disponibles : nous déciderons ensuite si le modèle utilise les deux indicateurs séparés ou leur résumé.

In [104]:
# Combiner les indicateurs en conservant la gestion des valeurs inconnues
train_features["neutral_now"] = (
    train_features["safety_car"].astype("boolean")
    | train_features["virtual_sc"].astype("boolean")
)

# Vérifier les effectifs dans la population évaluable
neutral_now_summary = (
    train_features.loc[train_features["scored"], "neutral_now"]
    .value_counts(dropna=False)
    .rename_axis("neutral_now")
    .to_frame("observations")
)

display(neutral_now_summary)

,observations
neutral_now,
False,1212
True,84


Examinons maintenant la distribution du prochain composé dans ces deux groupes :
$$
\widehat P(Y=k\mid N=n)
=
\frac{\#\{i:Y_i=k,\ N_i=n\}}
{\#\{i:N_i=n\}},
$$
où $N=\texttt{neutral\_now}$.

**Intuition :** nous cherchons à voir si les cibles se répartissent différemment lorsqu’une neutralisation est signalée.

Ce tableau ne mesurera ni la probabilité de s’arrêter immédiatement, ni l’effet causal d’une neutralisation : notre cible concerne le composé du prochain arrêt, quel que soit son délai, ou `NONE`.

In [105]:
# Utiliser uniquement les observations d'entraînement évaluables
neutral_analysis = train_features.loc[
    train_features["scored"]
]

# Compter les cibles dans chaque situation
neutral_target_counts = pd.crosstab(
    neutral_analysis["neutral_now"],
    neutral_analysis["next_compound"],
).reindex(
    index=[False, True],
    columns=CLASSES,
    fill_value=0,
)

# Calculer les proportions conditionnelles
neutral_target_pct = (
    neutral_target_counts.div(
        neutral_target_counts.sum(axis=1).replace(0, np.nan),
        axis=0,
    ) * 100
)

# Mesurer le nombre de courses derrière les observations
neutral_support = (
    neutral_analysis.groupby("neutral_now")
    .agg(
        observations=("driver_code", "size"),
        courses=("race_id", "nunique"),
    )
)

display(neutral_support)
display(neutral_target_counts)
display(neutral_target_pct.round(2))

,observations,courses
neutral_now,,
False,1212,72
True,84,7


next_compound,SOFT,MEDIUM,HARD,WET_OR_INTER,NONE
neutral_now,,,,,
False,174,281,457,35,265
True,16,24,12,0,32


next_compound,SOFT,MEDIUM,HARD,WET_OR_INTER,NONE
neutral_now,,,,,
False,14.36,23.18,37.71,2.89,21.86
True,19.05,28.57,14.29,0.00,38.10


>La distribution des cibles diffère selon la présence d’un signal `SC` ou `VSC` au dernier tour visible. La fréquence de `NONE` est notamment de $38,10 \%$ avec signal, contre $21,86 \%$ sans signal. Cette association descriptive motive l’examen de `neutral_now` comme variable candidate, sans établir d’effet causal.

Pour vérifier si les deux groupes présentent déjà des historiques d’arrêts différents :

In [106]:
# Comparer les arrêts passés selon le signal de neutralisation
neutral_stop_context = (
    neutral_analysis
    .assign(has_stopped=neutral_analysis["n_stops"].gt(0))
    .groupby("neutral_now")
    .agg(
        observations=("driver_code", "size"),
        courses=("race_id", "nunique"),
        share_already_stopped=("has_stopped", "mean"),
        mean_stops=("n_stops", "mean"),
        median_laps_since_stop=("laps_since_stop", "median"),
    )
)

display(neutral_stop_context.round(3))

,observations,courses,share_already_stopped,mean_stops,median_laps_since_stop
neutral_now,,,,,
False,1212,72,0.669,0.802,13.0
True,84,7,0.881,1.012,4.0


<details>
<summary>Les deux groupes ne sont effectivement pas au même stade de leur historique d’arrêts. </summary>

| Indicateur | Sans signal SC/VSC | Avec signal SC/VSC |
|---|---:|---:|
| Part ayant déjà un arrêt observé | $66,9 \%$ | $88,1 \%$ |
| Nombre moyen d’arrêts | $0,802$ | $1,012$ |
| Médiane de `laps_since_stop` | $13$ tours | $4$ tours |

**Intuition :** les voitures avec un signal de neutralisation ont plus souvent déjà effectué un arrêt. Leur valeur de `laps_since_stop` est aussi généralement plus faible. Ce contexte peut contribuer à la fréquence plus élevée de `NONE` observée précédemment : $38,10\ %$ contre $21,86 \%$.

Deux précautions :
- Cela ne prouve pas que leurs arrêts ont eu lieu pendant la neutralisation.
- Pour les voitures sans arrêt observé, `laps_since_stop` compte les tours depuis le départ. La médiane mélange donc deux situations.

Nous conservons `neutral_now` comme variable candidate. La prochaine étape est le résumé des neutralisations futures, avec la même logique que pour la pluie :
$$
N_h=S_h\lor V_h,
$$
puis trois variables :
- `future_neutral_laps` : nombre de tours renseignés avec SC ou VSC ;
- `future_neutral_known_laps` : nombre de tours dont cet état combiné est connu ;
- `future_neutral_first` : premier décalage avec neutralisation, ou $15$ si l’horizon complet n’en contient aucune.
Un tour portant les deux signaux sera compté une seule fois.


</details>

Construisons d’abord la séquence des $14$ prochains tours, avant de la résumer.

Pour chaque décalage $h$, nous combinons les deux informations oracle :
$$
N_h=S_h\lor V_h.
$$

<details>
<summary> Intuition :</summary>
Le tour est considéré comme neutralisé si au moins un des deux signaux est vrai. Si les deux sont vrais, il ne compte toujours que pour un tour.

Reprenons notre exemple 2018_01 à sa coupure enregistrée :

Les valeurs inconnues sont traitées logiquement :

| Safety car | VSC | État combiné |
|---|---|---|
| `True` | `NA` | `True` : une neutralisation est certaine |
| `False` | `NA` | `NA` : impossible de conclure |
| `False` | `False` | `False` |

Nous ne remplaçons donc pas les valeurs manquantes par `False`. Examinons la séquence obtenue avant de calculer les trois features.
</details>

In [107]:
# Retrouver la coupure de notre exemple
neutral_race_id = "2018_01"

neutral_cutoff = train_cutoffs.loc[
    train_cutoffs["race_id"].eq(neutral_race_id)
].iloc[0]

neutral_leader_lap = int(neutral_cutoff["cutoff_lap"])

# Ne lire que les lignes visibles à la coupure
neutral_visible = train_data.loc[
    train_data["race_id"].eq(neutral_race_id)
    & train_data["race_time_s"].notna()
    & train_data["race_time_s"].le(neutral_cutoff["cutoff_time_s"])
].copy()

source_laps = pd.Index(
    range(neutral_leader_lap - 13, neutral_leader_lap + 1),
    name="source_lap",
)

oracle_columns = [
    "safety_car_Lplus14",
    "virtual_sc_Lplus14",
]

oracle_source = neutral_visible.loc[
    neutral_visible["lap_number"].isin(source_laps)
]

# Vérifier la concordance des informations entre pilotes au même tour
known_values = (
    oracle_source.groupby("lap_number")[oracle_columns]
    .nunique(dropna=True)
)

assert known_values.le(1).all().all(), (
    "Des valeurs oracle contradictoires existent pour un même tour."
)

# Extraire les deux séquences en conservant les valeurs inconnues
neutral_oracle = (
    oracle_source.groupby("lap_number")[oracle_columns]
    .first()
    .reindex(source_laps)
    .astype("boolean")
)

neutral_timeline = pd.DataFrame({
    "offset": range(1, 15),
    "source_lap": source_laps.to_numpy(),
    "future_lap": source_laps.to_numpy() + 14,
    "safety_car": neutral_oracle["safety_car_Lplus14"].array,
    "virtual_sc": neutral_oracle["virtual_sc_Lplus14"].array,
})

neutral_timeline["neutral"] = (
    neutral_timeline["safety_car"]
    | neutral_timeline["virtual_sc"]
)

display(neutral_timeline)

,offset,source_lap,future_lap,safety_car,virtual_sc,neutral
0,1,15,29,True,False,True
1,2,16,30,True,False,True
2,3,17,31,True,False,True
3,4,18,32,False,False,False
4,5,19,33,False,False,False
5,6,20,34,False,False,False
6,7,21,35,False,False,False
7,8,22,36,False,False,False
8,9,23,37,False,False,False
9,10,24,38,False,False,False


L’oracle signale une safety car aux tours $29$, $30$ et $31$, puis aucun signal SC ou VSC aux tours $32$ à $42$. Les $14$ états combinés sont connus.

<details>
<summary> Nos trois features doivent donc être :</summary>

| Feature | Valeur | Interprétation |
|---|---:|---|
| `future_neutral_laps` | $3$ | Trois tours avec un signal SC ou VSC |
| `future_neutral_known_laps` | $14$ | Horizon entièrement renseigné |
| `future_neutral_first` | $1$ | Premier signal dès le prochain tour |

Le décalage $1$ ne signifie pas que la neutralisation commence à ce moment-là : elle peut déjà être en cours à la coupure. De même, un tour signalé ne signifie pas nécessairement que sa totalité se déroule sous neutralisation.

</details>

Calculons ces valeurs :


In [108]:
# Extraire la séquence combinée, indexée par décalage
neutral = (
    neutral_timeline
    .set_index("offset")["neutral"]
    .astype("boolean")
)

# Compter les tours connus et ceux avec un signal de neutralisation
future_neutral_known_laps = int(neutral.notna().sum())
future_neutral_laps = neutral.sum(min_count=1)

# Identifier la première occurrence dont le décalage est établi
neutral_offsets = neutral.index[neutral.fillna(False)]

if len(neutral_offsets) > 0:
    first_offset = int(neutral_offsets.min())
    preceding_values = neutral.loc[neutral.index < first_offset]

    future_neutral_first = (
        first_offset
        if preceding_values.notna().all()
        else np.nan
    )

elif neutral.notna().all():
    # Aucun signal dans un horizon entièrement connu
    future_neutral_first = 15

else:
    # Des inconnues empêchent de conclure
    future_neutral_first = np.nan

neutral_features = pd.DataFrame([{
    "future_neutral_laps": future_neutral_laps,
    "future_neutral_known_laps": future_neutral_known_laps,
    "future_neutral_first": future_neutral_first,
}])

display(neutral_features)

,future_neutral_laps,future_neutral_known_laps,future_neutral_first
0,3,14,1


Regroupons maintenant la reconstruction de la séquence et son résumé dans une fonction.

In [109]:
def build_neutral_features(visible, leader_lap):
    """Résume les signaux SC/VSC sur les 14 prochains tours du leader."""

    if visible.empty or visible["race_id"].nunique() != 1:
        raise ValueError("Une seule course non vide est attendue.")

    source_laps = pd.Index(
        range(leader_lap - 13, leader_lap + 1),
        name="source_lap",
    )

    oracle_columns = [
        "safety_car_Lplus14",
        "virtual_sc_Lplus14",
    ]

    oracle_source = visible.loc[
        visible["lap_number"].isin(source_laps)
    ]

    # Vérifier la concordance des valeurs connues entre pilotes
    known_values = (
        oracle_source.groupby("lap_number")[oracle_columns]
        .nunique(dropna=True)
    )

    if known_values.gt(1).any().any():
        raise ValueError("Valeurs oracle contradictoires pour un même tour.")

    # Conserver les inconnues lors de la reconstruction de l'horizon
    oracle = (
        oracle_source.groupby("lap_number")[oracle_columns]
        .first()
        .reindex(source_laps)
        .astype("boolean")
    )

    oracle.index = pd.RangeIndex(1, 15, name="offset")

    # Compter chaque tour une seule fois, même si les deux signaux sont vrais
    neutral = (
        oracle["safety_car_Lplus14"]
        | oracle["virtual_sc_Lplus14"]
    )

    known_laps = int(neutral.notna().sum())
    neutral_laps = neutral.sum(min_count=1)

    # Déterminer le premier décalage avec neutralisation
    event_offsets = neutral.index[neutral.fillna(False)]

    if len(event_offsets) > 0:
        first_offset = int(event_offsets.min())
        preceding_values = neutral.loc[neutral.index < first_offset]

        first_neutral = (
            first_offset
            if preceding_values.notna().all()
            else np.nan
        )

    elif neutral.notna().all():
        first_neutral = 15

    else:
        first_neutral = np.nan

    return pd.DataFrame([{
        "future_neutral_laps": neutral_laps,
        "future_neutral_known_laps": known_laps,
        "future_neutral_first": first_neutral,
    }])

In [110]:
neutral_features_check = build_neutral_features(
    neutral_visible,
    neutral_leader_lap,
)

pd.testing.assert_frame_equal(
    neutral_features_check,
    neutral_features,
)

display(neutral_features_check)
print("La fonction reproduit les trois variables de neutralisation future.")

,future_neutral_laps,future_neutral_known_laps,future_neutral_first
0,3,14,1


La fonction reproduit les trois variables de neutralisation future.


Appliquons-la aux $76$ courses d’entraînement, en conservant leurs coupures déjà fixées.

In [111]:
neutral_parts = []

for cutoff in train_cutoffs.itertuples(index=False):

    # Utiliser uniquement les tours terminés avant ou à la coupure
    visible = train_data.loc[
        train_data["race_id"].eq(cutoff.race_id)
        & train_data["race_time_s"].notna()
        & train_data["race_time_s"].le(cutoff.cutoff_time_s)
    ].copy()

    neutral_part = build_neutral_features(
        visible,
        int(cutoff.cutoff_lap),
    )

    neutral_part["race_id"] = cutoff.race_id
    neutral_parts.append(neutral_part)

neutral_features_train = pd.concat(neutral_parts, ignore_index=True)

neutral_columns = [
    "future_neutral_laps",
    "future_neutral_known_laps",
    "future_neutral_first",
]

# Associer le résumé de chaque course à toutes ses voitures retenues
previous_row_count = len(train_features)

train_features = (
    train_features
    .drop(columns=neutral_columns, errors="ignore")
    .merge(
        neutral_features_train,
        on="race_id",
        how="left",
        validate="many_to_one",
        indicator="_neutral_merge",
    )
)

# Vérifier la jointure sans interdire les inconnues légitimes
assert len(neutral_features_train) == 76
assert len(train_features) == previous_row_count
assert train_features["_neutral_merge"].eq("both").all()

train_features = train_features.drop(columns="_neutral_merge")

# Examiner la couverture une fois par course
display(
    neutral_features_train["future_neutral_known_laps"]
    .value_counts()
    .sort_index()
    .rename_axis("known_future_laps")
    .to_frame("courses")
)

# Décrire les séquences résumées, y compris les éventuelles inconnues
display(
    neutral_features_train
    .groupby(
        ["future_neutral_laps", "future_neutral_first"],
        dropna=False,
    )
    .size()
    .rename("courses")
    .reset_index()
)

,courses
known_future_laps,
14,76


,future_neutral_laps,future_neutral_first,courses
0,0,15,54
1,1,1,4
2,1,14,1
3,2,13,1
4,3,1,2
5,3,2,1
6,3,5,1
7,3,8,1
8,3,12,1
9,4,3,1


Les $76$ courses disposent d’un horizon entièrement renseigné. Parmi elles, $22$ présentent au moins un signal SC ou VSC dans les $14$ prochains tours.

Contrairement à notre résultat pour la pluie, `future_neutral_first` varie ici entre plusieurs valeurs même lorsque le nombre de tours concernés est identique. Cela justifie de conserver les deux résumés comme candidats.

`future_neutral_known_laps`, en revanche, vaut toujours $14$ dans cet entraînement : elle sert actuellement au contrôle de couverture.

Il reste à compléter ce bloc avec le drapeau rouge. Commençons par examiner son indicateur au dernier tour visible :

In [112]:
# Examiner le drapeau rouge dans les observations évaluables
red_flag_summary = (
    train_features.loc[train_features["scored"]]
    .groupby("red_flag", dropna=False)
    .agg(
        observations=("driver_code", "size"),
        courses=("race_id", "nunique"),
    )
)

display(red_flag_summary)

,observations,courses
red_flag,,
False,1282,76
True,14,1


Le drapeau rouge est signalé au dernier tour visible de $14$ voitures, toutes dans une seule course. Les $1 282$ autres observations valent False, sans valeur manquante.

La course concernée apparaît aussi dans le groupe `False`, puisque ce groupe contient les $76$ courses. Les pilotes de cette course n’ont donc pas tous le même indicateur sur leur dernière ligne visible. Ce n’est pas forcément contradictoire : leurs derniers tours visibles peuvent être différents.

**Intuition :** avant de nommer cette information `red_flag_now`, nous devons garder sa portée exacte en tête : elle décrit le dernier tour visible, pas nécessairement le statut précis à l’instant de coupure.

Identifions cette course et examinons les lignes concernées :

In [113]:
# Identifier la course avec un drapeau rouge sur une dernière ligne visible
red_flag_races = train_features.loc[
    train_features["scored"]
    & train_features["red_flag"].eq(True).fillna(False),
    "race_id",
].unique()

# Comparer les dernières lignes de toutes ses voitures évaluables
display(
    train_features.loc[
        train_features["scored"]
        & train_features["race_id"].isin(red_flag_races),
        [
            "race_id",
            "driver_code",
            "lap_number",
            "race_time_s",
            "cutoff_time_s",
            "red_flag",
        ],
    ].sort_values(["race_id", "lap_number", "driver_code"])
)

,race_id,driver_code,lap_number,race_time_s,cutoff_time_s,red_flag
550,2021_21,ALO,16,4232.869,4353.251,True
551,2021_21,BOT,16,4208.289,4353.251,True
552,2021_21,GAS,16,4212.191,4353.251,True
553,2021_21,GIO,16,4216.269,4353.251,True
554,2021_21,HAM,16,4197.214,4353.251,True
555,2021_21,LAT,16,4238.320,4353.251,True
556,2021_21,LEC,16,4214.823,4353.251,True
557,2021_21,NOR,16,4241.948,4353.251,True
558,2021_21,OCO,16,4190.284,4353.251,True
559,2021_21,RAI,16,4228.042,4353.251,True


La différence correspond ici à des derniers tours visibles différents :
- les $14$ voitures au tour $16$ ont `red_flag=True` ;
- VER, au tour $17$, a `red_flag=False` et termine exactement à la coupure :
$$
  T=4\,353{,}251\ \text{s}.
  $$
Il ne faut donc pas lire ce tableau comme « $14$ voitures sous drapeau rouge et VER sous drapeau vert au même instant ». Les indicateurs décrivent des tours différents, et `red_flag=False` ne suffit d’ailleurs pas à établir un drapeau vert.

Nous pouvons conserver l’information individuelle, avec un nom plus précis :

In [114]:
# Conserver le signal du dernier tour visible, sans prétendre décrire l'instant T
train_features["red_flag_last_lap"] = (
    train_features["red_flag"].astype("boolean")
)

<details>
<summary>Passons maintenant au drapeau rouge futur. </summary>

Comme pour la pluie et les neutralisations, nous reconstruirons les $14$ prochains tours à partir des colonnes oracle des lignes visibles.

Nous créerons deux variables :

| Variable | Définition |
|---|---|
| `future_red_flag` | Au moins un drapeau rouge signalé dans l’horizon |
| `future_red_known_laps` | Nombre de tours de l’horizon renseignés |

En notant $R_h$ le signal oracle au décalage $h$ :
$$
\texttt{future\_red\_flag}=
\begin{cases}
1 & \text{si au moins un }R_h=\mathrm{True},\\
0 & \text{si les 14 valeurs sont connues et toutes fausses},\\
\mathrm{NA} & \text{sinon}.
\end{cases}
$$
**Intuition :** un seul True suffit pour confirmer la présence d’un signal. En revanche, pour affirmer son absence sur tout l’horizon, nous devons connaître les $14$ valeurs.


**Reconstruisons d’abord la séquence oracle pour 2021_21, la course que nous venons d’examiner.**

La coupure est au tour $17$ du leader : les lignes sources des tours $4$ à $17$ décrivent donc les tours futurs $18$ à $31$.

Nous cherchons des signaux dans les tours futurs, pas celui déjà observé au tour $16$. La présence passée d’un drapeau rouge ne permet pas de déduire s’il y en aura un autre dans cet horizon. Regardons donc la séquence avant de calculer les deux variables.
</details>
 

In [115]:
# Retrouver la coupure de la course étudiée
red_race_id = "2021_21"

red_cutoff = train_cutoffs.loc[
    train_cutoffs["race_id"].eq(red_race_id)
].iloc[0]

red_leader_lap = int(red_cutoff["cutoff_lap"])

# Utiliser uniquement les lignes visibles à la coupure
red_visible = train_data.loc[
    train_data["race_id"].eq(red_race_id)
    & train_data["race_time_s"].notna()
    & train_data["race_time_s"].le(red_cutoff["cutoff_time_s"])
].copy()

red_source_laps = pd.Index(
    range(red_leader_lap - 13, red_leader_lap + 1),
    name="source_lap",
)

red_source = red_visible.loc[
    red_visible["lap_number"].isin(red_source_laps)
]

# Vérifier la concordance des valeurs connues entre pilotes
red_known_values = (
    red_source.groupby("lap_number")["red_flag_Lplus14"]
    .nunique(dropna=True)
)

assert red_known_values.le(1).all(), (
    "Des valeurs oracle contradictoires existent pour un même tour."
)

# Reconstruire l'horizon en conservant les valeurs inconnues
red_oracle = (
    red_source.groupby("lap_number")["red_flag_Lplus14"]
    .first()
    .reindex(red_source_laps)
    .astype("boolean")
)

red_timeline = pd.DataFrame({
    "offset": range(1, 15),
    "source_lap": red_source_laps.to_numpy(),
    "future_lap": red_source_laps.to_numpy() + 14,
    "red_flag": red_oracle.array,
})

display(red_timeline)

,offset,source_lap,future_lap,red_flag
0,1,4,18,False
1,2,5,19,False
2,3,6,20,False
3,4,7,21,False
4,5,8,22,False
5,6,9,23,False
6,7,10,24,False
7,8,11,25,False
8,9,12,26,False
9,10,13,27,False


Les $14$ valeurs sont connues et toutes valent `False` : aucun drapeau rouge n’est signalé par l’oracle aux tours $18$ à $31$.

In [116]:
# Extraire les signaux futurs en conservant les valeurs inconnues
red = red_timeline["red_flag"].astype("boolean")

future_red_known_laps = int(red.notna().sum())

if red.fillna(False).any():
    # Au moins un signal connu suffit à confirmer sa présence
    future_red_flag = 1

elif red.notna().all():
    # L'absence est établie sur tout l'horizon renseigné
    future_red_flag = 0

else:
    # Aucun signal vrai, mais certaines valeurs sont inconnues
    future_red_flag = np.nan

red_features = pd.DataFrame([{
    "future_red_flag": future_red_flag,
    "future_red_known_laps": future_red_known_laps,
}])

display(red_features)

,future_red_flag,future_red_known_laps
0,0,14


Regroupons maintenant la reconstruction et le résumé dans une fonction :

In [117]:
def build_red_flag_features(visible, leader_lap):
    """Résume le drapeau rouge oracle sur les 14 prochains tours."""

    if visible.empty or visible["race_id"].nunique() != 1:
        raise ValueError("Une seule course non vide est attendue.")

    source_laps = pd.Index(
        range(leader_lap - 13, leader_lap + 1),
        name="source_lap",
    )

    source = visible.loc[
        visible["lap_number"].isin(source_laps)
    ]

    # Vérifier la concordance des valeurs connues entre pilotes
    known_values = (
        source.groupby("lap_number")["red_flag_Lplus14"]
        .nunique(dropna=True)
    )

    if known_values.gt(1).any():
        raise ValueError("Valeurs oracle contradictoires pour un même tour.")

    # Reconstruire l'horizon sans remplacer les inconnues par False
    red = (
        source.groupby("lap_number")["red_flag_Lplus14"]
        .first()
        .reindex(source_laps)
        .astype("boolean")
    )

    known_laps = int(red.notna().sum())

    if red.fillna(False).any():
        red_flag = 1
    elif red.notna().all():
        red_flag = 0
    else:
        red_flag = np.nan

    return pd.DataFrame([{
        "future_red_flag": red_flag,
        "future_red_known_laps": known_laps,
    }])

In [118]:
red_features_check = build_red_flag_features(
    red_visible,
    red_leader_lap,
)

pd.testing.assert_frame_equal(
    red_features_check,
    red_features,
)

print("La fonction reproduit les deux variables de drapeau rouge futur.")

La fonction reproduit les deux variables de drapeau rouge futur.


Appliquons-la aux $76$ courses d’entraînement, toujours aux coupures déjà choisies.

Nous comptons chaque course une seule fois dans ces diagnostics. Le premier tableau mesure la disponibilité de l’information ; le second indique combien de coupures annoncent au moins un drapeau rouge dans l’horizon.

Une éventuelle valeur ``NA` dans le second tableau signifie « impossible de conclure », et non « absence de drapeau rouge ».

In [119]:
red_parts = []

for cutoff in train_cutoffs.itertuples(index=False):

    # Utiliser uniquement les tours visibles à cette coupure
    visible = train_data.loc[
        train_data["race_id"].eq(cutoff.race_id)
        & train_data["race_time_s"].notna()
        & train_data["race_time_s"].le(cutoff.cutoff_time_s)
    ].copy()

    red_part = build_red_flag_features(
        visible,
        int(cutoff.cutoff_lap),
    )

    red_part["race_id"] = cutoff.race_id
    red_parts.append(red_part)

red_features_train = pd.concat(red_parts, ignore_index=True)

red_columns = [
    "future_red_flag",
    "future_red_known_laps",
]

# Associer une ligne oracle à toutes les voitures de la course
previous_row_count = len(train_features)

train_features = (
    train_features
    .drop(columns=red_columns, errors="ignore")
    .merge(
        red_features_train,
        on="race_id",
        how="left",
        validate="many_to_one",
        indicator="_red_merge",
    )
)

# Vérifier la jointure sans interdire les inconnues légitimes
assert len(red_features_train) == 76
assert len(train_features) == previous_row_count
assert train_features["_red_merge"].eq("both").all()

train_features = train_features.drop(columns="_red_merge")

# Contrôler la couverture de l'horizon
display(
    red_features_train["future_red_known_laps"]
    .value_counts()
    .sort_index()
    .rename_axis("known_future_laps")
    .to_frame("courses")
)

# Compter les courses avec ou sans signal futur
display(
    red_features_train["future_red_flag"]
    .value_counts(dropna=False)
    .sort_index()
    .rename_axis("future_red_flag")
    .to_frame("courses")
)

,courses
known_future_laps,
14,76


,courses
future_red_flag,
0,75
1,1


L’horizon est entièrement renseigné pour les $76$ courses. Une seule annonce au moins un drapeau rouge dans les $14$ prochains tours.
$$
\frac{1}{76}\times100\approx1{,}32\%.
$$
Cela signifie :

- `future_red_flag` $=0$ pour $75$ courses ;
- `future_red_flag` $=1$ pour une course ;
- aucune valeur inconnue à ces coupures.

Cette variable est donc très rare dans notre entraînement. Même si plusieurs voitures partagent sa valeur $1$, elles appartiennent toutes au même contexte de course. Nous ne pourrons pas attribuer leurs particularités au seul drapeau rouge.

Identifions cette course :

In [120]:
# Identifier la coupure dont l'oracle signale un drapeau rouge
red_event_course = (
    red_features_train.loc[
        red_features_train["future_red_flag"].eq(1)
    ]
    .merge(
        train_cutoffs[
            [
                "race_id",
                "cutoff_lap",
                "cutoff_progress",
                "retained_cars",
                "scored_cars",
            ]
        ],
        on="race_id",
        validate="one_to_one",
    )
)

display(red_event_course)

,future_red_flag,future_red_known_laps,race_id,cutoff_lap,cutoff_progress,retained_cars,scored_cars
0,1,14,2023_19,26,0.366197,19,15


La seule course concernée est 2023_19, coupée au tour $26$ du leader, soit $36,62\%$ de la distance prévue.

>Les blocs prévus sont maintenant construits. Avant d’assembler une fonction unique, fixons explicitement notre liste de variables candidates.
>
>Ce n’est pas encore la sélection finale pour chaque modèle. Les constantes et les dépendances exactes seront traitées ensuite. Les compteurs `rain_known_laps` et `rain_recent_known_laps` restent, pour l’instant, des colonnes de contrôle hors de cette liste.

La cible, `scored`, les identifiants et `result_status` sont également exclus des variables explicatives.

In [121]:
feature_blocks = {
    "car_state": [
        "compound",
        "tyre_life_laps",
        "laps_remaining",
        "projected_tyre_age",
    ],
    "stop_history": [
        "n_stops",
        "laps_since_stop",
        "stopped_on_last_lap",
    ],
    "tyre_history": [
        "used_soft",
        "used_medium",
        "used_hard",
        "n_dry_used",
        "used_wet",
        "start_compound",
        "start_compound_missing",
    ],
    "weather_history": [
        "rain_now",
        "rain_share_so_far",
        "rain_share_recent",
    ],
    "current_track_state": [
        "neutral_now",
        "red_flag_last_lap",
    ],
    "field_state": [
        "field_on_soft",
        "field_on_medium",
        "field_on_hard",
        "field_on_wet_or_inter",
        "field_share_stopped",
        "field_mean_stops",
    ],
    "fitted_compounds": [
        "field_stops_seen",
        "field_fitted_soft",
        "field_fitted_medium",
        "field_fitted_hard",
        "field_fitted_wet_or_inter",
    ],
    "same_start_group": [
        "same_start_other_cars",
        "same_start_share_stopped",
    ],
    "oracle": [
        "future_rain_laps",
        "future_rain_known_laps",
        "future_rain_first",
        "future_neutral_laps",
        "future_neutral_known_laps",
        "future_neutral_first",
        "future_red_flag",
        "future_red_known_laps",
    ],
}

candidate_features = [
    column
    for columns in feature_blocks.values()
    for column in columns
]

# Vérifier que les variables sont présentes et sans doublon dans la liste
assert len(candidate_features) == len(set(candidate_features))

missing_columns = [
    column for column in candidate_features
    if column not in train_features.columns
]
assert not missing_columns, f"Colonnes absentes : {missing_columns}"

display(pd.DataFrame({
    "block": list(feature_blocks),
    "feature_count": [len(columns) for columns in feature_blocks.values()],
}))

print("Nombre de variables candidates :", len(candidate_features))

,block,feature_count
0,car_state,4
1,stop_history,3
2,tyre_history,7
3,weather_history,3
4,current_track_state,2
5,field_state,6
6,fitted_compounds,5
7,same_start_group,2
8,oracle,8


Nombre de variables candidates : 40


### Variables
Nous avons donc une liste explicite de ce que nous envisageons de donner aux modèles.

Avant d’assembler la fonction finale, faisons un contrôle global sur les $1 296$ observations évaluables d’entraînement :
- type de chaque variable ;
- valeurs manquantes ;
- nombre de valeurs distinctes, pour repérer les constantes.

>Une variable constante ne distingue aucune voiture dans cet entraînement. Une variable avec des `NA`, en revanche, n’est pas automatiquement inutile : nous avons déjà expliqué plusieurs absences structurelles, notamment pour les composés montés et les groupes de départ.

>Nous ne supprimons rien dans cette cellule. Les décisions de prétraitement seront ensuite appliquées dans les pipelines, en les apprenant uniquement sur les données d’entraînement de chaque pli.


In [122]:
# Extraire uniquement les variables candidates des observations évaluables
X_train_candidates = train_features.loc[
    train_features["scored"],
    candidate_features,
].copy()

# Résumer la disponibilité et la diversité de chaque variable
feature_audit = pd.DataFrame({
    "dtype": X_train_candidates.dtypes.astype(str),
    "missing_count": X_train_candidates.isna().sum(),
    "missing_percentage": 100 * X_train_candidates.isna().mean(),
    "unique_known_values": X_train_candidates.nunique(dropna=True),
})

# Associer chaque variable à son bloc
feature_to_block = {
    feature: block
    for block, features in feature_blocks.items()
    for feature in features
}

feature_audit.insert(
    0,
    "block",
    feature_audit.index.map(feature_to_block),
)

print("Dimensions :", X_train_candidates.shape)
display(feature_audit.round(2))

# Isoler les variables constantes ou entièrement manquantes
display(
    feature_audit.loc[
        feature_audit["unique_known_values"].le(1)
    ]
)

Dimensions : (1296, 40)


,block,dtype,missing_count,missing_percentage,unique_known_values
compound,car_state,str,0,0.00,4
tyre_life_laps,car_state,Int64,0,0.00,42
laps_remaining,car_state,Int64,0,0.00,25
projected_tyre_age,car_state,Int64,0,0.00,56
n_stops,stop_history,int64,0,0.00,5
laps_since_stop,stop_history,int64,0,0.00,37
stopped_on_last_lap,stop_history,int64,0,0.00,2
used_soft,tyre_history,int64,0,0.00,2
used_medium,tyre_history,int64,0,0.00,2
used_hard,tyre_history,int64,0,0.00,2


,block,dtype,missing_count,missing_percentage,unique_known_values
future_rain_known_laps,oracle,int64,0,0.0,1
future_neutral_known_laps,oracle,int64,0,0.0,1
future_red_known_laps,oracle,int64,0,0.0,1


>Les $40$ variables candidates sont présentes. Sept colonnes comportent des valeurs manquantes dont les causes ont été identifiées ; aucune colonne n’est entièrement manquante. Trois compteurs de couverture oracle sont constants. Les constantes, les dépendances entre variables et l’imputation seront traitées dans les prétraitements adaptés aux modèles.

### Assemblage

Assemblons les fonctions déjà vérifiées. Cette fonction ne choisira pas les voitures à prédire : elle recevra leur liste explicitement, ainsi que l’historique visible et le tour du leader.

Cela sépare deux tâches :
- sélectionner les voitures selon notre convention de simulation ou selon `sample_submission.csv` pour le test ;
- calculer leurs features à partir des informations visibles.

In [123]:
def build_all_features(visible, retained_drivers, leader_lap, recent_window=5):
    """Construit les 40 variables candidates pour une course-coupure."""

    if visible.empty or visible["race_id"].nunique() != 1:
        raise ValueError("Une seule course non vide est attendue.")

    if visible.duplicated(["driver_code", "lap_number"]).any():
        raise ValueError("Plusieurs lignes existent pour un même pilote-tour.")

    drivers = pd.Index(
        list(retained_drivers),
        name="driver_code",
    )

    if drivers.empty or drivers.has_duplicates:
        raise ValueError("La liste des pilotes doit être non vide et unique.")

    # Identifier la dernière ligne visible de chaque pilote
    last_rows = (
        visible
        .sort_values(["driver_code", "lap_number"])
        .drop_duplicates("driver_code", keep="last")
        .set_index("driver_code")
    )

    if not drivers.isin(last_rows.index).all():
        raise ValueError("Certains pilotes retenus n'ont aucun tour visible.")

    last_rows = last_rows.loc[drivers]

    # Construire l'état individuel de la voiture
    features = last_rows[
        ["compound", "tyre_life_laps", "laps_remaining"]
    ].copy()

    features["projected_tyre_age"] = (
        features["tyre_life_laps"] + features["laps_remaining"]
    )

    # Ajouter les blocs individuels calculés depuis l'historique visible
    individual_blocks = [
        build_stop_history(visible),
        build_tyre_usage(visible),
        build_start_compound(visible),
        build_rain_history(visible, recent_window=recent_window)[
            ["rain_now", "rain_share_so_far", "rain_share_recent"]
        ],
    ]

    for block in individual_blocks:
        features = features.join(
            block.loc[drivers],
            validate="one_to_one",
        )

    # Reprendre les signaux du dernier tour visible
    features["neutral_now"] = (
        last_rows["safety_car"].astype("boolean")
        | last_rows["virtual_sc"].astype("boolean")
    )

    features["red_flag_last_lap"] = (
        last_rows["red_flag"].astype("boolean")
    )

    # Préparer le peloton retenu pour les calculs collectifs
    field_snapshot = features.reset_index()
    field_snapshot["race_id"] = visible["race_id"].iloc[0]

    # Comparer chaque voiture aux autres de même composé de départ
    features = features.join(
        build_same_start_features(field_snapshot),
        validate="one_to_one",
    )

    # Calculer les blocs communs à la course-coupure
    shared_blocks = [
        build_field_features(field_snapshot),
        build_fitted_features(visible),
        build_rain_features(visible, leader_lap),
        build_neutral_features(visible, leader_lap),
        build_red_flag_features(visible, leader_lap),
    ]

    for block in shared_blocks:
        if len(block) != 1:
            raise ValueError("Un bloc collectif doit contenir une seule ligne.")

        for column in block.columns:
            if column in features.columns:
                raise ValueError(f"Variable dupliquée : {column}")

            features[column] = block[column].iloc[0]

    # Fixer explicitement les variables retournées et leur ordre
    return features.loc[:, candidate_features]

La cible et le résultat final ne sont jamais lus par cette fonction. En revanche, son argument visible doit avoir été filtré avant l’appel : elle ne calcule pas elle-même l’instant de coupure.

Vérifions d’abord notre exemple :

In [124]:
# Retrouver les voitures retenues et le tour de coupure de l'exemple
example_rows = train_features.loc[
    train_features["race_id"].eq("2018_01")
]

example_leader_lap = int(
    train_cutoffs.loc[
        train_cutoffs["race_id"].eq("2018_01"),
        "cutoff_lap",
    ].iloc[0]
)

# Reconstruire les features à partir de l'historique visible
assembled_check = build_all_features(
    visible=stop_visible,
    retained_drivers=example_rows["driver_code"],
    leader_lap=example_leader_lap,
    recent_window=5,
)

expected_features = (
    example_rows
    .set_index("driver_code")[candidate_features]
    .sort_index()
)

# Comparer les valeurs ; les types seront harmonisés au prétraitement
pd.testing.assert_frame_equal(
    assembled_check.sort_index(),
    expected_features,
    check_dtype=False,
)

print("Dimensions :", assembled_check.shape)
print("Les 40 variables reproduisent notre construction par étapes.")

Dimensions : (15, 40)
Les 40 variables reproduisent notre construction par étapes.


Les $40$ variables sont correctement reconstruites pour notre exemple. Vérifions maintenant que la fonction reproduit la table construite progressivement sur les $76$ courses d’entraînement, y compris ses valeurs manquantes.

>Nous vérifions toutes les voitures retenues, pas seulement les $1 296$ évaluables. Les autres participent aussi à la description du peloton.

>Ce contrôle établit la fidélité de l’assemblage à notre calcul par étapes. Il ne mesure pas encore les performances prédictives. Une fois réussi, nous pourrons utiliser la même fonction sur les courses de validation, sans modifier les définitions des variables.

In [125]:
rebuilt_parts = []

for cutoff in train_cutoffs.itertuples(index=False):

    # Reconstruire l'historique visible à la coupure déjà choisie
    visible = train_data.loc[
        train_data["race_id"].eq(cutoff.race_id)
        & train_data["race_time_s"].notna()
        & train_data["race_time_s"].le(cutoff.cutoff_time_s)
    ].copy()

    # Reprendre exactement les voitures retenues précédemment
    retained_drivers = train_features.loc[
        train_features["race_id"].eq(cutoff.race_id),
        "driver_code",
    ]

    rebuilt = build_all_features(
        visible=visible,
        retained_drivers=retained_drivers,
        leader_lap=int(cutoff.cutoff_lap),
        recent_window=5,
    ).reset_index()

    rebuilt["race_id"] = cutoff.race_id
    rebuilt_parts.append(rebuilt)

rebuilt_train_features = pd.concat(rebuilt_parts, ignore_index=True)

# Aligner les deux tables par course et pilote
comparison_keys = ["race_id", "driver_code"]

expected = (
    train_features
    .set_index(comparison_keys)[candidate_features]
    .sort_index()
)

actual = (
    rebuilt_train_features
    .set_index(comparison_keys)[candidate_features]
    .sort_index()
)

assert expected.index.is_unique
assert actual.index.is_unique

# Comparer les valeurs, les colonnes et les positions des valeurs manquantes
pd.testing.assert_frame_equal(
    actual,
    expected,
    check_dtype=False,
    check_exact=False,
    rtol=1e-12,
    atol=1e-12,
)

print("Courses vérifiées :", rebuilt_train_features["race_id"].nunique())
print("Voitures–coupures vérifiées :", len(actual))
print("Variables par voiture :", actual.shape[1])
print("La fonction reproduit toute la construction d'entraînement.")

Courses vérifiées : 76
Voitures–coupures vérifiées : 1413
Variables par voiture : 40
La fonction reproduit toute la construction d'entraînement.


>La reconstruction est vérifiée sur les $76$ courses : $1 413$ voitures–coupures, chacune décrite par $40$ variables.
>
>Parmi ces $1 413$ lignes, $1 296$ sont évaluables. Les $117$ autres participent au contexte du peloton, mais pas à la perte supervisée.

Construisons maintenant les features des $19$ courses de validation, avec les mêmes fonctions, la même fenêtre météo de cinq tours et les coupures déjà tirées.

In [126]:
# Reprendre les coupures des courses réservées à la validation
valid_cutoffs = cutoff_summary.loc[
    cutoff_summary["race_id"].isin(eda_valid["race_id"].unique())
]

valid_parts = []

for cutoff in valid_cutoffs.itertuples(index=False):

    # Utiliser uniquement les tours visibles à cette coupure
    visible = train_data.loc[
        train_data["race_id"].eq(cutoff.race_id)
        & train_data["race_time_s"].notna()
        & train_data["race_time_s"].le(cutoff.cutoff_time_s)
    ].copy()

    retained_drivers = eda_valid.loc[
        eda_valid["race_id"].eq(cutoff.race_id),
        "driver_code",
    ]

    features = build_all_features(
        visible=visible,
        retained_drivers=retained_drivers,
        leader_lap=int(cutoff.cutoff_lap),
        recent_window=5,
    ).reset_index()

    features["race_id"] = cutoff.race_id
    valid_parts.append(features)

valid_features = pd.concat(valid_parts, ignore_index=True)

# Ajouter les cibles et le masque d'évaluation séparément des features
valid_features = valid_features.merge(
    eda_valid[
        ["race_id", "driver_code", "next_compound", "scored"]
    ],
    on=["race_id", "driver_code"],
    how="left",
    validate="one_to_one",
    indicator="_metadata_merge",
)

assert len(valid_features) == len(eda_valid)
assert valid_features["_metadata_merge"].eq("both").all()

valid_features = valid_features.drop(columns="_metadata_merge")

assert set(valid_features["race_id"]).isdisjoint(
    train_features["race_id"]
)

print("Courses de validation :", valid_features["race_id"].nunique())
print("Voitures–coupures :", len(valid_features))
print("Observations évaluables :", int(valid_features["scored"].sum()))
print("Variables candidates :", len(candidate_features))

Courses de validation : 19
Voitures–coupures : 346
Observations évaluables : 313
Variables candidates : 40


>Cette étape calcule des features selon des règles déjà définies ; elle n’apprend aucun paramètre sur la validation. Les médianes d’imputation, les paramètres de standardisation et les modèles seront ajustés uniquement sur l’entraînement.

Les deux ensembles sont prêts, avec les mêmes $40$ variables candidates :
| Ensemble | Courses | Voitures retenues | Observations évaluables |
|---|---:|---:|---:|
| Entraînement | $76$ | $1 413$ | $1 296$ |
| Validation | $19$ | $346$ | $313$ |

Préparons maintenant les objets destinés aux modèles :
- `X` : les variables explicatives ;
- `y` : la cible `next_compound` ;
- `groups_train` : les courses, pour la _validation croisée interne_

In [127]:
# Sélectionner les observations qui contribuent à la perte supervisée
train_model_data = (
    train_features.loc[train_features["scored"]]
    .reset_index(drop=True)
)

valid_model_data = (
    valid_features.loc[valid_features["scored"]]
    .reset_index(drop=True)
)

# Séparer explicitement les features, la cible et les groupes
X_train = train_model_data[candidate_features].copy()
y_train = train_model_data["next_compound"].copy()
groups_train = train_model_data["race_id"].copy()

X_valid = valid_model_data[candidate_features].copy()
y_valid = valid_model_data["next_compound"].copy()

# Vérifier l'ordre des colonnes et la séparation des courses
assert X_train.columns.equals(X_valid.columns)
assert y_train.isin(CLASSES).all()
assert y_valid.isin(CLASSES).all()

assert set(groups_train).isdisjoint(
    valid_model_data["race_id"]
)

print("X_train :", X_train.shape)
print("y_train :", y_train.shape)
print("X_valid :", X_valid.shape)
print("y_valid :", y_valid.shape)

X_train : (1296, 40)
y_train : (1296,)
X_valid : (313, 40)
y_valid : (313,)


Nous disposons de $40$ variables candidates, avec $1 296$ observations d’entraînement et $313$ de validation, séparées par course.

Les valeurs manquantes concernent principalement :
- `start_compound` : composé du premier tour inconnu ;
- `same_start_other_cars` : groupe impossible à identifier ;
- `same_start_share_stopped` : groupe inconnu ou aucune autre voiture comparable ;
- les quatre `field_fitted_*` : aucun composé après arrêt identifiable.
Nous conservons ces absences dans les données partagées. Chaque imputation doit être intégrée au Pipeline (avec `from sklearn.impute import SimpleImputer (stratedy = "median" ou "mean" ou etc....)`) et ajustée uniquement sur l’apprentissage de chaque pli, jamais sur la validation.

Le traitement proposé est :

| Famille de modèles | Variables numériques manquantes | Catégories manquantes | Autres préparations |
|---|---|---|---|
| **LDA / QDA** | Médiane d’apprentissage | Catégorie `UNKNOWN`, puis encodage indicateur | Retrait des constantes et dépendances exactes ; régularisation des covariances à régler |
| **KNN** | Médiane d’apprentissage | `UNKNOWN`, puis encodage indicateur | Standardisation des numériques ; attention au poids des variables dans les distances |
| **Régression logistique** | Médiane d’apprentissage | `UNKNOWN`, puis encodage indicateur | Standardisation des numériques et régularisation |
| **Forêt aléatoire / bagging d’arbres**, pour notre première comparaison | Médiane d’apprentissage | `UNKNOWN`, puis encodage indicateur | Pas de standardisation nécessaire |

Pourquoi la médiane ? Elle fournit une valeur numérique utilisable par le modèle sans prétendre reconstituer la vraie information manquante :
$$
\widetilde x_{ij}=
\begin{cases}
x_{ij}, & \text{si la valeur est connue},\\
\operatorname{médiane}_{\text{apprentissage}}(X_j), & \text{sinon}.
\end{cases}
$$
Nous conservons le contexte de l’absence grâce aux variables déjà présentes : `start_compound_missing`, `field_stops_seen` et `same_start_other_cars`. Des indicateurs supplémentaires peuvent être étudiés, sans dupliquer automatiquement ceux qui existent déjà.

Deux précisions à ajouter :
- Pour LDA/QDA, la médiane ne résout pas les problèmes de covariance : l’encodage et les dépendances entre colonnes devront aussi être contrôlés. Par exemple, `UNKNOWN` encodé peut dupliquer `start_compound_missing`.
- Pour tous les modèles, on n’impute jamais `next_compound`. Seules les observations évaluables avec une cible connue servent à l’apprentissage supervisé.

- `groups_train` C’est une variable qui indique à quelle course appartient chaque observation de `X_train`. Elle sert à garder toutes les voitures d’une même course ensemble lors de la validation croisée. Les voitures d’une course partagent la météo, le circuit, la coupure et plusieurs variables de peloton. Si on répartissait les voitures au hasard, le modèle pourrait apprendre sur certaines voitures d’une course et être évalué sur les autres : cela mesurerait moins bien sa capacité à prédire une nouvelle course.